# ScentAI Stage 3 - Full Grounded Pipeline

This notebook connects the two independently validated boundaries:

`free-form query -> Gemma planner -> BGE-M3/Chroma/catalog -> ScentAI LoRA -> validator`

The vLLM/CUDA stack and CPU retrieval stack remain in separate uv environments.
The Colab kernel imports neither stack and only coordinates their localhost HTTP
APIs. Use a fresh **A100 High-RAM** runtime and choose **Run all**.

## Install the dependency-free uv launcher

In [ ]:
import shutil
import subprocess
import sys


def run_checked(command, *, label):
    print(f"\n[{label}]", " ".join(map(str, command)))
    subprocess.run([str(part) for part in command], check=True)


# uv is a standalone environment manager. --no-deps guarantees that installing
# this launcher cannot replace NumPy, Torch, Transformers, or any Colab library.
run_checked(
    [sys.executable, "-m", "pip", "install", "-q", "--no-deps", "uv"],
    label="install uv launcher",
)
UV = shutil.which("uv")
assert UV, "uv was installed but its executable is not on PATH."
print(subprocess.check_output([UV, "--version"], text=True).strip())

## Create the isolated vLLM environment

In [ ]:
from pathlib import Path

INFERENCE_ENV = Path("/content/scentai_inference_env")
INFERENCE_PYTHON = INFERENCE_ENV / "bin" / "python"
VLLM_EXECUTABLE = INFERENCE_ENV / "bin" / "vllm"
NINJA_EXECUTABLE = INFERENCE_ENV / "bin" / "ninja"
VLLM_VERSION = "0.25.1"

run_checked(
    [
        UV,
        "venv",
        "--clear",
        "--python",
        "3.12",
        "--seed",
        "--managed-python",
        str(INFERENCE_ENV),
    ],
    label="create clean Python 3.12 environment",
)
run_checked(
    [
        UV,
        "pip",
        "install",
        "--python",
        str(INFERENCE_PYTHON),
        f"vllm=={VLLM_VERSION}",
        "ninja",
        "--torch-backend=auto",
        "--strict",
    ],
    label="install vLLM with its matching PyTorch backend",
)
run_checked(
    [UV, "pip", "check", "--python", str(INFERENCE_PYTHON)],
    label="check isolated dependency graph",
)

assert INFERENCE_PYTHON.exists(), INFERENCE_PYTHON
assert VLLM_EXECUTABLE.exists(), VLLM_EXECUTABLE
assert NINJA_EXECUTABLE.exists(), NINJA_EXECUTABLE
print("ninja:", subprocess.check_output([str(NINJA_EXECUTABLE), "--version"], text=True).strip())

## Create the isolated CPU retrieval environment

In [ ]:
from pathlib import Path

RETRIEVAL_ENV = Path("/content/scentai_retrieval_env")
RETRIEVAL_PYTHON = RETRIEVAL_ENV / "bin" / "python"

run_checked(
    [
        UV, "venv", "--clear", "--python", "3.12", "--seed", "--managed-python",
        str(RETRIEVAL_ENV),
    ],
    label="create clean retrieval environment",
)
run_checked(
    [
        UV, "pip", "install", "--python", str(RETRIEVAL_PYTHON),
        "chromadb==1.5.9", "sentence-transformers==5.6.0",
        "--torch-backend=cpu", "--strict",
    ],
    label="install retrieval dependencies with CPU PyTorch",
)
run_checked(
    [UV, "pip", "check", "--python", str(RETRIEVAL_PYTHON)],
    label="check retrieval dependency graph",
)
assert RETRIEVAL_PYTHON.exists(), RETRIEVAL_PYTHON

## Verify the vLLM/CUDA binary graph

In [ ]:
import json

probe_code = r"""
import json
from importlib.metadata import version
import torch
import vllm

payload = {
    "vllm_distribution": version("vllm"),
    "vllm_module": vllm.__version__,
    "torch": torch.__version__,
    "torch_cuda": torch.version.cuda,
    "cuda_available": torch.cuda.is_available(),
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
}
print("SCENTAI_PROBE=" + json.dumps(payload))
"""
probe = subprocess.run(
    [str(INFERENCE_PYTHON), "-c", probe_code],
    text=True,
    capture_output=True,
)
if probe.returncode:
    print(probe.stdout)
    print(probe.stderr)
    raise RuntimeError("The isolated vLLM binary smoke test failed.")

probe_line = next(
    (line for line in probe.stdout.splitlines() if line.startswith("SCENTAI_PROBE=")),
    None,
)
assert probe_line, probe.stdout
inference_versions = json.loads(probe_line.split("=", 1)[1])
print(json.dumps(inference_versions, indent=2))
assert inference_versions["vllm_distribution"].startswith(VLLM_VERSION), inference_versions
assert inference_versions["cuda_available"], "Enable an A100 GPU runtime in Colab."

nvidia_smi = subprocess.run(
    [
        "nvidia-smi",
        "--query-gpu=name,memory.total",
        "--format=csv,noheader,nounits",
    ],
    text=True,
    capture_output=True,
)
if nvidia_smi.returncode:
    raise RuntimeError("nvidia-smi failed:\n" + nvidia_smi.stderr)
gpu_row = nvidia_smi.stdout.strip().splitlines()[0]
gpu_name, gpu_memory_mib = [part.strip() for part in gpu_row.rsplit(",", 1)]
gpu_memory_gb = float(gpu_memory_mib) / 1024
print(f"GPU: {gpu_name} ({gpu_memory_gb:.1f} GB)")
assert gpu_memory_gb >= 35, "Use an A100-class runtime with at least 35 GB of GPU memory."

## Verify the CPU retrieval dependency graph

In [ ]:
import json

probe_code = r'''import json
from importlib.metadata import version
import chromadb
import numpy
import scipy
import sklearn
import sentence_transformers
import torch
import transformers

print("SCENTAI_RETRIEVAL_PROBE=" + json.dumps({
    "chromadb": version("chromadb"),
    "sentence_transformers": version("sentence-transformers"),
    "transformers": version("transformers"),
    "torch": version("torch"),
    "torch_cuda": torch.version.cuda,
    "numpy": numpy.__version__,
    "scipy": scipy.__version__,
    "sklearn": sklearn.__version__,
}))
'''
probe = subprocess.run(
    [str(RETRIEVAL_PYTHON), "-c", probe_code],
    text=True,
    capture_output=True,
)
if probe.returncode:
    print(probe.stdout)
    print(probe.stderr)
    raise RuntimeError("The isolated retrieval environment failed its import smoke test.")
probe_line = next(
    (line for line in probe.stdout.splitlines() if line.startswith("SCENTAI_RETRIEVAL_PROBE=")),
    None,
)
assert probe_line, probe.stdout
retrieval_versions = json.loads(probe_line.split("=", 1)[1])
print(json.dumps(retrieval_versions, indent=2))
assert retrieval_versions["chromadb"] == "1.5.9", retrieval_versions
assert retrieval_versions["sentence_transformers"] == "5.6.0", retrieval_versions
assert retrieval_versions["torch_cuda"] is None, (
    "Retrieval must use CPU-only PyTorch so it cannot reserve inference VRAM."
)
print("Retrieval dependency smoke test: passed")

## Mount Drive, validate assets, and stage retrieval data locally

In [ ]:
import os
import time
from google.colab import drive, userdata

drive.mount("/content/drive")

PROJECT_DIR = Path("/content/drive/MyDrive/Perfume-Dataset")
MODEL_NAME = "google/gemma-4-12B-it"
LORA_NAME = "scentai"
FULL_ADAPTER_DIR = PROJECT_DIR / "models" / "scentai-gemma-4-12b-it-full-fastmodel-lora" / "best_lora_adapter"
PILOT_ADAPTER_DIR = PROJECT_DIR / "models" / "scentai-gemma-4-12b-it-pilot-fastmodel-lora" / "best_lora_adapter"
DRIVE_CHROMA_DIR = PROJECT_DIR / "chroma_db_bge_m3"
DRIVE_CATALOG = PROJECT_DIR / "scentai_catalog.sqlite3"
LOCAL_CHROMA_DIR = Path("/content/scentai_data/chroma_db_bge_m3")
LOCAL_CATALOG = Path("/content/scentai_data/scentai_catalog.sqlite3")

assert (DRIVE_CHROMA_DIR / "chroma.sqlite3").exists(), f"Missing Chroma DB: {DRIVE_CHROMA_DIR}"
assert DRIVE_CATALOG.exists(), f"Missing catalog: {DRIVE_CATALOG}"

expected_targets = {"q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"}

adapter_rejections = []
ADAPTER_DIR = None
adapter_config = None
for candidate in (FULL_ADAPTER_DIR, PILOT_ADAPTER_DIR):
    config_path = candidate / "adapter_config.json"
    weights_path = candidate / "adapter_model.safetensors"
    if not config_path.exists() or not weights_path.exists():
        adapter_rejections.append({"path": str(candidate), "reason": "missing files"})
        continue
    candidate_config = json.loads(config_path.read_text(encoding="utf-8"))
    problems = []
    if candidate_config.get("base_model_name_or_path") != MODEL_NAME:
        problems.append("base model mismatch")
    if int(candidate_config.get("r") or 0) != 16:
        problems.append("rank is not 16")
    if candidate_config.get("use_dora", False):
        problems.append("DoRA is not enabled in this vLLM path")
    if set(candidate_config.get("target_modules") or []) != expected_targets:
        problems.append("target modules differ")
    if problems:
        adapter_rejections.append({"path": str(candidate), "reason": ", ".join(problems)})
        continue
    ADAPTER_DIR = candidate
    adapter_config = candidate_config
    adapter_config_path = config_path
    adapter_weights_path = weights_path
    break

assert ADAPTER_DIR is not None, {"message": "No vLLM-compatible ScentAI adapter found", "rejections": adapter_rejections}
assert adapter_config is not None

try:
    HF_TOKEN = userdata.get("HF_TOKEN") or ""
except Exception:
    HF_TOKEN = ""

copy_started = time.perf_counter()
if LOCAL_CHROMA_DIR.parent.exists():
    shutil.rmtree(LOCAL_CHROMA_DIR.parent)
LOCAL_CHROMA_DIR.parent.mkdir(parents=True, exist_ok=True)
shutil.copytree(DRIVE_CHROMA_DIR, LOCAL_CHROMA_DIR)
shutil.copy2(DRIVE_CATALOG, LOCAL_CATALOG)
copy_elapsed = time.perf_counter() - copy_started

chroma_bytes = sum(path.stat().st_size for path in LOCAL_CHROMA_DIR.rglob("*") if path.is_file())
print("Adapter:", ADAPTER_DIR)
print("Adapter selection:", "full" if ADAPTER_DIR == FULL_ADAPTER_DIR else "pilot fallback")
if adapter_rejections:
    print("Skipped adapters:", json.dumps(adapter_rejections, indent=2))
print(f"Local Chroma: {chroma_bytes / (1024 ** 3):.2f} GB")
print(f"Local catalog: {LOCAL_CATALOG.stat().st_size / (1024 ** 2):.1f} MB")
print(f"Data staging: {copy_elapsed:.1f}s")
print("HF token:", "available" if HF_TOKEN else "not set; public model access will be attempted")

## Start the isolated BGE-M3 retrieval service

In [ ]:
SERVICE_SOURCE = 'from __future__ import annotations\n\nimport difflib\nimport json\nimport math\nimport os\nimport re\nimport sqlite3\nimport threading\nimport time\nimport unicodedata\nfrom collections import defaultdict\nfrom functools import lru_cache\nfrom http import HTTPStatus\nfrom http.server import BaseHTTPRequestHandler, ThreadingHTTPServer\nfrom pathlib import Path\nfrom typing import Any\n\n\nBGE_QUERY_PREFIX = "Represent this sentence for searching relevant passages: "\nDEFAULT_COLLECTION = "scentai_perfumes"\nDEFAULT_MODEL = "BAAI/bge-m3"\nVALID_GENDERS = {"male", "female", "unisex"}\nVALID_SEASONS = {"spring", "summer", "autumn", "winter"}\nVALID_TIMES = {"day", "night"}\nVALID_DISCOVERY_MODES = {"balanced", "mainstream", "niche"}\n\nCONCENTRATION_ALIAS_PATTERNS = (\n    (re.compile(r"\\be\\s*d\\s*p\\b|\\bepd\\b"), "eau de parfum"),\n    (re.compile(r"\\be\\s*d\\s*t\\b|\\betd\\b"), "eau de toilette"),\n    (re.compile(r"\\be\\s*d\\s*c\\b"), "eau de cologne"),\n    (re.compile(r"\\bparfume\\b"), "parfum"),\n)\nCONCENTRATION_SIGNATURES = (\n    ("extrait de parfum", "extrait"),\n    ("eau de parfum", "edp"),\n    ("eau de toilette", "edt"),\n    ("eau de cologne", "edc"),\n)\nCONCENTRATION_TOKENS = {"edp", "edt", "edc", "extrait", "parfum", "cologne"}\nEDITION_TOKENS = {\n    *CONCENTRATION_TOKENS,\n    "absolu", "absolute", "elixir", "extreme", "fraiche", "intense",\n    "limited", "edition", "oil", "sport", "spray",\n}\nIDENTITY_CONNECTORS = {"a", "by", "de", "des", "du", "for", "le", "la", "of", "pour", "the"}\n\n# These are spelling/market-name exceptions that cannot be inferred from initials\n# alone. Initialisms such as YSL, JPG, LV, MFK, PDM, and CDG are generated from\n# the catalog dynamically below.\nEXPLICIT_BRAND_ALIASES = {\n    "armani": "giorgio armani",\n    "cdg": "comme des garcons",\n    "ch": "carolina herrera",\n    "d g": "dolce gabbana",\n    "dg": "dolce gabbana",\n    "jpg": "jean paul gaultier",\n    "lv": "louis vuitton",\n    "mfk": "maison francis kurkdjian",\n    "pdm": "parfums de marly",\n    "tf": "tom ford",\n    "ysl": "yves saint laurent",\n}\n\n\ndef normalize_text(value: Any) -> str:\n    text = unicodedata.normalize("NFKD", str(value or "").lower())\n    text = "".join(char for char in text if not unicodedata.combining(char))\n    text = re.sub(r"[^a-z0-9]+", " ", text)\n    return " ".join(text.split())\n\n\ndef expand_concentration_aliases(value: Any) -> str:\n    """Expand common concentration abbreviations before identity matching."""\n    normalized = normalize_text(value)\n    for pattern, replacement in CONCENTRATION_ALIAS_PATTERNS:\n        normalized = pattern.sub(replacement, normalized)\n    return " ".join(normalized.split())\n\n\ndef identity_signature(value: Any) -> tuple[list[str], set[str]]:\n    """Return identity-bearing name tokens and explicit edition qualifiers."""\n    normalized = expand_concentration_aliases(value)\n    for phrase, marker in CONCENTRATION_SIGNATURES:\n        normalized = re.sub(rf"\\b{re.escape(phrase)}\\b", marker, normalized)\n    tokens = normalized.split()\n    qualifiers = {token for token in tokens if token in EDITION_TOKENS}\n    core = [\n        token for token in tokens\n        if token not in EDITION_TOKENS and token not in IDENTITY_CONNECTORS\n    ]\n    return core, qualifiers\n\n\ndef perfume_label(name: Any, brand: Any) -> str:\n    name_text = str(name or "").strip()\n    brand_text = str(brand or "").strip()\n    if not brand_text:\n        return name_text\n    normalized_name = normalize_text(name_text)\n    normalized_brand = normalize_text(brand_text)\n    if normalized_name == normalized_brand or normalized_name.endswith(f" by {normalized_brand}"):\n        return name_text\n    return f"{name_text} by {brand_text}"\n\n\ndef csv_terms(value: Any) -> set[str]:\n    return {normalize_text(part) for part in str(value or "").split(",") if normalize_text(part)}\n\n\ndef trait_variants(value: Any) -> set[str]:\n    """Return directional taxonomy variants for user-facing trait constraints."""\n    normalized = normalize_text(value)\n    return {\n        "musk": {"musk", "musky", "white musk"},\n        "musky": {"musk", "musky", "white musk"},\n        "leather": {"leather", "leathery", "suede"},\n        "leathery": {"leather", "leathery", "suede"},\n        "oud": {"oud", "aoud", "agarwood"},\n        "aoud": {"oud", "aoud", "agarwood"},\n        "agarwood": {"oud", "aoud", "agarwood"},\n        "smoke": {"smoke", "smoky"},\n        "smoky": {"smoke", "smoky"},\n    }.get(normalized, {normalized} if normalized else set())\n\n\ndef split_name_brand_hint(hint: str) -> tuple[str, str | None]:\n    normalized = normalize_text(hint)\n    if " by " not in normalized:\n        return normalized, None\n    name, brand = normalized.rsplit(" by ", 1)\n    return (name.strip(), brand.strip()) if name.strip() and brand.strip() else (normalized, None)\n\n\ndef confidence_adjusted_rating(rating: float, popularity: int, prior: float = 3.8) -> float:\n    if rating <= 0:\n        return 0.0\n    weight = popularity / (popularity + 250) if popularity > 0 else 0.0\n    return rating * weight + prior * (1.0 - weight)\n\n\ndef community_similarity_score(up_votes: int, down_votes: int) -> float:\n    total = up_votes + down_votes\n    if total <= 0:\n        return 0.0\n    approval = (up_votes + 1) / (total + 2)\n    confidence = min(math.log1p(total) / math.log(501), 1.0)\n    return approval * 0.72 + confidence * 0.28\n\n\ndef set_similarity(left: set[str], right: set[str]) -> float:\n    if not left or not right:\n        return 0.0\n    intersection = len(left & right)\n    jaccard = intersection / len(left | right)\n    overlap = intersection / min(len(left), len(right))\n    return jaccard * 0.65 + overlap * 0.35\n\n\ndef trait_match_strength(wanted: str, trait: str) -> float:\n    """Score canonical trait matches without rewarding accidental substrings."""\n    wanted_norm = normalize_text(wanted)\n    trait_norm = normalize_text(trait)\n    if not wanted_norm or not trait_norm:\n        return 0.0\n    if wanted_norm == trait_norm:\n        return 1.0\n    if trait_norm in trait_variants(wanted_norm):\n        return 1.0\n\n    if wanted_norm in {"spicy", "floral"} and wanted_norm in trait_norm.split():\n        return 0.75\n\n    # A multi-word planner term can be a useful refinement of a stored trait,\n    # but a single token such as "fresh" must not match "fresh spicy".\n    if len(wanted_norm.split()) > 1:\n        padded_wanted = f" {wanted_norm} "\n        padded_trait = f" {trait_norm} "\n        if padded_wanted in padded_trait or padded_trait in padded_wanted:\n            return 0.65\n    return 0.0\n\n\ndef metadata_has_term(metadata: dict[str, Any], term: str) -> bool:\n    term_norm = normalize_text(term)\n    if not term_norm:\n        return False\n    searchable = normalize_text(" ".join(\n        str(metadata.get(key) or "")\n        for key in ("name", "brand", "accords_csv", "notes_csv")\n    ))\n    if f" {term_norm} " in f" {searchable} ":\n        return True\n    traits = csv_terms(metadata.get("accords_csv")) | csv_terms(metadata.get("notes_csv"))\n    return any(trait_match_strength(term_norm, trait) > 0.0 for trait in traits)\n\n\nclass CatalogResolver:\n    def __init__(self, catalog_path: Path | str) -> None:\n        self.path = Path(catalog_path)\n        if not self.path.exists():\n            raise FileNotFoundError(f"Catalog does not exist: {self.path}")\n        self.connection = sqlite3.connect(\n            f"file:{self.path}?mode=ro",\n            uri=True,\n            check_same_thread=False,\n        )\n        self.connection.row_factory = sqlite3.Row\n        self.lock = threading.RLock()\n        self._brand_norm_to_name, self._brand_alias_to_norm = self._build_brand_alias_index()\n\n    def _build_brand_alias_index(self) -> tuple[dict[str, str], dict[str, str]]:\n        with self.lock:\n            rows = self.connection.execute(\n                """\n                SELECT brand, SUM(popularity) AS total_popularity\n                FROM perfumes\n                WHERE brand != \'\'\n                GROUP BY brand\n                ORDER BY total_popularity DESC\n                """\n            ).fetchall()\n\n        norm_to_name: dict[str, str] = {}\n        alias_targets: dict[str, set[str]] = defaultdict(set)\n        for row in rows:\n            brand = str(row["brand"] or "").strip()\n            brand_norm = normalize_text(brand)\n            if not brand_norm:\n                continue\n            norm_to_name.setdefault(brand_norm, brand)\n            alias_targets[brand_norm].add(brand_norm)\n            words = brand_norm.split()\n            if 2 <= len(words) <= 5:\n                initials = "".join(word[0] for word in words)\n                spaced_initials = " ".join(word[0] for word in words)\n                if len(initials) >= 2:\n                    alias_targets[initials].add(brand_norm)\n                    alias_targets[spaced_initials].add(brand_norm)\n\n        # Ambiguous initials are intentionally discarded. Guessing a brand is\n        # worse than falling through to the lexical resolver.\n        aliases = {\n            alias: next(iter(targets))\n            for alias, targets in alias_targets.items()\n            if len(targets) == 1\n        }\n        # A compact curated list overrides ambiguous generated initials only\n        # where the fragrance community has a stable, well-known shorthand.\n        for alias, target in EXPLICIT_BRAND_ALIASES.items():\n            if target in norm_to_name:\n                aliases[normalize_text(alias)] = target\n        return norm_to_name, aliases\n\n    def _canonical_brand_norm(self, hint: Any) -> str | None:\n        normalized = normalize_text(hint)\n        return self._brand_alias_to_norm.get(normalized)\n\n    def _extract_edge_brand(self, normalized_hint: str) -> tuple[str, str | None]:\n        tokens = normalized_hint.split()\n        if not tokens:\n            return normalized_hint, None\n        max_width = min(5, len(tokens))\n        matches: list[tuple[int, str, str]] = []\n        for width in range(1, max_width + 1):\n            prefix = " ".join(tokens[:width])\n            suffix = " ".join(tokens[-width:])\n            if prefix in self._brand_alias_to_norm:\n                remainder = tokens[width:]\n                if remainder[:1] == ["s"]:\n                    remainder = remainder[1:]\n                matches.append((width, " ".join(remainder), self._brand_alias_to_norm[prefix]))\n            if suffix in self._brand_alias_to_norm:\n                matches.append((width, " ".join(tokens[:-width]), self._brand_alias_to_norm[suffix]))\n        if not matches:\n            return normalized_hint, None\n        _, name_hint, brand_norm = max(matches, key=lambda item: (item[0], len(item[1])))\n        return name_hint.strip(), brand_norm\n\n    def _prepare_hint(self, hint: str) -> tuple[str, str | None, str]:\n        normalized = expand_concentration_aliases(hint)\n        name_hint, raw_brand_hint = split_name_brand_hint(normalized)\n        if raw_brand_hint:\n            brand_hint = self._canonical_brand_norm(raw_brand_hint)\n            if brand_hint:\n                return name_hint, brand_hint, f"{name_hint} {brand_hint}".strip()\n        edge_name, edge_brand = self._extract_edge_brand(normalized)\n        if edge_brand:\n            return edge_name, edge_brand, f"{edge_name} {edge_brand}".strip()\n        return normalized, None, normalized\n\n    def counts(self) -> dict[str, int]:\n        with self.lock:\n            perfume_count = int(self.connection.execute("SELECT COUNT(*) FROM perfumes").fetchone()[0])\n            edge_count = int(self.connection.execute("SELECT COUNT(*) FROM similarity_edges").fetchone()[0])\n        return {"perfumes": perfume_count, "similarity_edges": edge_count}\n\n    @lru_cache(maxsize=4096)\n    def canonical_brand(self, hint: str) -> str | None:\n        normalized = self._canonical_brand_norm(hint) or normalize_text(hint)\n        if not normalized:\n            return None\n        with self.lock:\n            row = self.connection.execute(\n                """\n                SELECT brand, SUM(popularity) AS total_popularity\n                FROM perfumes\n                WHERE brand_norm = ?\n                GROUP BY brand\n                ORDER BY total_popularity DESC\n                LIMIT 1\n                """,\n                (normalized,),\n            ).fetchone()\n        return str(row["brand"]) if row else None\n\n    @lru_cache(maxsize=4096)\n    def resolve(self, hint: str) -> dict[str, Any] | None:\n        normalized = expand_concentration_aliases(hint)\n        if not normalized:\n            return None\n        name_hint, brand_hint, label_hint = self._prepare_hint(hint)\n\n        with self.lock:\n            if brand_hint:\n                exact = self.connection.execute(\n                    """\n                    SELECT * FROM perfumes\n                    WHERE (name_norm = ? AND brand_norm = ?) OR label_norm = ?\n                    ORDER BY popularity DESC LIMIT 20\n                    """,\n                    (name_hint, brand_hint, label_hint),\n                ).fetchall()\n            else:\n                exact = self.connection.execute(\n                    """\n                    SELECT * FROM perfumes\n                    WHERE name_norm = ? OR label_norm = ?\n                    ORDER BY popularity DESC LIMIT 20\n                    """,\n                    (name_hint, label_hint),\n                ).fetchall()\n        if exact:\n            return dict(exact[0])\n\n        family_found, family = self._dominant_family_member(name_hint, brand_hint)\n        if family_found:\n            return family\n\n        tokens = [token for token in name_hint.split() if len(token) >= 2]\n        if not tokens and brand_hint and name_hint:\n            tokens = [name_hint]\n        if not tokens:\n            return None\n        clauses: list[str] = []\n        values: list[str] = []\n        for token in tokens[:5]:\n            clauses.append("(name_norm LIKE ? OR brand_norm LIKE ?)")\n            values.extend([f"%{token}%", f"%{token}%"])\n        brand_clause = " AND brand_norm = ?" if brand_hint else ""\n        if brand_hint:\n            values.append(brand_hint)\n        with self.lock:\n            rows = self.connection.execute(\n                "SELECT * FROM perfumes WHERE "\n                + "(" + " OR ".join(clauses) + ")"\n                + brand_clause\n                + " ORDER BY popularity DESC LIMIT 500",\n                values,\n            ).fetchall()\n            if not rows and brand_hint:\n                # When the brand is certain, a misspelled short product name\n                # should still be compared against that house\'s catalog.\n                rows = self.connection.execute(\n                    """\n                    SELECT * FROM perfumes\n                    WHERE brand_norm = ?\n                    ORDER BY popularity DESC, rating DESC\n                    LIMIT 500\n                    """,\n                    (brand_hint,),\n                ).fetchall()\n        ranked = sorted(\n            (dict(row) for row in rows),\n            key=lambda row: self._resolution_score(name_hint, brand_hint, row),\n            reverse=True,\n        )\n        if not ranked or self._resolution_score(name_hint, brand_hint, ranked[0]) < 0.60:\n            return None\n        return ranked[0]\n\n    def _dominant_family_member(\n        self,\n        name_hint: str,\n        brand_hint: str | None,\n    ) -> tuple[bool, dict[str, Any] | None]:\n        if len(name_hint.split()) < 2:\n            return False, None\n        values: list[Any] = [f"{name_hint} %"]\n        brand_clause = ""\n        if brand_hint:\n            brand_clause = " AND brand_norm = ?"\n            values.append(brand_hint)\n        with self.lock:\n            rows = self.connection.execute(\n                "SELECT * FROM perfumes WHERE name_norm LIKE ?"\n                + brand_clause\n                + " ORDER BY popularity DESC, rating DESC LIMIT 25",\n                values,\n            ).fetchall()\n        if not rows:\n            return False, None\n        ranked = [dict(row) for row in rows]\n        leader = ranked[0]\n        leader_popularity = int(leader.get("popularity") or 0)\n        runner_up = int(ranked[1].get("popularity") or 0) if len(ranked) > 1 else 0\n        clearly_dominant = (\n            leader_popularity >= 500\n            and (runner_up == 0 or leader_popularity >= runner_up * 2)\n        )\n        return True, leader if clearly_dominant else None\n\n    @staticmethod\n    def _resolution_score(\n        name_hint: str,\n        brand_hint: str | None,\n        row: dict[str, Any],\n    ) -> float:\n        name = str(row.get("name_norm") or "")\n        row_brand = str(row.get("brand_norm") or "")\n        hint_core, hint_qualifiers = identity_signature(name_hint)\n        name_core, name_qualifiers = identity_signature(name)\n        hint_core_text = " ".join(hint_core)\n        name_core_text = " ".join(name_core)\n        hint_tokens = set(hint_core)\n        name_tokens = set(name_core)\n        overlap = len(hint_tokens & name_tokens) / max(len(hint_tokens | name_tokens), 1)\n        coverage = len(hint_tokens & name_tokens) / max(len(hint_tokens), 1)\n        core_sequence = difflib.SequenceMatcher(None, hint_core_text, name_core_text).ratio()\n        full_sequence = difflib.SequenceMatcher(\n            None,\n            expand_concentration_aliases(name_hint),\n            name,\n        ).ratio()\n        core_lexical = max(core_sequence, overlap, coverage)\n        # Edition words such as "eau de toilette" are common across thousands\n        # of unrelated perfumes. The product-name core must dominate them.\n        lexical = core_lexical * 0.90 + full_sequence * 0.10\n        core_mismatch_penalty = (\n            0.38\n            if hint_tokens and name_tokens and not (hint_tokens & name_tokens) and core_sequence < 0.72\n            else 0.0\n        )\n\n        qualifier_score = 0.5\n        qualifier_penalty = 0.0\n        if hint_qualifiers:\n            qualifier_score = len(hint_qualifiers & name_qualifiers) / len(hint_qualifiers)\n            # Some catalog base releases omit EDT/EDC from their display name.\n            # Missing edition text is a moderate penalty; an explicitly\n            # conflicting concentration remains a hard penalty below.\n            qualifier_penalty += 0.06 * (\n                len(hint_qualifiers - name_qualifiers) / len(hint_qualifiers)\n            )\n            requested_concentration = hint_qualifiers & CONCENTRATION_TOKENS\n            candidate_concentration = name_qualifiers & CONCENTRATION_TOKENS\n            if requested_concentration and candidate_concentration and not (\n                requested_concentration & candidate_concentration\n            ):\n                qualifier_penalty += 0.32\n\n        brand_score = 1.0 if brand_hint and row_brand == brand_hint else (0.5 if not brand_hint else 0.0)\n        popularity = int(row.get("popularity") or 0)\n        popularity_score = min(math.log10(popularity + 1) / math.log10(50000), 1.0) if popularity else 0.0\n        return (\n            lexical * 0.72\n            + qualifier_score * core_lexical * 0.18\n            + brand_score * 0.07\n            + popularity_score * 0.03\n            - qualifier_penalty\n            - core_mismatch_penalty\n        )\n\n    def direct_similarity(self, source_id: int, limit: int = 200) -> list[dict[str, Any]]:\n        with self.lock:\n            rows = self.connection.execute(\n                """\n                SELECT p.*, e.up_votes, e.down_votes\n                FROM similarity_edges e\n                JOIN perfumes p ON p.perfume_id = e.target_id\n                WHERE e.source_id = ?\n                ORDER BY (e.up_votes - e.down_votes) DESC, e.up_votes DESC\n                LIMIT ?\n                """,\n                (source_id, limit),\n            ).fetchall()\n        return [dict(row) for row in rows]\n\n    def popular_candidates(\n        self,\n        filters: dict[str, Any],\n        required_terms: list[str],\n        exclude_terms: list[str],\n        *,\n        canonical_brand: str | None = None,\n        limit: int = 160,\n    ) -> list[dict[str, Any]]:\n        """Return popular eligible rows independently of the ANN neighborhood."""\n        clauses: list[str] = []\n        values: list[Any] = []\n\n        gender = normalize_text(filters.get("gender"))\n        if gender:\n            allowed = [gender] if gender == "unisex" else [gender, "unisex"]\n            clauses.append("gender IN (" + ", ".join("?" for _ in allowed) + ")")\n            values.extend(allowed)\n        if canonical_brand:\n            clauses.append("brand = ?")\n            values.append(canonical_brand)\n        if filters.get("min_rating") is not None:\n            clauses.append("rating >= ?")\n            values.append(float(filters["min_rating"]))\n        if filters.get("min_popularity") is not None:\n            clauses.append("popularity >= ?")\n            values.append(int(filters["min_popularity"]))\n\n        sql = "SELECT * FROM perfumes"\n        if clauses:\n            sql += " WHERE " + " AND ".join(clauses)\n        # Scan a generous popularity-first window, then apply taxonomy-aware\n        # trait and season/time checks in Python.\n        sql += " ORDER BY popularity DESC, rating DESC LIMIT 4000"\n        with self.lock:\n            rows = [dict(row) for row in self.connection.execute(sql, values).fetchall()]\n\n        season = normalize_text(filters.get("season"))\n        time_profile = normalize_text(filters.get("time"))\n        output: list[dict[str, Any]] = []\n        for row in rows:\n            if season and season not in csv_terms(row.get("seasons_csv")):\n                continue\n            if time_profile and time_profile not in csv_terms(row.get("time_profile_csv")):\n                continue\n            traits = csv_terms(row.get("accords_csv")) | csv_terms(row.get("notes_csv"))\n            if any(\n                not any(trait_match_strength(term, trait) > 0.0 for trait in traits)\n                for term in required_terms\n            ):\n                continue\n            if any(metadata_has_term(row, term) for term in exclude_terms):\n                continue\n            output.append(row)\n            if len(output) >= limit:\n                break\n        return output\n\n\nclass RetrievalEngine:\n    def __init__(\n        self,\n        db_dir: Path | str,\n        catalog_path: Path | str,\n        *,\n        collection_name: str = DEFAULT_COLLECTION,\n        model_name: str = DEFAULT_MODEL,\n    ) -> None:\n        import chromadb\n        from sentence_transformers import SentenceTransformer\n\n        self.started_at = time.time()\n        self.db_dir = Path(db_dir)\n        self.collection_name = collection_name\n        self.model_name = model_name\n        self.catalog = CatalogResolver(catalog_path)\n        self.client = chromadb.PersistentClient(path=str(self.db_dir))\n        self.collection = self.client.get_collection(collection_name)\n        self.model = SentenceTransformer(model_name, device="cpu")\n        self.encode_lock = threading.Lock()\n\n    def health(self) -> dict[str, Any]:\n        return {\n            "status": "ok",\n            "model": self.model_name,\n            "device": "cpu",\n            "collection": self.collection_name,\n            "collection_count": self.collection.count(),\n            "catalog": self.catalog.counts(),\n            "embedding_cache": self._encode.cache_info()._asdict(),\n            "uptime_seconds": round(time.time() - self.started_at, 2),\n        }\n\n    @lru_cache(maxsize=512)\n    def _encode(self, text: str) -> tuple[float, ...]:\n        with self.encode_lock:\n            vector = self.model.encode(text, normalize_embeddings=True)\n        return tuple(float(value) for value in vector)\n\n    def search(self, payload: dict[str, Any]) -> dict[str, Any]:\n        started = time.perf_counter()\n        query = str(payload.get("query") or "").strip()\n        if not query:\n            raise ValueError("query is required")\n        top_k = min(max(int(payload.get("top_k") or 10), 1), 30)\n        fetch_k = min(max(int(payload.get("fetch_k") or 300), top_k), 300)\n        filters = dict(payload.get("filters") or {})\n        exclude_terms = self._clean_terms(payload.get("exclude_terms") or [])\n        wanted_terms = self._clean_terms(payload.get("wanted_terms") or [])\n        required_terms = self._clean_terms(payload.get("required_terms") or [])\n        exclude_ids = self._clean_ids(payload.get("exclude_ids") or [])\n        discovery_mode = normalize_text(payload.get("discovery_mode") or "balanced")\n        if discovery_mode not in VALID_DISCOVERY_MODES:\n            raise ValueError(f"Unsupported discovery mode: {discovery_mode}")\n        where, canonical_brand = self._build_where(filters)\n        normalized_query = normalize_text(query)\n        padded_query = f" {normalized_query} "\n\n        encoded_query = BGE_QUERY_PREFIX + query\n        before = self._encode.cache_info().hits\n        embedding = [list(self._encode(encoded_query))]\n        cache_hit = self._encode.cache_info().hits > before\n        raw = self.collection.query(\n            query_embeddings=embedding,\n            n_results=fetch_k,\n            where=where,\n            include=["documents", "metadatas", "distances"],\n        )\n\n        raw_rows = [(*row, {"semantic_ann"}) for row in zip(\n            raw["documents"][0],\n            raw["metadatas"][0],\n            raw["distances"][0],\n        )]\n\n        # ANN relevance and catalog popularity are complementary candidate\n        # generators. This branch prevents famous, eligible perfumes from\n        # disappearing merely because their names/cards are not close enough\n        # to a short lifestyle query in the first ANN neighborhood.\n        if discovery_mode != "niche":\n            popular_rows = self.catalog.popular_candidates(\n                filters,\n                required_terms,\n                exclude_terms,\n                canonical_brand=canonical_brand,\n                limit=200 if discovery_mode == "mainstream" else 140,\n            )\n            popular_ids = [str(row["perfume_id"]) for row in popular_rows]\n            if popular_ids:\n                popular = self.collection.get(\n                    ids=popular_ids,\n                    include=["documents", "metadatas", "embeddings"],\n                )\n                existing = {int(metadata.get("perfume_id") or 0): index for index, (_, metadata, _, _) in enumerate(raw_rows)}\n                for document, metadata, vector in zip(\n                    popular.get("documents") or [],\n                    popular.get("metadatas") or [],\n                    popular.get("embeddings") if popular.get("embeddings") is not None else [],\n                ):\n                    perfume_id = int(metadata.get("perfume_id") or 0)\n                    if perfume_id in existing:\n                        raw_rows[existing[perfume_id]][3].add("catalog_popular")\n                        continue\n                    semantic = sum(float(left) * float(right) for left, right in zip(embedding[0], vector))\n                    raw_rows.append((document, metadata, 1.0 - semantic, {"catalog_popular"}))\n        fetched_trait_sets = [\n            csv_terms(metadata.get("accords_csv")) | csv_terms(metadata.get("notes_csv"))\n            for _, metadata, _, _ in raw_rows\n        ]\n        supported_wanted_terms = [\n            term\n            for term in wanted_terms\n            if any(\n                any(trait_match_strength(term, trait) > 0.0 for trait in traits)\n                for traits in fetched_trait_sets\n            )\n        ]\n        supported_required_terms = [\n            term\n            for term in required_terms\n            if any(\n                any(trait_match_strength(term, trait) > 0.0 for trait in traits)\n                for traits in fetched_trait_sets\n            )\n        ]\n\n        candidates: list[dict[str, Any]] = []\n        collision_brands: set[str] = set()\n        score_weights = {\n            # Balanced keeps semantic fit as the largest signal. Popularity\n            # earns a candidate entry into the race, not an automatic win.\n            "balanced": (0.62, 0.16, 0.08, 0.14),\n            "mainstream": (0.48, 0.29, 0.09, 0.14),\n            "niche": (0.70, 0.05, 0.11, 0.14),\n        }[discovery_mode]\n        semantic_weight, popularity_weight, rating_weight, wanted_weight = score_weights\n        for document, metadata, distance, source_pools in raw_rows:\n            meta = dict(metadata)\n            perfume_id = int(meta.get("perfume_id") or 0)\n            if perfume_id in exclude_ids:\n                continue\n            if self._has_excluded_term(meta, exclude_terms):\n                continue\n            semantic = max(0.0, 1.0 - float(distance))\n            popularity = int(meta.get("popularity") or 0)\n            rating = confidence_adjusted_rating(float(meta.get("rating") or 0.0), popularity)\n            popularity_score = min(math.log10(popularity + 1) / math.log10(50000), 1.0) if popularity else 0.0\n            rating_score = min(max((rating - 3.2) / 1.3, 0.0), 1.0) if rating else 0.0\n            traits = csv_terms(meta.get("accords_csv")) | csv_terms(meta.get("notes_csv"))\n            if required_terms and (\n                len(supported_required_terms) != len(required_terms)\n                or any(\n                    not any(trait_match_strength(term, trait) > 0.0 for trait in traits)\n                    for term in supported_required_terms\n                )\n            ):\n                continue\n            wanted_matches = {\n                term: max((trait_match_strength(term, trait) for trait in traits), default=0.0)\n                for term in supported_wanted_terms\n            }\n            wanted_matches = {\n                term: strength for term, strength in wanted_matches.items() if strength > 0.0\n            }\n            wanted_hits = len(wanted_matches)\n            wanted_score = (\n                sum(wanted_matches.values()) / len(supported_wanted_terms)\n                if supported_wanted_terms\n                else 0.0\n            )\n            brand_norm = normalize_text(meta.get("brand"))\n            brand_collision = bool(\n                not canonical_brand\n                and len(brand_norm) >= 4\n                and f" {brand_norm} " in padded_query\n            )\n            if brand_collision:\n                collision_brands.add(brand_norm)\n            collision_penalty = 0.14 if brand_collision else 0.0\n            final_score = (\n                semantic * semantic_weight\n                + popularity_score * popularity_weight\n                + rating_score * rating_weight\n                + wanted_score * wanted_weight\n                - collision_penalty\n            )\n            candidates.append(\n                {\n                    "perfume_id": perfume_id,\n                    "name": str(meta.get("name") or ""),\n                    "brand": str(meta.get("brand") or ""),\n                    "label": perfume_label(meta.get("name"), meta.get("brand")),\n                    "document": document,\n                    "metadata": meta,\n                    "distance": round(float(distance), 6),\n                    "semantic_score": round(semantic, 6),\n                    "score": round(final_score, 6),\n                    "reasons": {\n                        "semantic": round(semantic, 4),\n                        "popularity": round(popularity_score, 4),\n                        "rating": round(rating_score, 4),\n                        "wanted_term_hits": wanted_hits,\n                        "wanted_term_score": round(wanted_score, 4),\n                        "wanted_term_matches": {\n                            term: round(strength, 2)\n                            for term, strength in wanted_matches.items()\n                        },\n                        "accidental_brand_collision_penalty": collision_penalty,\n                        "candidate_sources": sorted(source_pools),\n                    },\n                }\n            )\n        candidates.sort(key=lambda item: item["score"], reverse=True)\n        selected = self._diversify(\n            candidates,\n            top_k,\n            brand_limit=top_k if canonical_brand else 2,\n            per_brand_limits={brand: 1 for brand in collision_brands},\n        )\n        return {\n            "route": "semantic",\n            "query": query,\n            "filters": {**filters, "canonical_brand": canonical_brand},\n            "exclude_terms": exclude_terms,\n            "wanted_terms": wanted_terms,\n            "supported_wanted_terms": supported_wanted_terms,\n            "ignored_wanted_terms": [term for term in wanted_terms if term not in supported_wanted_terms],\n            "required_terms": required_terms,\n            "supported_required_terms": supported_required_terms,\n            "exclude_ids": sorted(exclude_ids),\n            "discovery_mode": discovery_mode,\n            "accidental_brand_collisions": sorted(collision_brands),\n            "embedding_cache_hit": cache_hit,\n            "elapsed_seconds": round(time.perf_counter() - started, 4),\n            "result_count": len(selected),\n            "results": selected,\n        }\n\n    def resolve(self, payload: dict[str, Any]) -> dict[str, Any]:\n        hint = str(payload.get("hint") or "").strip()\n        if not hint:\n            raise ValueError("hint is required")\n        row = self.catalog.resolve(hint)\n        return {"hint": hint, "resolved": self._public_catalog_row(row) if row else None}\n\n    def similar(self, payload: dict[str, Any]) -> dict[str, Any]:\n        started = time.perf_counter()\n        hint = str(payload.get("hint") or "").strip()\n        if not hint:\n            raise ValueError("hint is required")\n        source = self.catalog.resolve(hint)\n        if not source:\n            raise LookupError(f"Could not resolve perfume: {hint}")\n        top_k = min(max(int(payload.get("top_k") or 10), 1), 30)\n        exclude_terms = self._clean_terms(payload.get("exclude_terms") or [])\n        required_terms = self._clean_terms(payload.get("required_terms") or [])\n        exclude_ids = self._clean_ids(payload.get("exclude_ids") or [])\n        exclude_source_brand = bool(payload.get("exclude_source_brand", False))\n        source_accords = csv_terms(source.get("accords_csv"))\n        source_notes = csv_terms(source.get("notes_csv"))\n\n        scored: list[dict[str, Any]] = []\n        for row in self.catalog.direct_similarity(int(source["perfume_id"]), limit=max(200, top_k * 15)):\n            if int(row.get("perfume_id") or 0) in exclude_ids:\n                continue\n            if exclude_source_brand and normalize_text(row.get("brand")) == normalize_text(source.get("brand")):\n                continue\n            if self._has_excluded_term(row, exclude_terms):\n                continue\n            row_traits = csv_terms(row.get("accords_csv")) | csv_terms(row.get("notes_csv"))\n            if any(\n                not any(trait_match_strength(term, trait) > 0.0 for trait in row_traits)\n                for term in required_terms\n            ):\n                continue\n            graph = community_similarity_score(int(row.get("up_votes") or 0), int(row.get("down_votes") or 0))\n            accord_similarity = set_similarity(source_accords, csv_terms(row.get("accords_csv")))\n            note_similarity = set_similarity(source_notes, csv_terms(row.get("notes_csv")))\n            structure = accord_similarity * 0.68 + note_similarity * 0.32\n            popularity = int(row.get("popularity") or 0)\n            quality = min(math.log10(popularity + 1) / math.log10(50000), 1.0) if popularity else 0.0\n            final_score = graph * 0.68 + structure * 0.24 + quality * 0.08\n            public = self._public_catalog_row(row)\n            assert public is not None\n            public.update(\n                {\n                    "score": round(final_score, 6),\n                    "community_similarity": round(graph, 6),\n                    "accord_similarity": round(accord_similarity, 6),\n                    "note_similarity": round(note_similarity, 6),\n                    "up_votes": int(row.get("up_votes") or 0),\n                    "down_votes": int(row.get("down_votes") or 0),\n                }\n            )\n            scored.append(public)\n        scored.sort(key=lambda item: item["score"], reverse=True)\n        selected = self._diversify(scored, top_k, brand_limit=2)\n        return {\n            "route": "community_similarity",\n            "hint": hint,\n            "source": self._public_catalog_row(source),\n            "exclude_terms": exclude_terms,\n            "required_terms": required_terms,\n            "exclude_ids": sorted(exclude_ids),\n            "elapsed_seconds": round(time.perf_counter() - started, 4),\n            "result_count": len(selected),\n            "results": selected,\n        }\n\n    def _build_where(self, filters: dict[str, Any]) -> tuple[dict[str, Any] | None, str | None]:\n        clauses: list[dict[str, Any]] = []\n        gender = normalize_text(filters.get("gender"))\n        if gender:\n            if gender not in VALID_GENDERS:\n                raise ValueError(f"Unsupported gender filter: {gender}")\n            allowed = [gender] if gender == "unisex" else [gender, "unisex"]\n            clauses.append({"gender": {"$in": allowed}})\n        season = normalize_text(filters.get("season"))\n        if season:\n            if season not in VALID_SEASONS:\n                raise ValueError(f"Unsupported season filter: {season}")\n            clauses.append({f"season_{season}": True})\n        time_profile = normalize_text(filters.get("time"))\n        if time_profile:\n            if time_profile not in VALID_TIMES:\n                raise ValueError(f"Unsupported time filter: {time_profile}")\n            clauses.append({f"time_{time_profile}": True})\n        if filters.get("min_rating") is not None:\n            clauses.append({"rating": {"$gte": float(filters["min_rating"])}})\n        if filters.get("min_popularity") is not None:\n            clauses.append({"popularity": {"$gte": int(filters["min_popularity"])}})\n\n        canonical_brand = None\n        brand_hint = str(filters.get("brand") or "").strip()\n        if brand_hint:\n            canonical_brand = self.catalog.canonical_brand(brand_hint)\n            if not canonical_brand:\n                raise LookupError(f"Unknown brand: {brand_hint}")\n            clauses.append({"brand": canonical_brand})\n\n        if not clauses:\n            return None, canonical_brand\n        if len(clauses) == 1:\n            return clauses[0], canonical_brand\n        return {"$and": clauses}, canonical_brand\n\n    @staticmethod\n    def _clean_terms(values: Any) -> list[str]:\n        if not isinstance(values, list):\n            raise ValueError("wanted_terms, required_terms, and exclude_terms must be JSON arrays")\n        return list(dict.fromkeys(term for value in values if (term := normalize_text(value))))[:30]\n\n    @staticmethod\n    def _clean_ids(values: Any) -> set[int]:\n        if not isinstance(values, list):\n            raise ValueError("exclude_ids must be a JSON array")\n        output: set[int] = set()\n        for value in values[:200]:\n            try:\n                perfume_id = int(value)\n            except (TypeError, ValueError):\n                continue\n            if perfume_id > 0:\n                output.add(perfume_id)\n        return output\n\n    @staticmethod\n    def _has_excluded_term(metadata: dict[str, Any], excluded: list[str]) -> bool:\n        return any(metadata_has_term(metadata, term) for term in excluded)\n\n    @staticmethod\n    def _diversify(\n        items: list[dict[str, Any]],\n        top_k: int,\n        *,\n        brand_limit: int,\n        per_brand_limits: dict[str, int] | None = None,\n    ) -> list[dict[str, Any]]:\n        output: list[dict[str, Any]] = []\n        seen_ids: set[int] = set()\n        seen_labels: set[str] = set()\n        brand_counts: dict[str, int] = {}\n        normalized_limits = {\n            normalize_text(brand): int(limit)\n            for brand, limit in (per_brand_limits or {}).items()\n        }\n        for item in items:\n            perfume_id = int(item.get("perfume_id") or 0)\n            label = normalize_text(perfume_label(item.get("name"), item.get("brand")))\n            brand = normalize_text(item.get("brand"))\n            current_limit = normalized_limits.get(brand, brand_limit)\n            if (\n                perfume_id in seen_ids\n                or label in seen_labels\n                or brand_counts.get(brand, 0) >= current_limit\n            ):\n                continue\n            output.append(item)\n            seen_ids.add(perfume_id)\n            seen_labels.add(label)\n            brand_counts[brand] = brand_counts.get(brand, 0) + 1\n            if len(output) >= top_k:\n                break\n        return output\n\n    @staticmethod\n    def _public_catalog_row(row: dict[str, Any] | None) -> dict[str, Any] | None:\n        if not row:\n            return None\n        return {\n            "perfume_id": int(row["perfume_id"]),\n            "name": str(row.get("name") or ""),\n            "brand": str(row.get("brand") or ""),\n            "label": perfume_label(row.get("name"), row.get("brand")),\n            "gender": str(row.get("gender") or ""),\n            "year": row.get("year"),\n            "rating": row.get("rating"),\n            "popularity": int(row.get("popularity") or 0),\n            "longevity": row.get("longevity"),\n            "sillage": row.get("sillage"),\n            "value_score": row.get("value_score"),\n            "accords_csv": str(row.get("accords_csv") or ""),\n            "notes_csv": str(row.get("notes_csv") or ""),\n            "seasons_csv": str(row.get("seasons_csv") or ""),\n            "time_profile_csv": str(row.get("time_profile_csv") or ""),\n        }\n\n\nclass RetrievalRequestHandler(BaseHTTPRequestHandler):\n    engine: RetrievalEngine\n    server_version = "ScentAIRetrieval/1.0"\n\n    def do_GET(self) -> None:  # noqa: N802\n        if self.path == "/health":\n            self._send_json(HTTPStatus.OK, self.engine.health())\n            return\n        self._send_json(HTTPStatus.NOT_FOUND, {"error": "not_found"})\n\n    def do_POST(self) -> None:  # noqa: N802\n        try:\n            payload = self._read_json()\n            if self.path == "/search":\n                output = self.engine.search(payload)\n            elif self.path == "/resolve":\n                output = self.engine.resolve(payload)\n            elif self.path == "/similar":\n                output = self.engine.similar(payload)\n            else:\n                self._send_json(HTTPStatus.NOT_FOUND, {"error": "not_found"})\n                return\n            self._send_json(HTTPStatus.OK, output)\n        except LookupError as exc:\n            self._send_json(HTTPStatus.NOT_FOUND, {"error": str(exc)})\n        except (TypeError, ValueError) as exc:\n            self._send_json(HTTPStatus.BAD_REQUEST, {"error": str(exc)})\n        except Exception as exc:\n            self._send_json(HTTPStatus.INTERNAL_SERVER_ERROR, {"error": repr(exc)})\n\n    def _read_json(self) -> dict[str, Any]:\n        length = int(self.headers.get("Content-Length") or 0)\n        if length <= 0 or length > 1_000_000:\n            raise ValueError("Request body must contain at most 1 MB of JSON")\n        payload = json.loads(self.rfile.read(length).decode("utf-8"))\n        if not isinstance(payload, dict):\n            raise ValueError("JSON request must be an object")\n        return payload\n\n    def _send_json(self, status: HTTPStatus, payload: dict[str, Any]) -> None:\n        encoded = json.dumps(payload, ensure_ascii=False, allow_nan=False).encode("utf-8")\n        self.send_response(status.value)\n        self.send_header("Content-Type", "application/json; charset=utf-8")\n        self.send_header("Content-Length", str(len(encoded)))\n        self.end_headers()\n        self.wfile.write(encoded)\n\n    def log_message(self, format: str, *args: Any) -> None:\n        print(f"[retrieval-http] {self.address_string()} {format % args}", flush=True)\n\n\ndef main() -> None:\n    db_dir = Path(os.environ["SCENTAI_CHROMA_DIR"])\n    catalog_path = Path(os.environ["SCENTAI_CATALOG_PATH"])\n    host = os.environ.get("SCENTAI_RETRIEVAL_HOST", "127.0.0.1")\n    port = int(os.environ.get("SCENTAI_RETRIEVAL_PORT", "8020"))\n    engine = RetrievalEngine(db_dir, catalog_path)\n    RetrievalRequestHandler.engine = engine\n    server = ThreadingHTTPServer((host, port), RetrievalRequestHandler)\n    print(json.dumps({"event": "ready", "host": host, "port": port, **engine.health()}), flush=True)\n    server.serve_forever()\n\n\nif __name__ == "__main__":\n    main()\n'


import urllib.error
import urllib.request

SERVICE_PATH = Path("/content/scentai_retrieval_service.py")
SERVICE_LOG = Path("/content/scentai_retrieval_service.log")
SERVICE_PATH.write_text(SERVICE_SOURCE, encoding="utf-8")
subprocess.run([str(RETRIEVAL_PYTHON), "-m", "py_compile", str(SERVICE_PATH)], check=True)

RETRIEVAL_HOST = "127.0.0.1"
RETRIEVAL_PORT = 8020
RETRIEVAL_URL = f"http://{RETRIEVAL_HOST}:{RETRIEVAL_PORT}"


def get_json(url, *, timeout=10):
    with urllib.request.urlopen(url, timeout=timeout) as response:
        return json.loads(response.read().decode("utf-8"))


def service_is_healthy():
    try:
        return get_json(f"{RETRIEVAL_URL}/health", timeout=2).get("status") == "ok"
    except (OSError, urllib.error.URLError, json.JSONDecodeError):
        return False


def service_log_tail(line_count=80):
    if not SERVICE_LOG.exists():
        return "<service log has not been created>"
    return "\\n".join(
        SERVICE_LOG.read_text(encoding="utf-8", errors="replace").splitlines()[-line_count:]
    )


if service_is_healthy():
    print("Reusing healthy retrieval service.")
else:
    service_environment = os.environ.copy()
    service_environment.update({
        "PYTHONUNBUFFERED": "1",
        "ANONYMIZED_TELEMETRY": "False",
        "TOKENIZERS_PARALLELISM": "false",
        "HF_HOME": "/content/huggingface_retrieval_cache",
        "SCENTAI_CHROMA_DIR": str(LOCAL_CHROMA_DIR),
        "SCENTAI_CATALOG_PATH": str(LOCAL_CATALOG),
        "SCENTAI_RETRIEVAL_HOST": RETRIEVAL_HOST,
        "SCENTAI_RETRIEVAL_PORT": str(RETRIEVAL_PORT),
        "OMP_NUM_THREADS": str(min(os.cpu_count() or 2, 8)),
    })
    if HF_TOKEN:
        service_environment["HF_TOKEN"] = HF_TOKEN
    service_log_handle = SERVICE_LOG.open("w", encoding="utf-8")
    service_process = subprocess.Popen(
        [str(RETRIEVAL_PYTHON), str(SERVICE_PATH)],
        stdout=service_log_handle,
        stderr=subprocess.STDOUT,
        env=service_environment,
    )
    print("Starting BGE-M3 retrieval service. First model download/load can take several minutes.")
    deadline = time.monotonic() + 1800
    next_status = time.monotonic() + 30
    while not service_is_healthy():
        return_code = service_process.poll()
        if return_code is not None:
            service_log_handle.close()
            raise RuntimeError(
                f"Retrieval service exited with code {return_code}. Last log lines:\\n{service_log_tail()}"
            )
        if time.monotonic() >= deadline:
            service_process.terminate()
            service_log_handle.close()
            raise TimeoutError("Retrieval service startup timed out.\\n" + service_log_tail())
        if time.monotonic() >= next_status:
            print("Still loading...\\n" + service_log_tail(10))
            next_status = time.monotonic() + 30
        time.sleep(3)

health = get_json(f"{RETRIEVAL_URL}/health")
print(json.dumps(health, indent=2))
assert health["collection_count"] == 131930, health
assert health["catalog"]["perfumes"] == 131930, health
assert health["catalog"]["similarity_edges"] > 600000, health
print("Retrieval service preflight: passed")

## Start the isolated Gemma 4 + ScentAI LoRA vLLM service

In [ ]:
import os
import time
import urllib.error
import urllib.request

HOST = "127.0.0.1"
PORT = 8010
BASE_URL = f"http://{HOST}:{PORT}"
SERVER_LOG = Path("/content/scentai_vllm_server.log")
START_TIMEOUT_SECONDS = 2400


def get_json(url, *, timeout=10):
    with urllib.request.urlopen(url, timeout=timeout) as response:
        return json.loads(response.read().decode("utf-8"))


def server_is_healthy():
    try:
        with urllib.request.urlopen(f"{BASE_URL}/health", timeout=2) as response:
            return response.status == 200
    except (OSError, urllib.error.URLError):
        return False


def log_tail(line_count=80):
    if not SERVER_LOG.exists():
        return "<server log has not been created>"
    lines = SERVER_LOG.read_text(encoding="utf-8", errors="replace").splitlines()
    return "\n".join(lines[-line_count:])


lora_descriptor = json.dumps(
    {
        "name": LORA_NAME,
        "path": str(ADAPTER_DIR),
        "base_model_name": MODEL_NAME,
    }
)
server_command = [
    str(VLLM_EXECUTABLE),
    "serve",
    MODEL_NAME,
    "--host",
    HOST,
    "--port",
    str(PORT),
    "--dtype",
    "bfloat16",
    "--max-model-len",
    "4096",
    "--gpu-memory-utilization",
    "0.86",
    "--enable-prefix-caching",
    "--enable-lora",
    "--lora-modules",
    lora_descriptor,
    "--max-lora-rank",
    "16",
    "--max-loras",
    "1",
    "--generation-config",
    "vllm",
]

required_models = {MODEL_NAME, LORA_NAME}
if server_is_healthy():
    model_ids = {item["id"] for item in get_json(f"{BASE_URL}/v1/models").get("data", [])}
    if not required_models.issubset(model_ids):
        raise RuntimeError(
            f"Port {PORT} belongs to another server. Found {sorted(model_ids)}. "
            "Restart the Colab runtime and Run all."
        )
    print("Reusing healthy ScentAI vLLM server.")
else:
    server_environment = os.environ.copy()
    server_environment["PYTHONUNBUFFERED"] = "1"
    server_environment["HF_HOME"] = "/content/huggingface_cache"
    # The vLLM entry point uses the isolated interpreter, but child JIT processes
    # search PATH for build tools. Explicitly expose the environment's ninja.
    server_environment["PATH"] = (
        str(INFERENCE_ENV / "bin")
        + os.pathsep
        + server_environment.get("PATH", "")
    )
    if HF_TOKEN:
        server_environment["HF_TOKEN"] = HF_TOKEN

    server_log_handle = SERVER_LOG.open("w", encoding="utf-8")
    server_process = subprocess.Popen(
        server_command,
        stdout=server_log_handle,
        stderr=subprocess.STDOUT,
        env=server_environment,
    )
    print("Starting vLLM. The first base-model download and compile can take several minutes.")
    deadline = time.monotonic() + START_TIMEOUT_SECONDS
    next_status = time.monotonic() + 30
    while not server_is_healthy():
        return_code = server_process.poll()
        if return_code is not None:
            server_log_handle.close()
            raise RuntimeError(
                f"vLLM exited with code {return_code}. Last log lines:\n{log_tail()}"
            )
        if time.monotonic() >= deadline:
            server_process.terminate()
            server_log_handle.close()
            raise TimeoutError(
                f"vLLM was not healthy after {START_TIMEOUT_SECONDS}s. Last log lines:\n{log_tail()}"
            )
        if time.monotonic() >= next_status:
            print("Still loading...\n" + log_tail(10))
            next_status = time.monotonic() + 30
        time.sleep(3)

    model_ids = {item["id"] for item in get_json(f"{BASE_URL}/v1/models").get("data", [])}
    if not required_models.issubset(model_ids):
        server_process.terminate()
        server_log_handle.close()
        raise RuntimeError(
            f"Server started without both model IDs. Found {sorted(model_ids)}.\n{log_tail()}"
        )

print("Served models:", sorted(model_ids))
print("vLLM server preflight: passed")

## Prove the base and LoRA model endpoints

In [ ]:
def post_json(path, payload, *, timeout=600):
    request = urllib.request.Request(
        f"{BASE_URL}{path}",
        data=json.dumps(payload).encode("utf-8"),
        headers={"Content-Type": "application/json"},
        method="POST",
    )
    try:
        with urllib.request.urlopen(request, timeout=timeout) as response:
            return json.loads(response.read().decode("utf-8"))
    except urllib.error.HTTPError as exc:
        body = exc.read().decode("utf-8", errors="replace")
        raise RuntimeError(f"vLLM HTTP {exc.code}: {body}") from exc


def chat(model, messages, *, max_tokens=160):
    response = post_json(
        "/v1/chat/completions",
        {
            "model": model,
            "messages": messages,
            "temperature": 0.0,
            "max_tokens": max_tokens,
            "stream": False,
        },
    )
    choices = response.get("choices") or []
    if not choices:
        raise RuntimeError(f"vLLM returned no choices: {response}")
    content = str(choices[0].get("message", {}).get("content") or "").strip()
    if not content:
        raise RuntimeError(f"vLLM returned an empty answer: {response}")
    return content, response.get("usage") or {}


base_answer, base_usage = chat(
    MODEL_NAME,
    [{"role": "user", "content": "Reply with exactly: BASE_MODEL_OK"}],
    max_tokens=16,
)
print("Base response:", base_answer)
print("Base usage:", base_usage)

lora_smoke_prompt = """You are ScentAI, a grounded perfume consultant.
Use only the supplied perfume cards and do not invent facts.

[PERFUMES]
Versace Pour Homme by Versace
Accords: floral, musky, fresh spicy, citrus, aromatic, green, fresh
Best seasons: spring, summer
Time: day

Prada L'Homme by Prada
Accords: iris, powdery, clean, woody, amber
Best seasons: spring, summer, autumn
Time: day

Question: Recommend one clean office scent without vanilla and briefly explain why."""
lora_answer, lora_usage = chat(
    LORA_NAME,
    [{"role": "user", "content": lora_smoke_prompt}],
    max_tokens=180,
)
print("\nLoRA response:\n" + lora_answer)
print("LoRA usage:", lora_usage)
assert any(name in lora_answer for name in ("Versace Pour Homme", "Prada L'Homme")), (
    "The LoRA answered, but did not mention a supplied perfume."
)
print("\nCLEAN INFERENCE SMOKE TEST: PASSED")

## Load the dependency-free Stage 3 orchestrator

In [ ]:
ORCHESTRATOR_SOURCE = 'from __future__ import annotations\n\nimport json\nimport re\nimport time\nimport unicodedata\nimport urllib.error\nimport urllib.request\nfrom collections import OrderedDict\nfrom typing import Any\n\n\nPLANNER_INTENTS = {\n    "recommendation",\n    "similarity",\n    "alternative",\n    "comparison",\n    "perfume_profile",\n    "exact_lookup",\n    "unsupported_price",\n    "unsupported_availability",\n    "unsupported_medical",\n    "unsupported_social_claim",\n    "unsupported_layering",\n}\n\nUNSUPPORTED_ANSWERS = {\n    "unsupported_price": "I do not have a live price source, so I cannot safely answer current price or budget questions.",\n    "unsupported_availability": "I do not have a live stock source, so I cannot verify current availability or discontinued status.",\n    "unsupported_medical": "I cannot guarantee allergy or medical safety from perfume database fields. Check the ingredient label and consult a qualified professional when needed.",\n    "unsupported_social_claim": "The database does not record compliment outcomes, so I cannot make a grounded compliment claim.",\n    "unsupported_layering": "The database does not contain verified layering guidance, so I cannot make a grounded layering claim.",\n}\n\nUNSUPPORTED_ANSWERS_TR = {\n    "unsupported_price": "Canlı bir fiyat kaynağım olmadığı için güncel fiyat veya bütçe sorularını güvenilir biçimde yanıtlayamam.",\n    "unsupported_availability": "Canlı stok kaynağım olmadığı için güncel bulunabilirlik veya üretimden kalkma durumunu doğrulayamam.",\n    "unsupported_medical": "Parfüm veritabanı alanlarından alerji veya tıbbi güvenlik garantisi veremem. Gerektiğinde içerik etiketini kontrol edin ve yetkin bir uzmana danışın.",\n    "unsupported_social_claim": "Veritabanı iltifat sonucunu kaydetmediği için buna dayalı güvenilir bir iddiada bulunamam.",\n    "unsupported_layering": "Veritabanında doğrulanmış katmanlama bilgisi bulunmadığı için güvenilir bir katmanlama önerisi veremem.",\n}\n\nSTATIC_MESSAGES = {\n    "retrieval_error": {\n        "en": "The perfume retrieval service could not complete this request. No recommendation was generated.",\n        "tr": "Parfüm arama servisi bu isteği tamamlayamadı; bu nedenle bir öneri üretilmedi.",\n    },\n    "comparison_unresolved": {\n        "en": "I could not resolve two distinct perfume records for a grounded comparison.",\n        "tr": "Veriye dayalı bir karşılaştırma için iki farklı parfüm kaydını güvenilir biçimde eşleştiremedim.",\n    },\n    "no_safe_match": {\n        "en": "I could not find a safe grounded match after applying the requested constraints.",\n        "tr": "İstenen koşulları uyguladıktan sonra veriye dayalı güvenli bir eşleşme bulamadım.",\n    },\n}\n\nPLANNER_PROMPT = """You are the semantic planner for ScentAI, a grounded perfume assistant.\nRead the user\'s meaning rather than routing by fixed keywords. Return exactly one compact JSON object.\n\nSchema:\n{\n  "intent": "recommendation|similarity|alternative|comparison|perfume_profile|exact_lookup|unsupported_price|unsupported_availability|unsupported_medical|unsupported_social_claim|unsupported_layering",\n  "confidence": 0.0,\n  "semantic_query": "concise English occasion, mood, and wear-context search description",\n  "perfumes": [{"value": "catalog name or user wording", "evidence": "exact quote"}],\n  "requested_brand": {"value": "brand", "evidence": "exact quote"},\n  "gender": {"value": "male|female|unisex", "evidence": "exact quote"},\n  "season": {"value": "spring|summer|autumn|winter", "evidence": "exact quote"},\n  "time_profile": {"value": "day|night", "evidence": "exact quote"},\n  "wanted_terms": [{"value": "accord or note", "evidence": "exact quote"}],\n  "required_terms": [{"value": "required accord or note", "evidence": "exact quote"}],\n  "excluded_terms": [{"value": "accord, note, brand, or perfume", "evidence": "exact quote"}],\n  "requested_count": {"value": 3, "evidence": "exact quote"},\n  "requested_fields": [{"value": "rating|popularity|year|longevity|sillage|value|accords|notes|seasons|time", "evidence": "exact quote"}],\n  "discovery_mode": {"value": "balanced|mainstream|niche", "evidence": "exact quote"},\n  "conversation_action": "new_request|more_options|refine_previous"\n}\n\nRules:\n- Omit empty fields.\n- semantic_query is a model-written retrieval query, not a hard constraint. Keep occasion, mood, audience, season, and wear context; remove requested counts, perfume/brand names, required traits, and excluded traits. Example: \'date night fragrance with vanilla and warm spice\' becomes \'romantic date night fragrance\'.\n- Every populated constraint needs an exact quote copied from the user.\n- Term values must use the English database taxonomy even when evidence is in another language. For example, Turkish \'vanilya\' maps to \'vanilla\'; a warm/date-night use of \'baharatlı\' maps to \'warm spicy\'.\n- Put a trait in required_terms when the user says it must be present (for example \'olsun\', \'must have\', or \'with vanilla\'). Use wanted_terms for softer preferences.\n- A noun-category request such as \'rose fragrances\', \'coconut perfumes\', or \'iris scents\' makes that trait required. Turkish \'-li/-lı\' trait wording and \'X karakterli\' also make it required unless the wording is explicitly soft.\n- Mood and style descriptions such as clean, unusual, skin scent, softer amber, polished, or sporty are retrieval preferences, not required database traits unless the user explicitly says that exact accord or note must be present.\n- A negative form such as without vanilla, less leathery, misk icermeyen, or vanilyasiz must never also appear in required_terms.\n- Extract every item in a coordinated exclusion list. For \'no vanilla, oud, or smoke\', return all three exclusions rather than only the first.\n- discovery_mode is mainstream only when the user explicitly prioritizes famous, popular, recognizable, or mass-appeal choices; niche only when they explicitly request hidden gems, obscure, unusual, or niche discovery. Otherwise omit it and balanced will be used.\n- Use conversation_action only when conversation context is supplied. more_options means the user wants different recommendations under the previous constraints. refine_previous means the user modifies the previous request. Otherwise use new_request.\n- For more_options or refine_previous, extract only constraints explicitly present in the current message; the application will safely inherit the remaining previous constraints.\n- Never invent a perfume, brand, or hard exclusion.\n- Set requested_brand only when the wording clearly identifies a perfume house or brand. A style word such as clean, fresh, rose, musk, or office is not a brand by itself.\n- similarity means smell/profile similarity; alternative means a replacement or dupe.\n- comparison requires two perfumes and asks for differences, choice, or contrast.\n- perfume_profile asks what one perfume is like, its vibe, performance, versatility, or wear situations.\n- exact_lookup asks for literal database fields rather than interpretation.\n- Any request that asks which perfume is guaranteed/certain to receive compliments or attract someone is unsupported_social_claim. Ordinary date-night or popular-fragrance advice is still recommendation.\n- Do not use Markdown or explanatory text outside the JSON object."""\n\nLEGACY_ANSWER_PROMPT = """You are ScentAI, a warm and perceptive perfume consultant.\nAnswer in the same language as the user. Use only the supplied database cards.\n\nGrounding rules:\n- Recommend or discuss only perfume names printed in the supplied cards.\n- Never invent notes, accords, ratings, votes, years, performance, seasons, or use cases.\n- Treat character and vibe as careful interpretations of the printed accords, notes, season, time, longevity, and sillage. Phrase interpretations naturally, not as certainty beyond the card.\n- Obey every excluded term. Do not mention an excluded candidate even to say it was excluded.\n- For recommendations, explain character, suitable setting, practical wear, and a meaningful tradeoff. Do not write like a search engine.\n- For comparisons, discuss both perfumes separately, explain their different character and likely use, then summarize the practical choice. Do not choose a winner unless the user asks.\n- Avoid repetitive templates such as \'it matches through\', \'I would include it for\', or a bare list of database fields.\n- Do not use rigid section labels such as \'Neden\', \'Karakter\', \'Pratik Kullanım\', or \'Önemli Bir Uzlaşı\'. Give each recommendation one natural, compact paragraph.\n- Keep the answer concise. Use numbered recommendations when recommending multiple perfumes.\n- When a requested number is supplied, recommend exactly that many distinct perfumes."""\n\n\nADVISOR_ANSWER_PROMPT = """You are ScentAI, a perceptive perfume consultant helping a real person make a choice.\nAnswer in the same language as the user. The database cards are a verified shortlist, not a ranking you must copy.\n\nYour job:\n- First understand the experience the user is seeking: mood, identity, setting, practicality, and tradeoffs.\n- Select the strongest options from the cards using holistic fit. A lower-ranked card may be the best recommendation.\n- Give each recommendation a distinct role. Explain how it feels different from the other choices and who or what situation it suits.\n- Turn accords, notes, season, time, longevity, and sillage into useful advice. Do not merely recite fields.\n- Treat each card\'s longevity and sillage calibration as authoritative. Never describe a strong or very strong score as moderate, light, weak, restrained, or intimate; never describe a light or restrained score as strong or room-filling.\n- If a performance value is not recorded, do not infer longevity or projection. Never convert a score into promised hours, guaranteed room projection, compliments, or social outcomes.\n- Offer real decision support: indicate the safer choice, the more characterful direction, or the key compromise when those distinctions are supported by the cards.\n- Use careful interpretive language for vibe and character. These are grounded readings of the recorded profile, not objective facts.\n- Sound natural and engaged. Vary sentence structure and avoid repeating a fixed paragraph template.\n- Write one cohesive consultation, not three database-card summaries. Later recommendations should contrast with or build on earlier ones instead of restarting the same explanation.\n- Use a different grammatical opening and rhetorical purpose for every numbered item. Vary sentence count and rhythm as well as vocabulary.\n- Select only the decisive evidence for each option. Do not enumerate accords and notes in every paragraph.\n- Avoid stock constructions such as \'I included this because\', \'this option stands out\', \'bu secenegi buraya koyuyorum\', \'bir adim one cikar\', \'notalarla desteklenen bir yapi\', and repeated \'profil ciziyor\' sentences.\n- In Turkish, use natural Turkish descriptions instead of calling accords \'kodlar\' or ingredients \'malzemeler\'. Keep exact perfume names unchanged.\n\nGrounding boundaries:\n- Recommend or discuss only exact perfume names printed in the supplied cards.\n- Never invent notes, accords, ratings, votes, years, performance, seasons, availability, price, or social outcomes.\n- Obey every required and excluded term. Never mention an excluded candidate, even as a warning.\n- Do not assume retrieval order equals recommendation order.\n- If the cards do not support a claim, leave it out.\n- When comparing, discuss both perfumes\' character and likely use, then explain the practical difference. Do not choose a winner unless asked.\n- When recommending multiple perfumes, use a numbered list with exactly the requested number of distinct perfumes.\n- Give each perfume one compact, natural paragraph. Do not use rigid labels such as Why, Character, Practical Use, Tradeoff, Neden, Karakter, or Best pick.\n- End with a short, situational decision cue when useful; do not repeat the recommendation list.\n- Keep a three-recommendation answer under roughly 320 words. Do not add a greeting unless the conversation genuinely calls for one."""\n\n\nANSWER_PROMPT = ADVISOR_ANSWER_PROMPT\n\n\ndef normalize_text(value: Any) -> str:\n    text = str(value or "").lower().replace("ı", "i")\n    text = unicodedata.normalize("NFKD", text)\n    text = "".join(char for char in text if not unicodedata.combining(char))\n    text = re.sub(r"[^a-z0-9]+", " ", text)\n    return " ".join(text.split())\n\n\nTURKISH_LANGUAGE_MARKERS = {\n    "aksam", "ama", "anlat", "bana", "baska", "ben", "bir", "bisey", "bu",\n    "daha", "dusun", "dusunuyorsun", "erkek", "ferah", "gibi", "gunduz",\n    "hakkinda", "icin", "ile", "istiyorum", "kadar", "kadin", "kis", "mi",\n    "miyim", "nasil", "ne", "neden", "olan", "olarak", "olsun", "oner",\n    "oneri", "sence", "tane", "ve", "veya", "yaz",\n}\nENGLISH_LANGUAGE_MARKERS = {\n    "a", "about", "and", "another", "are", "best", "compare", "for", "give",\n    "how", "i", "in", "is", "like", "me", "more", "of", "recommend", "scent",\n    "something", "than", "the", "this", "to", "what", "which", "with", "without",\n}\n\n\ndef language_marker_scores(value: Any) -> tuple[int, int]:\n    raw = str(value or "").lower()\n    tokens = normalize_text(raw).split()\n    turkish_score = sum(token in TURKISH_LANGUAGE_MARKERS for token in tokens)\n    english_score = sum(token in ENGLISH_LANGUAGE_MARKERS for token in tokens)\n    if re.search(r"[çğıöşü]", raw):\n        turkish_score += 3\n    return turkish_score, english_score\n\n\ndef detect_response_language(query: str, context: dict[str, Any] | None = None) -> str:\n    """Resolve Turkish/English output locally so prompt examples cannot choose it."""\n    normalized = normalize_text(query)\n    if re.search(r"\\b(?:answer|respond|reply) in english\\b", normalized) or re.search(\n        r"\\bingilizce (?:cevap|yanit|yanitla|yaz)\\b", normalized\n    ):\n        return "en"\n    if re.search(r"\\b(?:answer|respond|reply) in turkish\\b", normalized) or re.search(\n        r"\\bturkce (?:cevap|yanit|yanitla|yaz)\\b", normalized\n    ):\n        return "tr"\n    turkish_score, english_score = language_marker_scores(query)\n    if turkish_score > english_score:\n        return "tr"\n    if english_score > turkish_score:\n        return "en"\n    previous_language = str((context or {}).get("previous_response_language") or "").lower()\n    return previous_language if previous_language in {"en", "tr"} else "en"\n\n\ndef output_language_instruction(language: str) -> str:\n    name = "Turkish" if language == "tr" else "English"\n    return (\n        f"{name} ({language}) is the required response language. "\n        f"Write every explanatory sentence in {name}. The language of database cards, "\n        "earlier responses, examples, or taxonomy terms must not override this requirement. "\n        "Keep perfume and brand names exactly as printed in the cards."\n    )\n\n\ndef response_language_matches(answer: str, expected_language: str) -> bool:\n    """Reject only clear language mismatches; catalog names may remain multilingual."""\n    if len(normalize_text(answer).split()) < 8:\n        return True\n    turkish_score, english_score = language_marker_scores(answer)\n    if expected_language == "tr":\n        return not (english_score >= 4 and english_score >= turkish_score + 3)\n    return not (turkish_score >= 4 and turkish_score >= english_score + 2)\n\n\ndef static_message(key: str, language: str) -> str:\n    return STATIC_MESSAGES[key]["tr" if language == "tr" else "en"]\n\n\ndef unsupported_answer(intent: str, language: str) -> str | None:\n    answers = UNSUPPORTED_ANSWERS_TR if language == "tr" else UNSUPPORTED_ANSWERS\n    return answers.get(intent)\n\n\ndef parse_json_object(raw: str) -> dict[str, Any]:\n    start = raw.find("{")\n    if start < 0:\n        raise ValueError("Planner returned no JSON object")\n    payload, _ = json.JSONDecoder().raw_decode(raw[start:])\n    if not isinstance(payload, dict):\n        raise ValueError("Planner JSON must be an object")\n    return payload\n\n\ndef evidence_supported(evidence: Any, query: str) -> bool:\n    evidence_norm = normalize_text(evidence)\n    query_norm = normalize_text(query)\n    return bool(evidence_norm and f" {evidence_norm} " in f" {query_norm} ")\n\n\ndef evidenced_values(raw: Any, query: str, *, limit: int = 20) -> list[Any]:\n    values = raw if isinstance(raw, list) else ([] if raw is None else [raw])\n    output: list[Any] = []\n    for item in values:\n        if not isinstance(item, dict) or not evidence_supported(item.get("evidence"), query):\n            continue\n        value = item.get("value")\n        if value is None or value == "":\n            continue\n        output.append(value)\n        if len(output) >= limit:\n            break\n    return output\n\n\ndef first_evidenced_value(raw: Any, query: str) -> Any | None:\n    values = evidenced_values(raw, query, limit=1)\n    return values[0] if values else None\n\n\ndef explicit_brand_context(query: str, brand: str, evidence: str) -> bool:\n    query_norm = normalize_text(query)\n    brand_norm = normalize_text(brand)\n    evidence_norm = normalize_text(evidence)\n    if not brand_norm or not evidence_norm or evidence_norm not in query_norm:\n        return False\n    escaped = re.escape(brand_norm)\n    patterns = (\n        rf"\\b(?:from|by|brand|house|house of)\\s+{escaped}\\b",\n        rf"\\b{escaped}\\s+(?:brand|house|perfume|perfumes|fragrance|fragrances|cologne|colognes|scent|scents|parfum|parfumu|parfumleri)\\b",\n        rf"\\b{escaped}\\s+(?:den|dan|ten|tan)\\b",\n    )\n    return any(re.search(pattern, query_norm) for pattern in patterns)\n\n\ndef canonical_trait_term(value: Any) -> str:\n    normalized = normalize_text(value)\n    aliases = {\n        "vanilya": "vanilla",\n        "vanila": "vanilla",\n        "baharatli": "spicy",\n        "baharatl": "spicy",\n        "baharat": "spicy",\n        "tutun": "tobacco",\n        "deri": "leather",\n        "misk": "musk",\n        "duman": "smoky",\n        "dumanli": "smoky",\n        "gul": "rose",\n        "tatli": "sweet",\n        "paculi": "patchouli",\n        "tutsu": "incense",\n        "kahve": "coffee",\n        "karamel": "caramel",\n        "ananas": "pineapple",\n        "hindistan cevizi": "coconut",\n        "kehribar": "amber",\n        "aromatik": "aromatic",\n        "hayvansi": "animalic",\n        "tarcin": "cinnamon",\n        "pudrali": "powdery",\n        "yesil": "green",\n        "ciceksi": "floral",\n        "odunsu": "woody",\n        "narenciye": "citrus",\n        "leathery": "leather",\n        "musky": "musk",\n        "smoke": "smoky",\n        "agarwood": "oud",\n        "aoud": "oud",\n    }\n    return aliases.get(normalized, normalized)\n\n\ndef hard_trait_requested(query: str, evidence: Any, trait: Any = None) -> bool:\n    """Recover explicit hard-positive wording without making soft tastes strict."""\n    query_norm = normalize_text(query)\n    evidence_norm = normalize_text(evidence)\n    trait_norm = canonical_trait_term(trait if trait is not None else evidence)\n    if not evidence_norm or evidence_norm not in query_norm:\n        return False\n    negative_patterns = (\n        r"\\b(?:without|avoid|avoiding|exclude|excluding|less|olmayan|icermeyen|icermesin)\\b",\n        r"\\b\\w+s[ıiuu]z\\b",\n    )\n    if any(re.search(pattern, evidence_norm) for pattern in negative_patterns):\n        return False\n    if any(token.endswith(("li", "lu")) for token in evidence_norm.split()):\n        return True\n    if re.search(r"\\b(?:must|must have|required|requires|containing|contains|mutlaka|olsun|iceren|bulunan|karakterli|agirlikli)\\b", evidence_norm):\n        return True\n\n    style_descriptors = {\n        "clean", "unusual", "skin scent", "soft amber", "sporty",\n        "professional", "polished", "versatile", "characterful",\n    }\n\n    escaped = re.escape(evidence_norm)\n    for match in re.finditer(rf"\\b{escaped}\\b", query_norm):\n        before = " ".join(query_norm[:match.start()].split()[-4:])\n        after = " ".join(query_norm[match.end():].split()[:7])\n        if re.search(r"\\b(?:prefer|preferably|ideally|maybe|perhaps|tercihen|mumkunse)\\b", before):\n            continue\n        if re.search(r"\\b(?:softer|lighter|less|daha yumusak|daha hafif)\\b", f"{before} {evidence_norm} {after}"):\n            continue\n        if re.search(r"\\b(?:must have|must include|must be|with|containing|contains|requires|required)\\s+(?:a|an|the)?\\s*$", before):\n            return True\n        if re.match(r"^(?:must|required|mutlaka|olsun|iceren|bulunan|karakterli|agirlikli)\\b", after):\n            return True\n        # A trailing hard marker scopes over coordinated traits:\n        # "vanilla and spice must be present" / "vanilya ve baharat mutlaka bulunan".\n        if re.match(\n            r"^(?:(?:and|ve|ile)\\s+){1,2}.{0,50}\\b"\n            r"(?:must|required|mutlaka|olsun|iceren|bulunan|karakterli|agirlikli)\\b",\n            after,\n        ):\n            return True\n        if trait_norm in style_descriptors:\n            continue\n        if re.match(r"^(?:fragrance|fragrances|perfume|perfumes|scent|scents|parfum|parfumu|parfumler|koku)\\b", after):\n            return True\n        # Planner evidence may include the category noun ("rose fragrances")\n        # instead of returning only the trait token.\n        if re.search(\n            r"\\b(?:fragrance|fragrances|perfume|perfumes|scent|scents|parfum|parfumu|parfumler|koku)\\b",\n            evidence_norm,\n        ):\n            return True\n    return False\n\n\ndef explicit_unsupported_intent(query: str) -> str | None:\n    """Deterministic safety recovery for claims the catalog cannot support."""\n    normalized = f" {normalize_text(query)} "\n    social_terms = (" compliment ", " compliments ", " iltifat ", " attract ", " attraction ")\n    if any(term in normalized for term in social_terms):\n        return "unsupported_social_claim"\n    return None\n\n\ndef infer_explicit_audience(query: str) -> str | None:\n    normalized = f" {normalize_text(query)} "\n    if any(term in normalized for term in (" unisex ", " gender neutral ", " cinsiyetsiz ")):\n        return "unisex"\n    if any(term in normalized for term in (" women ", " women s ", " woman ", " female ", " kadin ", " kadinlar ")):\n        return "female"\n    if any(term in normalized for term in (" men ", " men s ", " man ", " male ", " erkek ", " erkekler ")):\n        return "male"\n    return None\n\n\ndef infer_explicit_season(query: str) -> str | None:\n    normalized = f" {normalize_text(query)} "\n    aliases = {\n        "spring": (" spring ", " ilkbahar "),\n        "summer": (" summer ", " yaz "),\n        "autumn": (" autumn ", " fall ", " sonbahar "),\n        "winter": (" winter ", " kis "),\n    }\n    matches = [season for season, terms in aliases.items() if any(term in normalized for term in terms)]\n    return matches[0] if len(matches) == 1 else None\n\n\ndef infer_explicit_time_profile(query: str) -> str | None:\n    normalized = f" {normalize_text(query)} "\n    if any(term in normalized for term in (" night ", " nighttime ", " evening ", " date ", " gece ", " aksam ")):\n        return "night"\n    if any(term in normalized for term in (" daytime ", " day wear ", " office ", " gym ", " gunduz ", " ofis ")):\n        return "day"\n    return None\n\n\ndef infer_coordinated_exclusions(query: str) -> list[str]:\n    """Recover explicit English negative lists without a fixed trait vocabulary."""\n    normalized = str(query or "").lower().replace("ı", "i")\n    normalized = unicodedata.normalize("NFKD", normalized)\n    normalized = "".join(char for char in normalized if not unicodedata.combining(char))\n    normalized = re.sub(r"[^a-z0-9,.;]+", " ", normalized)\n    normalized = " ".join(normalized.split())\n    match = re.search(\n        r"\\b(?:without|no|avoid|avoiding|exclude|excluding)\\b\\s+(.{1,120}?)"\n        r"(?:\\b(?:accord|accords|note|notes)\\b|[.;]|$)",\n        normalized,\n    )\n    if not match:\n        return []\n    phrase = match.group(1).strip()\n    # Stop at a new imperative clause instead of treating the command as a\n    # scent trait: "exclude tobacco and show three different choices".\n    phrase = re.split(\n        r"(?:\\b(?:and|or)\\s+|[,;]\\s*)(?=(?:show|recommend|give|suggest|list|find|provide|keep|make|tell)\\b)",\n        phrase,\n        maxsplit=1,\n    )[0].strip()\n    if "," not in phrase and not re.search(r"\\b(?:and|or)\\b", phrase):\n        return []\n    parts = re.split(r"\\s*,\\s*|\\b(?:and|or)\\b", phrase)\n    noise = {"a", "all", "any", "anything", "kind", "kinds", "of", "the"}\n    output: list[str] = []\n    for part in parts:\n        tokens = [token for token in normalize_text(part).split() if token not in noise]\n        if not tokens or len(tokens) > 4:\n            continue\n        term = canonical_trait_term(" ".join(tokens))\n        if term:\n            output.append(term)\n    return list(dict.fromkeys(output))\n\n\ndef infer_explicit_exclusions(query: str) -> list[str]:\n    """Recover simple explicit negatives when the planner omits them.\n\n    The extraction is grammar based rather than tied to a fixed fragrance-trait\n    vocabulary, so new catalog terms receive the same treatment.\n    """\n    normalized = normalize_text(query)\n    if not normalized:\n        return []\n\n    output: list[str] = []\n\n    def add(value: str) -> None:\n        tokens = [\n            token for token in normalize_text(value).split()\n            if token not in {\n                "a", "an", "any", "the", "accord", "accords", "note", "notes",\n                "akor", "nota", "pls", "please", "all", "kind", "kinds", "type", "types", "of",\n                "heavy", "strong", "prominent", "dominant", "noticeable",\n            }\n        ]\n        if not tokens or len(tokens) > 3:\n            return\n        if tokens[0].endswith("ing") or tokens[0] in {\n            "declaring", "choosing", "naming", "recommending", "showing", "giving",\n            "need", "preference", "more", "fewer", "doubt",\n        }:\n            return\n        if any(\n            token in {\n                "winner", "choice", "choices", "option", "options", "price", "budget",\n                "longevity", "projection", "sillage", "performance",\n            }\n            or token.isdigit()\n            for token in tokens\n        ):\n            return\n        term = canonical_trait_term(" ".join(tokens))\n        if term:\n            output.append(term)\n\n    # English: "not sweet", "without vanilla", "must not contain musk".\n    for match in re.finditer(\n        r"\\b(?:without|no)\\s+"\n        r"([a-z0-9]+(?:\\s+[a-z0-9]+){0,2}?)"\n        r"(?=\\s+(?:accord|accords|note|notes|fragrance|fragrances|perfume|perfumes|scent|scents|and|or|but|with|for)|[,.;]|$)",\n        normalized,\n    ):\n        add(match.group(1))\n    for match in re.finditer(\n        r"\\b(?:must\\s+)?not\\s+(?:contain|include|have)\\s+([a-z0-9]+(?:\\s+[a-z0-9]+){0,2}?)"\n        r"(?=\\s+(?:accord|accords|note|notes|and|or|but|with|for)|[,.;]|$)",\n        normalized,\n    ):\n        add(match.group(1))\n    for match in re.finditer(\n        r"\\bnot\\s+(?!too\\b|very\\b|overly\\b)([a-z0-9]+)"\n        r"(?=\\s+(?:accord|accords|note|notes|fragrance|fragrances|perfume|perfumes|scent|scents|and|or|but|with|for)|[,.;]|$)",\n        normalized,\n    ):\n        add(match.group(1))\n    for match in re.finditer(r"\\b(?:less|except)\\s+([a-z0-9]+)\\b", normalized):\n        add(match.group(1))\n    for match in re.finditer(r"\\b([a-z0-9]+(?:\\s+[a-z0-9]+){0,1})\\s+free\\b", normalized):\n        add(match.group(1))\n    for match in re.finditer(\n        r"\\b(?:avoid|exclude|excluding)\\s+"\n        r"(?:(?:anything\\s+with|all\\s+(?:kinds?|types?)\\s+of)\\s+)?([a-z0-9]+)\\b",\n        normalized,\n    ):\n        add(match.group(1))\n    for match in re.finditer(\n        r"\\b(?:do\\s+not|dont|don\\s+t)\\s+want\\s+(?:a\\s+|an\\s+|any\\s+)?"\n        r"([a-z0-9]+(?:\\s+[a-z0-9]+){0,1})\\b",\n        normalized,\n    ):\n        add(match.group(1))\n    for match in re.finditer(r"\\bnothing\\s+([a-z0-9]+)\\b", normalized):\n        add(match.group(1))\n\n    # Turkish and mixed Turkish/English: "sweet olmayan", "misk icermeyen".\n    for match in re.finditer(\n        r"\\b([a-z0-9]+)\\s+(?:olmayan|icermeyen|icermesin|barindirmayan|barindirmasin|istemiyorum)\\b",\n        normalized,\n    ):\n        add(match.group(1))\n    for match in re.finditer(r"\\b([a-z0-9]+)\\s+haric\\b", normalized):\n        add(match.group(1))\n\n    # Productive Turkish privative suffixes after ASCII normalization.\n    for token in normalized.split():\n        match = re.fullmatch(r"([a-z0-9]{3,}?)(?:siz|suz)", token)\n        if match:\n            add(match.group(1))\n\n    unique = list(dict.fromkeys(output))\n    atomic = {term for term in unique if " " not in term}\n    return [\n        term for term in unique\n        if " " not in term or not all(part in atomic for part in term.split())\n    ]\n\n\ndef infer_explicit_requirements(query: str) -> list[str]:\n    """Recover only unambiguously hard-positive grammar omitted by the planner."""\n    normalized = normalize_text(query)\n    if not normalized:\n        return []\n    output: list[str] = []\n\n    def add(value: str) -> None:\n        tokens = [\n            token for token in normalize_text(value).split()\n            if token not in {\n                "akor", "accord", "nota", "note", "icin", "for",\n                "ve", "ile", "ama", "daha",\n            }\n        ]\n        term = canonical_trait_term(" ".join(tokens))\n        if term:\n            output.append(term)\n\n    for match in re.finditer(\n        r"\\bmust\\s+(?:have|include|contain|feature)\\s+"\n        r"(?:a\\s+|an\\s+|the\\s+)?([a-z0-9]+(?:\\s+[a-z0-9]+){0,1}?)"\n        r"(?=\\s+(?:accord|accords|note|notes|and|or|but|without|for)|$)",\n        normalized,\n    ):\n        add(match.group(1))\n\n    for match in re.finditer(\n        r"\\b([a-z0-9]+(?:\\s+[a-z0-9]+)?)(?:\\s+(?:ve|ile)\\s+([a-z0-9]+(?:\\s+[a-z0-9]+)?))?"\n        r"\\s+(?:akor\\s+|nota\\s+)?mutlaka\\s+bulunan\\b",\n        normalized,\n    ):\n        add(match.group(1))\n        if match.group(2):\n            add(match.group(2))\n\n    for match in re.finditer(\n        r"\\b([a-z0-9]+(?:\\s+[a-z0-9]+)?)\\s+(?:iceren|karakterli|agirlikli)\\b",\n        normalized,\n    ):\n        add(match.group(1))\n    for match in re.finditer(r"\\b([a-z0-9]+)\\s+(?:olsun|icersin|barindirsin)\\b", normalized):\n        add(match.group(1))\n\n    return list(dict.fromkeys(output))\n\n\ndef normalize_plan(raw_plan: dict[str, Any], query: str) -> dict[str, Any]:\n    intent = normalize_text(raw_plan.get("intent")).replace(" ", "_")\n    if intent not in PLANNER_INTENTS:\n        intent = "recommendation"\n    try:\n        confidence = min(max(float(raw_plan.get("confidence") or 0.0), 0.0), 1.0)\n    except (TypeError, ValueError):\n        confidence = 0.0\n\n    required_items = raw_plan.get("required_terms")\n    required_items = required_items if isinstance(required_items, list) else []\n    accepted_required: list[str] = []\n    downgraded_required: list[str] = []\n    for item in required_items:\n        if not isinstance(item, dict) or not evidence_supported(item.get("evidence"), query):\n            continue\n        term = canonical_trait_term(item.get("value"))\n        if not term:\n            continue\n        target = accepted_required if hard_trait_requested(query, item.get("evidence"), term) else downgraded_required\n        target.append(term)\n\n    plan: dict[str, Any] = {\n        "intent": intent,\n        "confidence": confidence,\n        "perfumes": [str(value).strip() for value in evidenced_values(raw_plan.get("perfumes"), query, limit=3)],\n        "wanted_terms": [canonical_trait_term(value) for value in evidenced_values(raw_plan.get("wanted_terms"), query)],\n        "required_terms": accepted_required,\n        "excluded_terms": [canonical_trait_term(value) for value in evidenced_values(raw_plan.get("excluded_terms"), query)],\n        "requested_fields": [normalize_text(value) for value in evidenced_values(raw_plan.get("requested_fields"), query)],\n    }\n    wanted_items = raw_plan.get("wanted_terms")\n    wanted_items = wanted_items if isinstance(wanted_items, list) else []\n    promoted_required = [\n        canonical_trait_term(item.get("value"))\n        for item in wanted_items\n        if isinstance(item, dict)\n        and evidence_supported(item.get("evidence"), query)\n        and hard_trait_requested(query, item.get("evidence"), item.get("value"))\n    ]\n    plan["required_terms"] = list(dict.fromkeys([\n        *plan["required_terms"],\n        *filter(None, promoted_required),\n        *infer_explicit_requirements(query),\n    ]))\n    if any(term.endswith(" spicy") for term in plan["required_terms"]):\n        plan["required_terms"] = [term for term in plan["required_terms"] if term != "spicy"]\n    plan["wanted_terms"] = list(dict.fromkeys([\n        *plan["wanted_terms"],\n        *downgraded_required,\n    ]))\n    plan["wanted_terms"] = [\n        term for term in plan["wanted_terms"] if term not in plan["required_terms"]\n    ]\n    plan["excluded_terms"] = list(dict.fromkeys([\n        *plan["excluded_terms"],\n        *infer_coordinated_exclusions(query),\n        *infer_explicit_exclusions(query),\n    ]))\n    atomic_exclusions = {term for term in plan["excluded_terms"] if " " not in term}\n    plan["excluded_terms"] = [\n        term for term in plan["excluded_terms"]\n        if " " not in term or not all(part in atomic_exclusions for part in term.split())\n    ]\n    semantic_query = str(raw_plan.get("semantic_query") or "").strip()\n    if 3 <= len(semantic_query) <= 240 and "\\n" not in semantic_query:\n        plan["semantic_query"] = semantic_query\n    for field, allowed in {\n        "gender": {"male", "female", "unisex"},\n        "season": {"spring", "summer", "autumn", "winter"},\n        "time_profile": {"day", "night"},\n    }.items():\n        value = normalize_text(first_evidenced_value(raw_plan.get(field), query))\n        if value in allowed:\n            plan[field] = value\n    inferred_constraints = {\n        "gender": infer_explicit_audience(query),\n        "season": infer_explicit_season(query),\n        "time_profile": infer_explicit_time_profile(query),\n    }\n    for field, value in inferred_constraints.items():\n        if field not in plan and value:\n            plan[field] = value\n\n    count = first_evidenced_value(raw_plan.get("requested_count"), query)\n    try:\n        count_value = int(count)\n    except (TypeError, ValueError):\n        count_value = 0\n    if 1 <= count_value <= 5:\n        plan["requested_count"] = count_value\n    elif intent in {"recommendation", "similarity", "alternative"}:\n        inferred_count = infer_requested_count(query)\n        if inferred_count is not None:\n            plan["requested_count"] = inferred_count\n\n    brand_item = raw_plan.get("requested_brand")\n    if isinstance(brand_item, dict) and evidence_supported(brand_item.get("evidence"), query):\n        brand = str(brand_item.get("value") or "").strip()\n        evidence = str(brand_item.get("evidence") or "")\n        if explicit_brand_context(query, brand, evidence):\n            brand_field = "reference_brand" if intent in {"similarity", "alternative"} and plan["perfumes"] else "requested_brand"\n            plan[brand_field] = brand\n\n    plan["wanted_terms"] = list(dict.fromkeys(filter(None, plan["wanted_terms"])))\n    plan["required_terms"] = list(dict.fromkeys(filter(None, plan["required_terms"])))\n    plan["excluded_terms"] = list(dict.fromkeys(filter(None, plan["excluded_terms"])))\n    excluded = set(plan["excluded_terms"])\n    plan["required_terms"] = [term for term in plan["required_terms"] if term not in excluded]\n    plan["wanted_terms"] = [term for term in plan["wanted_terms"] if term not in excluded]\n    plan["requested_fields"] = list(dict.fromkeys(filter(None, plan["requested_fields"])))\n    discovery_item = raw_plan.get("discovery_mode")\n    if isinstance(discovery_item, dict) and evidence_supported(discovery_item.get("evidence"), query):\n        discovery_mode = normalize_text(discovery_item.get("value"))\n        if discovery_mode in {"balanced", "mainstream", "niche"}:\n            plan["discovery_mode"] = discovery_mode\n    if "discovery_mode" not in plan:\n        inferred_discovery_mode = infer_discovery_mode(query)\n        if inferred_discovery_mode:\n            plan["discovery_mode"] = inferred_discovery_mode\n    conversation_action = normalize_text(raw_plan.get("conversation_action")).replace(" ", "_")\n    if conversation_action in {"new_request", "more_options", "refine_previous"}:\n        plan["conversation_action"] = conversation_action\n    else:\n        plan["conversation_action"] = "new_request"\n    unsupported_intent = explicit_unsupported_intent(query)\n    if unsupported_intent:\n        plan["intent"] = unsupported_intent\n        plan["perfumes"] = []\n    return plan\n\n\ndef inherit_conversation_plan(plan: dict[str, Any], context: dict[str, Any] | None) -> dict[str, Any]:\n    """Apply model-decided follow-up semantics without keyword routing."""\n    if not context or plan.get("conversation_action") not in {"more_options", "refine_previous"}:\n        return plan\n    previous = context.get("previous_plan")\n    if not isinstance(previous, dict):\n        return plan\n    merged = dict(plan)\n    action = merged["conversation_action"]\n    if action == "more_options" and previous.get("semantic_query"):\n        # Phrases such as "other options" carry no new scent semantics; keep\n        # the previous retrieval meaning even if the planner emitted a generic\n        # paraphrase for the follow-up sentence.\n        merged["semantic_query"] = previous["semantic_query"]\n    for key in (\n        "semantic_query",\n        "requested_brand",\n        "reference_brand",\n        "gender",\n        "season",\n        "time_profile",\n        "requested_count",\n        "discovery_mode",\n    ):\n        if not merged.get(key) and previous.get(key):\n            merged[key] = previous[key]\n    current_required = set(plan.get("required_terms", []))\n    current_excluded = set(plan.get("excluded_terms", []))\n    for key in ("wanted_terms", "required_terms", "excluded_terms"):\n        merged[key] = list(dict.fromkeys([\n            *previous.get(key, []),\n            *merged.get(key, []),\n        ]))\n    # A new negative instruction overrides an old positive preference and a\n    # newly required trait overrides an old exclusion.\n    merged["wanted_terms"] = [term for term in merged["wanted_terms"] if term not in current_excluded]\n    merged["required_terms"] = [term for term in merged["required_terms"] if term not in current_excluded]\n    merged["excluded_terms"] = [term for term in merged["excluded_terms"] if term not in current_required]\n    if merged.get("intent") == "recommendation" and previous.get("intent") in {\n        "recommendation", "similarity", "alternative",\n    }:\n        merged["intent"] = previous["intent"]\n        if not merged.get("perfumes") and previous.get("perfumes"):\n            merged["perfumes"] = list(previous["perfumes"])\n    merged["exclude_candidate_ids"] = list(dict.fromkeys(\n        int(value)\n        for value in context.get("previous_recommendation_ids", [])\n        if str(value).isdigit() and int(value) > 0\n    ))\n    merged["inherited_previous_constraints"] = True\n    return merged\n\n\ndef infer_requested_count(query: str) -> int | None:\n    normalized = normalize_text(query)\n    number_tokens = {\n        "1": 1, "one": 1, "a": 1, "bir": 1,\n        "2": 2, "two": 2, "iki": 2,\n        "3": 3, "three": 3, "uc": 3,\n        "4": 4, "four": 4, "dort": 4,\n        "5": 5, "five": 5, "bes": 5,\n    }\n    token_pattern = "|".join(sorted((re.escape(token) for token in number_tokens), key=len, reverse=True))\n    noun = r"(?:perfume|perfumes|fragrance|fragrances|scent|scents|option|options|parfum|parfumu|koku|secenek)"\n    action = r"(?:recommend|suggest|give|show|list|provide|oner|tavsiye)"\n    patterns = (\n        rf"\\b{action}\\b.{{0,35}}?\\b({token_pattern})\\b(?:.{{0,20}}?\\b{noun}\\b)?",\n        rf"\\b({token_pattern})\\b(?:\\s+adet)?\\s+{noun}\\b.{{0,25}}?\\b{action}\\b",\n    )\n    for pattern in patterns:\n        match = re.search(pattern, normalized)\n        if match:\n            return number_tokens.get(match.group(1))\n    return None\n\n\ndef infer_discovery_mode(query: str) -> str | None:\n    """Recover an explicit discovery preference when planner JSON omits it."""\n    normalized = f" {normalize_text(query)} "\n    niche_phrases = (\n        " niche ", " obscure ", " hidden gem ", " hidden gems ",\n        " underrated ", " less known ", " not popular ",\n        " nis ", " az bilinen ", " populer olmayan ", " kesif ",\n    )\n    mainstream_phrases = (\n        " popular ", " mainstream ", " famous ", " well known ",\n        " recognizable ", " mass appeal ", " crowd pleasing ",\n        " populer ", " unlu ", " cok bilinen ", " herkesin bildigi ",\n    )\n    if any(phrase in normalized for phrase in niche_phrases):\n        return "niche"\n    if any(phrase in normalized for phrase in mainstream_phrases):\n        return "mainstream"\n    return None\n\n\ndef semantic_search_query(query: str, plan: dict[str, Any]) -> str:\n    """Remove output-format quantities that distort embedding retrieval."""\n    planner_query = str(plan.get("semantic_query") or "").strip()\n    if planner_query:\n        return planner_query\n    count = int(plan.get("requested_count") or 0)\n    if not count:\n        return query.strip()\n    number_words = {\n        1: "one", 2: "two", 3: "three", 4: "four", 5: "five",\n    }\n    tokens = [str(count)]\n    if count in number_words:\n        tokens.append(number_words[count])\n    cleaned = query\n    for token in tokens:\n        cleaned = re.sub(rf"\\bexactly\\s+{re.escape(token)}\\b", " ", cleaned, flags=re.I)\n        cleaned = re.sub(rf"\\b{re.escape(token)}\\s+(?:options?|recommendations?|choices?)\\b", " ", cleaned, flags=re.I)\n        cleaned = re.sub(rf"\\b{re.escape(token)}\\b", " ", cleaned, flags=re.I)\n    cleaned = re.sub(r"\\bexactly\\b", " ", cleaned, flags=re.I)\n    cleaned = re.sub(\n        r"\\b(?:recommend|suggest|list|show|provide|give)(?:\\s+me)?\\s*[.!?]*$",\n        " ",\n        cleaned,\n        flags=re.I,\n    )\n    cleaned = " ".join(cleaned.split()).strip()\n    return cleaned or query.strip()\n\n\nclass JsonHttpClient:\n    def __init__(self, base_url: str, *, timeout: int = 600) -> None:\n        self.base_url = base_url.rstrip("/")\n        self.timeout = timeout\n\n    def get(self, path: str) -> dict[str, Any]:\n        with urllib.request.urlopen(self.base_url + path, timeout=self.timeout) as response:\n            return json.loads(response.read().decode("utf-8"))\n\n    def post(self, path: str, payload: dict[str, Any]) -> dict[str, Any]:\n        request = urllib.request.Request(\n            self.base_url + path,\n            data=json.dumps(payload).encode("utf-8"),\n            headers={"Content-Type": "application/json"},\n            method="POST",\n        )\n        try:\n            with urllib.request.urlopen(request, timeout=self.timeout) as response:\n                return json.loads(response.read().decode("utf-8"))\n        except urllib.error.HTTPError as exc:\n            body = exc.read().decode("utf-8", errors="replace")\n            raise RuntimeError(f"HTTP {exc.code} from {path}: {body}") from exc\n\n\nclass VLLMClient:\n    def __init__(self, http: JsonHttpClient) -> None:\n        self.http = http\n\n    def chat(\n        self,\n        model: str,\n        messages: list[dict[str, str]],\n        *,\n        max_tokens: int,\n        json_mode: bool = False,\n    ) -> tuple[str, dict[str, Any]]:\n        payload: dict[str, Any] = {\n            "model": model,\n            "messages": messages,\n            "temperature": 0.0,\n            "max_tokens": max_tokens,\n            "stream": False,\n        }\n        if json_mode:\n            payload["response_format"] = {"type": "json_object"}\n        started = time.perf_counter()\n        structured_mode_fallback = False\n        try:\n            response = self.http.post("/v1/chat/completions", payload)\n        except RuntimeError:\n            if not json_mode:\n                raise\n            payload.pop("response_format", None)\n            structured_mode_fallback = True\n            response = self.http.post("/v1/chat/completions", payload)\n        choices = response.get("choices") or []\n        if not choices:\n            raise RuntimeError(f"vLLM returned no choices: {response}")\n        answer = str(choices[0].get("message", {}).get("content") or "").strip()\n        if not answer:\n            raise RuntimeError("vLLM returned an empty message")\n        return answer, {\n            "elapsed_seconds": round(time.perf_counter() - started, 4),\n            "usage": response.get("usage") or {},\n            "finish_reason": choices[0].get("finish_reason"),\n            "structured_mode_fallback": structured_mode_fallback,\n        }\n\n\nclass RetrievalClient:\n    def __init__(self, http: JsonHttpClient) -> None:\n        self.http = http\n\n    def health(self) -> dict[str, Any]:\n        return self.http.get("/health")\n\n    def search(self, payload: dict[str, Any]) -> dict[str, Any]:\n        return self.http.post("/search", payload)\n\n    def resolve(self, hint: str) -> dict[str, Any] | None:\n        return self.http.post("/resolve", {"hint": hint}).get("resolved")\n\n    def similar(self, payload: dict[str, Any]) -> dict[str, Any]:\n        return self.http.post("/similar", payload)\n\n\ndef candidate_record(item: dict[str, Any]) -> dict[str, Any]:\n    metadata = dict(item.get("metadata") or {})\n    merged = {**metadata, **{key: value for key, value in item.items() if key != "metadata"}}\n    merged["perfume_id"] = int(merged.get("perfume_id") or 0)\n    merged["name"] = str(merged.get("name") or "")\n    merged["brand"] = str(merged.get("brand") or "")\n    merged["label"] = str(merged.get("label") or f"{merged[\'name\']} by {merged[\'brand\']}")\n    return merged\n\n\nPERFORMANCE_CALIBRATION = {\n    # Thresholds are catalog percentiles from the current 131,930-record snapshot.\n    "longevity": {\n        "scale": 5,\n        "bands": (\n            (2.31, "light; bottom catalog quartile"),\n            (3.01, "moderate-light; below catalog median"),\n            (3.57, "moderate; typical catalog range"),\n            (4.30, "strong; top catalog quartile"),\n            (float("inf"), "very strong; approximately top 5% of the catalog"),\n        ),\n    },\n    "sillage": {\n        "scale": 4,\n        "bands": (\n            (1.68, "restrained; bottom catalog quartile"),\n            (2.15, "moderate-restrained; below catalog median"),\n            (2.57, "moderate; typical catalog range"),\n            (3.15, "noticeable/strong; top catalog quartile"),\n            (float("inf"), "very strong; approximately top 5% of the catalog"),\n        ),\n    },\n}\n\n\ndef calibrated_performance(metric: str, value: Any) -> str:\n    config = PERFORMANCE_CALIBRATION[metric]\n    if value in (None, ""):\n        return "not recorded; do not infer performance"\n    try:\n        score = float(value)\n    except (TypeError, ValueError):\n        return "not recorded; do not infer performance"\n    band = next(label for upper_bound, label in config["bands"] if score < upper_bound)\n    return f"{score:g}/{config[\'scale\']} ({band})"\n\n\ndef card_text(candidate: dict[str, Any]) -> str:\n    fields = [\n        f"Name: {candidate[\'label\']}",\n        f"Gender: {candidate.get(\'gender\') or \'not recorded\'}",\n        f"Year: {candidate.get(\'year\') if candidate.get(\'year\') is not None else \'not recorded\'}",\n        f"Accords: {candidate.get(\'accords_csv\') or \'not recorded\'}",\n        f"Notes: {candidate.get(\'notes_csv\') or \'not recorded\'}",\n        f"Rating: {candidate.get(\'rating\') if candidate.get(\'rating\') is not None else \'not recorded\'}",\n        f"Votes: {candidate.get(\'popularity\') if candidate.get(\'popularity\') is not None else \'not recorded\'}",\n        f"Longevity: {calibrated_performance(\'longevity\', candidate.get(\'longevity\'))}",\n        f"Sillage: {calibrated_performance(\'sillage\', candidate.get(\'sillage\'))}",\n        f"Value: {candidate.get(\'value_score\') if candidate.get(\'value_score\') is not None else \'not recorded\'}",\n        f"Seasons: {candidate.get(\'seasons_csv\') or \'not recorded\'}",\n        f"Time: {candidate.get(\'time_profile_csv\') or \'not recorded\'}",\n    ]\n    return "\\n".join(fields)\n\n\ndef extract_numbered_recommendations(answer: str, candidates: list[dict[str, Any]]) -> tuple[list[str], list[str]]:\n    known = [(candidate["label"], normalize_text(candidate["label"]), normalize_text(candidate["name"])) for candidate in candidates]\n    selected: list[str] = []\n    unknown: list[str] = []\n    for line in answer.splitlines():\n        match = re.match(r"^\\s*(?:\\d{1,2}[.)]|[-*])\\s+(.+?)\\s*$", line)\n        if not match:\n            continue\n        # Keep punctuation inside catalog names (for example A*Men); normalizing\n        # it away as formatting changes the identity tokens.\n        heading = match.group(1).replace("`", "").strip()\n        heading_norm = normalize_text(heading)\n        matches: list[tuple[int, int, str]] = []\n        padded_heading = f" {heading_norm} "\n        for label, label_norm, name_norm in known:\n            phrases = ((label_norm, 2), (name_norm, 1))\n            for phrase, phrase_priority in phrases:\n                if not phrase:\n                    continue\n                if heading_norm == phrase:\n                    match_class = 4\n                elif heading_norm.startswith(phrase + " "):\n                    match_class = 3\n                elif f" {phrase} " in padded_heading:\n                    match_class = phrase_priority\n                else:\n                    continue\n                matches.append((match_class, len(phrase), label))\n        found = max(matches, default=(0, 0, None))[2]\n        if not found and " by " in heading_norm:\n            heading_name, heading_brand = heading_norm.rsplit(" by ", 1)\n            scoped_aliases = [\n                candidate["label"]\n                for candidate in candidates\n                if normalize_text(candidate.get("brand")) == heading_brand\n                and (\n                    normalize_text(candidate.get("name")) == heading_name\n                    or normalize_text(candidate.get("name")).endswith(f" {heading_name}")\n                )\n            ]\n            if len(scoped_aliases) == 1:\n                found = scoped_aliases[0]\n        if found:\n            if found not in selected:\n                selected.append(found)\n        elif " by " in f" {heading_norm} ":\n            unknown.append(heading)\n    return selected, unknown\n\n\ndef candidate_mentions(answer: str, candidates: list[dict[str, Any]]) -> list[str]:\n    answer_norm = f" {normalize_text(answer)} "\n    output = []\n    for candidate in candidates:\n        name = normalize_text(candidate["name"])\n        label = normalize_text(candidate["label"])\n        if (name and f" {name} " in answer_norm) or (label and f" {label} " in answer_norm):\n            output.append(candidate["label"])\n    return list(dict.fromkeys(output))\n\n\ndef candidate_has_term(candidate: dict[str, Any], term: str) -> bool:\n    searchable = normalize_text(" ".join(str(candidate.get(key) or "") for key in ("name", "brand", "accords_csv", "notes_csv")))\n    term_norm = canonical_trait_term(term)\n    if f" {term_norm} " in f" {searchable} ":\n        return True\n    variants = {\n        "musk": {"musk", "musky", "white musk"},\n        "leather": {"leather", "leathery", "suede"},\n        "oud": {"oud", "aoud", "agarwood"},\n        "smoky": {"smoke", "smoky"},\n    }.get(term_norm, {term_norm})\n    traits = {\n        normalize_text(part)\n        for key in ("accords_csv", "notes_csv")\n        for part in str(candidate.get(key) or "").split(",")\n        if normalize_text(part)\n    }\n    if traits & variants:\n        return True\n    if term_norm in {"spicy", "floral"}:\n        return any(term_norm in trait.split() for trait in traits)\n    return False\n\n\ndef keep_required_candidates(candidates: list[dict[str, Any]], plan: dict[str, Any]) -> list[dict[str, Any]]:\n    required = plan.get("required_terms", [])\n    if not required:\n        return candidates\n    return [\n        candidate\n        for candidate in candidates\n        if all(candidate_has_term(candidate, term) for term in required)\n    ]\n\n\ndef unique_candidates(candidates: list[dict[str, Any]]) -> list[dict[str, Any]]:\n    """Deduplicate catalog aliases/duplicate rows while preserving rank order."""\n    output: list[dict[str, Any]] = []\n    seen_ids: set[int] = set()\n    seen_labels: set[str] = set()\n    for candidate in candidates:\n        perfume_id = int(candidate.get("perfume_id") or 0)\n        label = normalize_text(candidate.get("label"))\n        if (perfume_id and perfume_id in seen_ids) or (label and label in seen_labels):\n            continue\n        output.append(candidate)\n        if perfume_id:\n            seen_ids.add(perfume_id)\n        if label:\n            seen_labels.add(label)\n    return output\n\n\ndef candidate_answer_sections(answer: str, candidates: list[dict[str, Any]]) -> dict[str, str]:\n    if len(candidates) == 1:\n        return {candidates[0]["label"]: answer}\n    sections: dict[str, list[str]] = {}\n    active_label: str | None = None\n    for line in answer.splitlines():\n        if re.match(r"^\\s*\\d{1,2}[.)]\\s+", line):\n            line_mentions = candidate_mentions(line, candidates)\n            active_label = line_mentions[0] if line_mentions else None\n            if active_label:\n                sections.setdefault(active_label, [])\n        if active_label:\n            sections[active_label].append(line)\n    if sections:\n        return {label: "\\n".join(lines) for label, lines in sections.items()}\n\n    # Comparisons are usually written as natural paragraphs instead of numbered\n    # recommendations. Attribute only unambiguous paragraphs/sentences so a\n    # statement about one perfume cannot leak into the other perfume\'s checks.\n    for paragraph in re.split(r"\\n\\s*\\n+", str(answer or "")):\n        paragraph = paragraph.strip()\n        if not paragraph:\n            continue\n        paragraph_mentions = candidate_mentions(paragraph, candidates)\n        if len(paragraph_mentions) == 1:\n            sections.setdefault(paragraph_mentions[0], []).append(paragraph)\n            continue\n        if len(paragraph_mentions) > 1:\n            for sentence in re.split(r"(?<=[.!?])\\s+", paragraph):\n                sentence_mentions = candidate_mentions(sentence, candidates)\n                if len(sentence_mentions) == 1:\n                    sections.setdefault(sentence_mentions[0], []).append(sentence.strip())\n    return {label: "\\n".join(parts) for label, parts in sections.items()}\n\n\ndef explicit_performance_labels(text: str, metric: str) -> set[str]:\n    clauses = [normalize_text(part) for part in re.split(r"[,;.\\n]+", str(text or ""))]\n    clauses = [\n        re.sub(\n            r"\\b(?:more|less|daha)\\s+(?:noticeable|pronounced|strong|light|weak|restrained|belirgin|guclu|hafif|zayif)\\b",\n            "relative",\n            re.sub(\n                r"\\b(?:moderate restrained|moderate light|orta kisitli|orta hafif)\\b",\n                "moderate",\n                clause,\n            ),\n        )\n        for clause in clauses\n    ]\n    noun_patterns = {\n        "longevity": r"(?:longevity|lasting power|kalicilik|kaliciligi|kaliciliga)",\n        "sillage": r"(?:sillage|projection|yayilim|yayilimi|yayilima)",\n    }\n    nouns = noun_patterns[metric]\n    low_extra = r"|short lived" if metric == "longevity" else r"|soft|subtle|close to skin|yumusak"\n    high_extra = r"|long lasting|excellent" if metric == "longevity" else r"|noticeable|pronounced|projecting|belirgin"\n    labels = {\n        "low": rf"(?:light|weak|low|short|restrained|intimate|hafif|zayif|dusuk|kisa|tene yakin{low_extra})",\n        "moderate": r"(?:moderate|average|medium|orta|ortalama)",\n        "high": rf"(?:very strong|strong|high|powerful|exceptional|cok guclu|guclu|yuksek{high_extra})",\n    }\n    observed: set[str] = set()\n    for group, adjective in labels.items():\n        connector = (\n            r"(?: is classified as| is rated as| is considered| can be described as|"\n            r" is| feels| remains| stays| olarak tanimlanabilir| olarak siniflandirilabilir|"\n            r" seviyesindedir| seviyesi| ise| degeri)?"\n        )\n        patterns = (\n            rf"{nouns}{connector}(?: at| bir)?(?: quite| rather| relatively| oldukca| hayli| biraz)? {adjective}\\b",\n            rf"{adjective}(?: level| seviyede| seviyedeki| bir)? {nouns}\\b",\n        )\n        if any(re.search(pattern, clause) for clause in clauses for pattern in patterns):\n            observed.add(group)\n\n    # Natural prose frequently inserts particles and qualifiers that rigid\n    # grammar patterns cannot enumerate (for example, "yayilimi da guclu bir\n    # seviyede"). Assign each class adjective to the nearest performance noun,\n    # within a small window, to avoid leaking a sillage class into longevity or\n    # one comparison subject into another.\n    for clause in clauses:\n        noun_hits: list[tuple[int, int, str]] = []\n        for noun_metric, noun_pattern in noun_patterns.items():\n            noun_hits.extend(\n                (match.start(), match.end(), noun_metric)\n                for match in re.finditer(rf"\\b{noun_pattern}\\b", clause)\n            )\n        if not noun_hits:\n            continue\n        for group, adjective in labels.items():\n            for adjective_match in re.finditer(rf"\\b{adjective}\\b", clause):\n                prefix = clause[max(0, adjective_match.start() - 28):adjective_match.start()]\n                if re.search(r"(?:\\bnot|\\bno|\\bdegil|\\bisn t|\\bwasn t)\\s+(?:particularly\\s+)?$", prefix):\n                    continue\n                nearby: list[tuple[int, int, str]] = []\n                for noun_start, noun_end, noun_metric in noun_hits:\n                    between = (\n                        clause[noun_end:adjective_match.start()]\n                        if noun_end <= adjective_match.start()\n                        else clause[adjective_match.end():noun_start]\n                    )\n                    word_distance = len(between.split())\n                    if word_distance <= 5:\n                        char_distance = max(\n                            noun_start - adjective_match.end(),\n                            adjective_match.start() - noun_end,\n                            0,\n                        )\n                        nearby.append((word_distance, char_distance, noun_metric))\n                if nearby:\n                    best_distance = min(item[:2] for item in nearby)\n                    nearest_metrics = {\n                        noun_metric\n                        for word_distance, char_distance, noun_metric in nearby\n                        if (word_distance, char_distance) == best_distance\n                    }\n                    if nearest_metrics == {metric}:\n                        observed.add(group)\n    return observed\n\n\ndef expected_performance_group(metric: str, value: Any) -> str:\n    if value in (None, ""):\n        return "unknown"\n    try:\n        score = float(value)\n    except (TypeError, ValueError):\n        return "unknown"\n    if metric == "longevity":\n        return "low" if score < 2.31 else "moderate" if score < 3.57 else "high"\n    return "low" if score < 1.68 else "moderate" if score < 2.57 else "high"\n\n\ndef performance_claim_violations(answer: str, candidates: list[dict[str, Any]]) -> list[dict[str, Any]]:\n    sections = candidate_answer_sections(answer, candidates)\n    violations: list[dict[str, Any]] = []\n    for candidate in candidates:\n        section = sections.get(candidate["label"])\n        if not section:\n            continue\n        for metric in ("longevity", "sillage"):\n            observed = explicit_performance_labels(section, metric)\n            if not observed:\n                continue\n            expected = expected_performance_group(metric, candidate.get(metric))\n            if expected == "unknown" or observed != {expected}:\n                violations.append({\n                    "candidate": candidate["label"],\n                    "metric": metric,\n                    "expected": expected,\n                    "observed": sorted(observed),\n                })\n    return violations\n\n\ndef validate_answer(\n    answer: str,\n    plan: dict[str, Any],\n    candidates: list[dict[str, Any]],\n    *,\n    response_language: str | None = None,\n) -> dict[str, Any]:\n    reasons: list[str] = []\n    mentions = candidate_mentions(answer, candidates)\n    numbered, unknown = extract_numbered_recommendations(answer, candidates)\n    if not answer.strip():\n        reasons.append("empty_answer")\n    if unknown:\n        reasons.append("unsupported_numbered_perfume")\n    intent = plan["intent"]\n    if intent == "comparison":\n        missing = [candidate["label"] for candidate in candidates[:2] if candidate["label"] not in mentions]\n        if missing:\n            reasons.append("comparison_missing_perfume")\n    elif intent != "exact_lookup" and not mentions:\n        reasons.append("no_context_perfume_mentioned")\n\n    expected_count = int(plan.get("requested_count") or 0)\n    if expected_count and len(numbered) != expected_count:\n        reasons.append("requested_count_mismatch")\n\n    violated = []\n    recommendation_labels = numbered or mentions\n    by_label = {candidate["label"]: candidate for candidate in candidates}\n    for label in recommendation_labels:\n        candidate = by_label.get(label)\n        if candidate and any(candidate_has_term(candidate, term) for term in plan.get("excluded_terms", [])):\n            violated.append(label)\n    if violated:\n        reasons.append("strict_filter_violation")\n\n    missing_required = []\n    for label in recommendation_labels:\n        candidate = by_label.get(label)\n        if candidate and any(not candidate_has_term(candidate, term) for term in plan.get("required_terms", [])):\n            missing_required.append(label)\n    if missing_required:\n        reasons.append("required_trait_violation")\n\n    performance_violations = performance_claim_violations(answer, candidates)\n    if performance_violations:\n        reasons.append("performance_calibration_violation")\n\n    generic_patterns = (\n        r"\\bit matches through\\b",\n        r"\\bi would include it for\\b",\n        r"\\bit earns a place here with\\b",\n        r"\\bthe fit comes from\\b",\n        r"\\bbu parfumu buraya dahil etme sebebim\\b",\n        r"\\bnotalari ve akorlari uzerinden\\b",\n        r"\\bonemli bir uzlasi\\b",\n        r"\\buygun bir hava ciziyor\\b",\n    )\n    normalized_answer = normalize_text(answer)\n    if any(re.search(pattern, normalized_answer, re.I) for pattern in generic_patterns):\n        reasons.append("mechanical_template_language")\n    if response_language and not response_language_matches(answer, response_language):\n        reasons.append("wrong_response_language")\n    return {\n        "pass": not reasons,\n        "reasons": list(dict.fromkeys(reasons)),\n        "mentioned_candidates": mentions,\n        "numbered_recommendations": numbered,\n        "unsupported_numbered_items": unknown,\n        "strict_filter_violations": violated,\n        "required_trait_violations": missing_required,\n        "performance_calibration_violations": performance_violations,\n        "response_language": response_language,\n    }\n\n\ndef exact_lookup_answer(\n    candidate: dict[str, Any],\n    requested_fields: list[str],\n    *,\n    response_language: str = "en",\n) -> str:\n    field_map = OrderedDict([\n        ("rating", candidate.get("rating")),\n        ("popularity", candidate.get("popularity")),\n        ("year", candidate.get("year")),\n        ("longevity", candidate.get("longevity")),\n        ("sillage", candidate.get("sillage")),\n        ("value", candidate.get("value_score")),\n        ("accords", candidate.get("accords_csv")),\n        ("notes", candidate.get("notes_csv")),\n        ("seasons", candidate.get("seasons_csv")),\n        ("time", candidate.get("time_profile_csv")),\n    ])\n    selected = requested_fields or list(field_map)\n    labels_tr = {\n        "rating": "Puan",\n        "popularity": "Oy sayısı",\n        "year": "Yıl",\n        "longevity": "Kalıcılık",\n        "sillage": "Yayılım",\n        "value": "Fiyat-performans",\n        "accords": "Akorlar",\n        "notes": "Notalar",\n        "seasons": "Mevsimler",\n        "time": "Kullanım zamanı",\n    }\n    lines = [\n        f"{candidate[\'label\']} için veritabanı kaydı:"\n        if response_language == "tr"\n        else f"Database record for {candidate[\'label\']}:"\n    ]\n    for key in selected:\n        if key in field_map:\n            value = field_map[key]\n            if response_language == "tr":\n                lines.append(f"- {labels_tr[key]}: {value if value not in (None, \'\') else \'kayıtlı değil\'}")\n            else:\n                lines.append(f"- {key.title()}: {value if value not in (None, \'\') else \'not recorded\'}")\n    return "\\n".join(lines)\n\n\ndef fallback_answer(\n    plan: dict[str, Any],\n    candidates: list[dict[str, Any]],\n    *,\n    query: str = "",\n    response_language: str | None = None,\n) -> str:\n    language = response_language or detect_response_language(query)\n    if not candidates:\n        return static_message("no_safe_match", language)\n    if plan["intent"] == "exact_lookup":\n        return exact_lookup_answer(\n            candidates[0],\n            plan.get("requested_fields", []),\n            response_language=language,\n        )\n    if plan["intent"] == "comparison" and len(candidates) >= 2:\n        sections = ["Veritabanı karşılaştırması:" if language == "tr" else "Grounded comparison:"]\n        for candidate in candidates[:2]:\n            if language == "tr":\n                sections.extend([\n                    f"\\n{candidate[\'label\']}",\n                    f"- Karakter: {candidate.get(\'accords_csv\') or \'kayıtlı değil\'}",\n                    f"- Kullanım: {candidate.get(\'seasons_csv\') or \'kayıtlı değil\'}; {candidate.get(\'time_profile_csv\') or \'kayıtlı değil\'}",\n                    f"- Performans: kalıcılık {candidate.get(\'longevity\') or \'kayıtlı değil\'}, yayılım {candidate.get(\'sillage\') or \'kayıtlı değil\'}",\n                ])\n            else:\n                sections.extend([\n                    f"\\n{candidate[\'label\']}",\n                    f"- Character: {candidate.get(\'accords_csv\') or \'not recorded\'}",\n                    f"- Wear: {candidate.get(\'seasons_csv\') or \'not recorded\'}; {candidate.get(\'time_profile_csv\') or \'not recorded\'}",\n                    f"- Performance: longevity {candidate.get(\'longevity\') or \'not recorded\'}, sillage {candidate.get(\'sillage\') or \'not recorded\'}",\n                ])\n        return "\\n".join(sections)\n    count = int(plan.get("requested_count") or 3)\n    if language == "tr":\n        lines = ["Kayıtlara göre en güçlü seçenekler:"]\n        for index, candidate in enumerate(candidates[:count], 1):\n            lines.extend([\n                f"{index}. {candidate[\'label\']}",\n                f"{candidate.get(\'accords_csv\') or \'Kayıtlı akor bulunmuyor\'} karakteriyle öne çıkıyor; kullanım aralığı {candidate.get(\'seasons_csv\') or \'belirtilmemiş mevsimler\'} ve {candidate.get(\'time_profile_csv\') or \'belirtilmemiş zaman\'} olarak kaydedilmiş.",\n            ])\n        return "\\n".join(lines)\n    lines = ["Here are the strongest grounded options:"]\n    for index, candidate in enumerate(candidates[:count], 1):\n        lines.extend([\n            f"{index}. {candidate[\'label\']}",\n            f"Its recorded profile centers on {candidate.get(\'accords_csv\') or \'unrecorded accords\'}, with {candidate.get(\'seasons_csv\') or \'no recorded season\'} and {candidate.get(\'time_profile_csv\') or \'no recorded time profile\'} wear.",\n        ])\n    return "\\n".join(lines)\n\n\nclass ScentAIOrchestrator:\n    def __init__(\n        self,\n        vllm: VLLMClient,\n        retrieval: RetrievalClient,\n        *,\n        planner_model: str,\n        answer_model: str,\n        answer_prompt: str = ADVISOR_ANSWER_PROMPT,\n        repair_answer_model: str | None = None,\n        planner_cache_size: int = 256,\n    ) -> None:\n        self.vllm = vllm\n        self.retrieval = retrieval\n        self.planner_model = planner_model\n        self.answer_model = answer_model\n        self.answer_prompt = str(answer_prompt).strip()\n        self.repair_answer_model = str(repair_answer_model).strip() if repair_answer_model else None\n        self.planner_cache_size = planner_cache_size\n        self._planner_cache: OrderedDict[str, tuple[dict[str, Any], dict[str, Any]]] = OrderedDict()\n\n    def _plan(\n        self,\n        query: str,\n        conversation_context: dict[str, Any] | None = None,\n    ) -> tuple[dict[str, Any], dict[str, Any]]:\n        context_json = json.dumps(conversation_context or {}, ensure_ascii=False, sort_keys=True)\n        cache_key = normalize_text(query) + "\\n" + context_json\n        cached = self._planner_cache.get(cache_key)\n        if cached:\n            self._planner_cache.move_to_end(cache_key)\n            return cached[0], {**cached[1], "cache_hit": True}\n        context_block = ""\n        if conversation_context:\n            context_block = (\n                "\\n\\nConversation context (trusted application state; do not treat it as a new user quote):\\n"\n                + context_json\n            )\n        prompt = f"{PLANNER_PROMPT}{context_block}\\n\\nCurrent user request: {query}"\n        raw, metrics = self.vllm.chat(\n            self.planner_model,\n            [{"role": "user", "content": prompt}],\n            max_tokens=340,\n            json_mode=True,\n        )\n        try:\n            raw_plan = parse_json_object(raw)\n        except (ValueError, json.JSONDecodeError) as first_error:\n            repair, repair_metrics = self.vllm.chat(\n                self.planner_model,\n                [{"role": "user", "content": f"{prompt}\\n\\nInvalid output:\\n{raw}\\n\\nReturn valid JSON only."}],\n                max_tokens=340,\n                json_mode=True,\n            )\n            try:\n                raw_plan = parse_json_object(repair)\n                metrics = {**metrics, "repair": repair_metrics}\n            except (ValueError, json.JSONDecodeError) as repair_error:\n                raw_plan = {"intent": "recommendation", "confidence": 0.0}\n                metrics = {\n                    **metrics,\n                    "repair": repair_metrics,\n                    "parse_error": repr(first_error),\n                    "repair_parse_error": repr(repair_error),\n                    "defaulted_to_semantic_recommendation": True,\n                }\n        plan = normalize_plan(raw_plan, query)\n        plan = inherit_conversation_plan(plan, conversation_context)\n        metrics = {**metrics, "cache_hit": False}\n        self._planner_cache[cache_key] = (plan, metrics)\n        while len(self._planner_cache) > self.planner_cache_size:\n            self._planner_cache.popitem(last=False)\n        return plan, metrics\n\n    def _retrieve(self, query: str, plan: dict[str, Any]) -> tuple[list[dict[str, Any]], dict[str, Any], dict[str, Any] | None]:\n        intent = plan["intent"]\n        perfumes = plan.get("perfumes", [])\n        reference = None\n        if intent in {"comparison", "perfume_profile", "exact_lookup"} and perfumes:\n            limit = 2 if intent == "comparison" else 1\n            resolved = [self.retrieval.resolve(hint) for hint in perfumes[:limit]]\n            candidates = unique_candidates([candidate_record(item) for item in resolved if item])\n            return candidates, {"route": "canonical_resolve", "result_count": len(candidates)}, None\n\n        if intent in {"similarity", "alternative"} and perfumes:\n            reference_hint = perfumes[0]\n            if plan.get("reference_brand") and " by " not in normalize_text(reference_hint):\n                reference_hint = f"{reference_hint} by {plan[\'reference_brand\']}"\n            result = self.retrieval.similar({\n                "hint": reference_hint,\n                "top_k": max(int(plan.get("requested_count") or 3) + 5, 8),\n                "exclude_terms": plan.get("excluded_terms", []),\n                "required_terms": plan.get("required_terms", []),\n                "exclude_ids": plan.get("exclude_candidate_ids", []),\n            })\n            reference = result.get("source")\n            candidates = keep_required_candidates(\n                unique_candidates([candidate_record(item) for item in result.get("results", [])]),\n                plan,\n            )\n            requested_count = int(plan.get("requested_count") or 3)\n            if len(candidates) < requested_count and reference:\n                source = candidate_record(reference)\n                source_traits = [\n                    normalize_text(term)\n                    for field in ("accords_csv", "notes_csv")\n                    for term in str(source.get(field) or "").split(",")\n                    if normalize_text(term)\n                ]\n                existing_ids = [int(candidate["perfume_id"]) for candidate in candidates]\n                fallback_result = self.retrieval.search({\n                    "query": query,\n                    "top_k": max(requested_count * 3, 10),\n                    "filters": {},\n                    "wanted_terms": list(dict.fromkeys([*plan.get("wanted_terms", []), *source_traits[:6]])),\n                    "required_terms": plan.get("required_terms", []),\n                    "exclude_terms": list(dict.fromkeys([*plan.get("excluded_terms", []), normalize_text(source["name"])])),\n                    "exclude_ids": list(dict.fromkeys([\n                        *plan.get("exclude_candidate_ids", []),\n                        int(source["perfume_id"]),\n                        *existing_ids,\n                    ])),\n                    "discovery_mode": plan.get("discovery_mode", "balanced"),\n                })\n                supplements = keep_required_candidates(\n                    unique_candidates([candidate_record(item) for item in fallback_result.get("results", [])]),\n                    plan,\n                )\n                candidates = unique_candidates([*candidates, *supplements])\n                fallback_result["route"] = (\n                    "hybrid_similarity_supplement"\n                    if existing_ids else "semantic_similarity_fallback"\n                )\n                fallback_result["graph_result_count"] = len(existing_ids)\n                fallback_result["result_count"] = len(candidates)\n                result = fallback_result\n            return candidates, result, reference\n\n        filters = {\n            key: value\n            for key, value in {\n                "brand": plan.get("requested_brand"),\n                "gender": plan.get("gender"),\n                "season": plan.get("season"),\n                "time": plan.get("time_profile"),\n            }.items()\n            if value\n        }\n        search_query = semantic_search_query(query, plan)\n        result = self.retrieval.search({\n            "query": search_query,\n            "top_k": max(int(plan.get("requested_count") or 3) + 5, 8),\n            "filters": filters,\n            "wanted_terms": plan.get("wanted_terms", []),\n            "required_terms": plan.get("required_terms", []),\n            "exclude_terms": plan.get("excluded_terms", []),\n            "exclude_ids": plan.get("exclude_candidate_ids", []),\n            "discovery_mode": plan.get("discovery_mode", "balanced"),\n        })\n        result["original_query"] = query\n        result["semantic_query"] = search_query\n        candidates = keep_required_candidates(\n            unique_candidates([candidate_record(item) for item in result.get("results", [])]),\n            plan,\n        )\n        return candidates, result, reference\n\n    def run(\n        self,\n        query: str,\n        *,\n        conversation_context: dict[str, Any] | None = None,\n        answer_prompt_override: str | None = None,\n        answer_model_override: str | None = None,\n    ) -> dict[str, Any]:\n        query = str(query or "").strip()\n        if not query:\n            raise ValueError("query is required")\n        started = time.perf_counter()\n        response_language = detect_response_language(query, conversation_context)\n        try:\n            plan, planner_metrics = self._plan(query, conversation_context)\n        except Exception as exc:\n            plan = normalize_plan({"intent": "recommendation", "confidence": 0.0}, query)\n            planner_metrics = {\n                "cache_hit": False,\n                "error": repr(exc),\n                "defaulted_to_semantic_recommendation": True,\n            }\n        plan["response_language"] = response_language\n        unsupported = unsupported_answer(plan["intent"], response_language)\n        if unsupported:\n            return {\n                "query": query,\n                "route": plan["intent"],\n                "plan": plan,\n                "response_language": response_language,\n                "answer": unsupported,\n                "candidates": [],\n                "validation": {"pass": True, "reasons": []},\n                "generation_attempts": 0,\n                "timings": {"planner": planner_metrics, "total_seconds": round(time.perf_counter() - started, 4)},\n            }\n\n        retrieval_started = time.perf_counter()\n        try:\n            candidates, retrieval_result, reference = self._retrieve(query, plan)\n        except Exception as exc:\n            return {\n                "query": query,\n                "route": "retrieval_error",\n                "plan": plan,\n                "response_language": response_language,\n                "answer": static_message("retrieval_error", response_language),\n                "candidates": [],\n                "validation": {"pass": False, "reasons": ["retrieval_error"]},\n                "generation_attempts": 0,\n                "error": repr(exc),\n                "timings": {"planner": planner_metrics, "total_seconds": round(time.perf_counter() - started, 4)},\n            }\n        retrieval_seconds = round(time.perf_counter() - retrieval_started, 4)\n        if plan["intent"] == "comparison" and len(candidates) != 2:\n            return {\n                "query": query,\n                "route": "comparison_unresolved",\n                "plan": plan,\n                "response_language": response_language,\n                "answer": static_message("comparison_unresolved", response_language),\n                "candidates": candidates,\n                "validation": {"pass": True, "reasons": []},\n                "generation_attempts": 0,\n                "timings": {"planner": planner_metrics, "retrieval_seconds": retrieval_seconds, "total_seconds": round(time.perf_counter() - started, 4)},\n            }\n        if not candidates:\n            return {\n                "query": query,\n                "route": "no_safe_match",\n                "plan": plan,\n                "response_language": response_language,\n                "answer": static_message("no_safe_match", response_language),\n                "candidates": [],\n                "validation": {"pass": True, "reasons": []},\n                "generation_attempts": 0,\n                "timings": {"planner": planner_metrics, "retrieval_seconds": retrieval_seconds, "total_seconds": round(time.perf_counter() - started, 4)},\n            }\n\n        if plan["intent"] == "exact_lookup":\n            answer = exact_lookup_answer(\n                candidates[0],\n                plan.get("requested_fields", []),\n                response_language=response_language,\n            )\n            return {\n                "query": query,\n                "route": "deterministic_exact_lookup",\n                "plan": plan,\n                "response_language": response_language,\n                "answer": answer,\n                "candidates": candidates,\n                "reference": reference,\n                "validation": {"pass": True, "reasons": []},\n                "generation_attempts": 0,\n                "timings": {"planner": planner_metrics, "retrieval_seconds": retrieval_seconds, "total_seconds": round(time.perf_counter() - started, 4)},\n            }\n\n        context = "\\n\\n".join(f"[CARD {index}]\\n{card_text(candidate)}" for index, candidate in enumerate(candidates, 1))\n        reference_text = card_text(candidate_record(reference)) if reference else "None"\n        active_answer_prompt = str(answer_prompt_override or self.answer_prompt).strip()\n        active_answer_model = str(answer_model_override or self.answer_model).strip()\n        user_prompt = (\n            f"{active_answer_prompt}\\n\\n"\n            f"Validated plan: {json.dumps(plan, ensure_ascii=False)}\\n\\n"\n            f"[CONVERSATION CONTEXT]\\n{json.dumps(conversation_context or {}, ensure_ascii=False)}\\n\\n"\n            f"[REFERENCE]\\n{reference_text}\\n\\n"\n            f"[DATABASE CARDS]\\n{context}\\n\\n"\n            f"[USER REQUEST]\\n{query}\\n\\n"\n            f"[OUTPUT LANGUAGE - HARD REQUIREMENT]\\n{output_language_instruction(response_language)}"\n        )\n        failures: list[dict[str, Any]] = []\n        generation_metrics: list[dict[str, Any]] = []\n        attempted_generations = 0\n        answer = ""\n        validation: dict[str, Any] = {"pass": False, "reasons": ["not_generated"]}\n        answer_count = (\n            2 if plan["intent"] == "comparison"\n            else 1 if plan["intent"] == "perfume_profile"\n            else int(plan.get("requested_count") or 3)\n        )\n        answer_max_tokens = min(720, max(320, 230 + answer_count * 100))\n        for attempt in (1, 2):\n            attempted_generations += 1\n            messages = [{"role": "user", "content": user_prompt}]\n            if attempt == 2:\n                validation_details = {\n                    key: value\n                    for key, value in validation.items()\n                    if key not in {"pass", "mentioned_candidates"} and value\n                }\n                correction = (\n                    "The previous answer failed deterministic validation: "\n                    + ", ".join(validation["reasons"])\n                    + ". Exact validator details: "\n                    + json.dumps(validation_details, ensure_ascii=False)\n                    + ". Write a new answer using only the exact database card names, obeying all exclusions and the requested count. "\n                    + "If the failure concerns mechanical language, rebuild the prose with different openings, sentence rhythms, and evidence choices for every recommendation. "\n                    + "If it concerns performance calibration, follow the card\'s longevity and sillage bands exactly, or omit the performance claim when it is not recorded. "\n                    + output_language_instruction(response_language)\n                )\n                messages.extend([\n                    {"role": "assistant", "content": answer},\n                    {"role": "user", "content": correction},\n                ])\n            try:\n                generation_model = (\n                    active_answer_model\n                    if attempt == 1 or answer_model_override or not self.repair_answer_model\n                    else self.repair_answer_model\n                )\n                answer, metrics = self.vllm.chat(\n                    generation_model,\n                    messages,\n                    max_tokens=answer_max_tokens,\n                )\n            except Exception as exc:\n                metrics = {"error": repr(exc), "model": generation_model}\n                validation = {"pass": False, "reasons": ["model_generation_error"]}\n                generation_metrics.append(metrics)\n                failures.append(validation)\n                answer = "No answer was produced."\n                continue\n            metrics = {**metrics, "model": generation_model}\n            generation_metrics.append(metrics)\n            validation = validate_answer(\n                answer,\n                plan,\n                candidates,\n                response_language=response_language,\n            )\n            if metrics.get("finish_reason") == "length":\n                validation = {\n                    **validation,\n                    "pass": False,\n                    "reasons": list(dict.fromkeys([*validation["reasons"], "truncated_generation"])),\n                }\n            if validation["pass"]:\n                break\n            failures.append({**validation, "rejected_answer": answer})\n\n        route = "llm_grounded"\n        if not validation["pass"]:\n            answer = fallback_answer(\n                plan,\n                candidates,\n                query=query,\n                response_language=response_language,\n            )\n            validation = validate_answer(\n                answer,\n                plan,\n                candidates,\n                response_language=response_language,\n            )\n            route = "validated_template_fallback"\n        elif plan["intent"] == "comparison":\n            route = "llm_grounded_comparison"\n        elif plan["intent"] in {"similarity", "alternative"}:\n            route = "llm_grounded_similarity"\n        elif plan["intent"] == "perfume_profile":\n            route = "llm_grounded_profile"\n\n        return {\n            "query": query,\n            "route": route,\n            "answer_model": active_answer_model,\n            "answer_prompt_mode": (\n                "advisor" if active_answer_prompt == ADVISOR_ANSWER_PROMPT\n                else "legacy" if active_answer_prompt == LEGACY_ANSWER_PROMPT\n                else "custom"\n            ),\n            "response_language": response_language,\n            "plan": plan,\n            "answer": answer,\n            "candidates": candidates,\n            "reference": reference,\n            "retrieval": {\n                "route": retrieval_result.get("route"),\n                "elapsed_seconds": retrieval_result.get("elapsed_seconds"),\n                "result_count": retrieval_result.get("result_count", len(candidates)),\n                "semantic_query": retrieval_result.get("semantic_query"),\n                "supported_wanted_terms": retrieval_result.get("supported_wanted_terms", []),\n                "ignored_wanted_terms": retrieval_result.get("ignored_wanted_terms", []),\n                "discovery_mode": retrieval_result.get("discovery_mode", plan.get("discovery_mode", "balanced")),\n                "excluded_candidate_ids": retrieval_result.get("exclude_ids", []),\n            },\n            "validation": validation,\n            "generation_attempts": attempted_generations,\n            "generation_failures": failures,\n            "timings": {\n                "planner": planner_metrics,\n                "retrieval_seconds": retrieval_seconds,\n                "generation": generation_metrics,\n                "total_seconds": round(time.perf_counter() - started, 4),\n            },\n        }\n\n\nclass ScentAISession:\n    """Small stateful wrapper for grounded, multi-turn perfume conversations."""\n\n    def __init__(self, pipeline: ScentAIOrchestrator) -> None:\n        self.pipeline = pipeline\n        self.reset()\n\n    def reset(self) -> None:\n        self.last_result: dict[str, Any] | None = None\n        self.recommendation_ids: list[int] = []\n\n    def run(self, query: str) -> dict[str, Any]:\n        context = None\n        if self.last_result:\n            context = {\n                "previous_query": self.last_result.get("query"),\n                "previous_plan": self.last_result.get("plan"),\n                "previous_response_language": self.last_result.get("response_language"),\n                "previous_recommendations": [\n                    {"perfume_id": candidate["perfume_id"], "label": candidate["label"]}\n                    for candidate in self.last_result.get("candidates", [])\n                    if candidate["perfume_id"] in self.recommendation_ids\n                ],\n                "previous_recommendation_ids": list(self.recommendation_ids),\n            }\n        result = self.pipeline.run(query, conversation_context=context)\n        mentioned = set(result.get("validation", {}).get("mentioned_candidates", []))\n        recommended_ids = [\n            int(candidate["perfume_id"])\n            for candidate in result.get("candidates", [])\n            if candidate.get("label") in mentioned\n        ]\n        if result.get("plan", {}).get("conversation_action") in {"more_options", "refine_previous"}:\n            self.recommendation_ids = list(dict.fromkeys([*self.recommendation_ids, *recommended_ids]))\n        else:\n            self.recommendation_ids = recommended_ids\n        self.last_result = result\n        return result\n'
ORCHESTRATOR_PATH = Path("/content/scentai_orchestrator.py")
ORCHESTRATOR_PATH.write_text(ORCHESTRATOR_SOURCE, encoding="utf-8")
compile(ORCHESTRATOR_SOURCE, str(ORCHESTRATOR_PATH), "exec")
exec(compile(ORCHESTRATOR_SOURCE, str(ORCHESTRATOR_PATH), "exec"), globals())

vllm_client = VLLMClient(JsonHttpClient(BASE_URL, timeout=600))
retrieval_client = RetrievalClient(JsonHttpClient(RETRIEVAL_URL, timeout=180))
pipeline = ScentAIOrchestrator(
    vllm_client,
    retrieval_client,
    planner_model=MODEL_NAME,
    answer_model=MODEL_NAME,
    repair_answer_model=LORA_NAME,
)

assert retrieval_client.health()["status"] == "ok"
served_model_ids = {item["id"] for item in get_json(f"{BASE_URL}/v1/models").get("data", [])}
assert {MODEL_NAME, LORA_NAME}.issubset(served_model_ids), served_model_ids
print("Stage 3 orchestrator ready.")

## End-to-end contract tests

These cases cover the failure modes found during earlier experiments: style words
mistaken for brands, explicit brand requests, similarity with a negative trait,
and ambiguous family-name resolution in comparisons. Failed model generations
are retried once and then replaced by a grounded fallback.

In [ ]:
from datetime import datetime, timezone

contract_cases = [
    {
        "name": "clean_office_not_clean_brand",
        "query": "I need a clean office scent without vanilla. Recommend exactly 3.",
        "expected_language": "en",
    },
    {
        "name": "explicit_versace_brand",
        "query": "Recommend exactly 5 men's fragrances from Versace.",
        "expected_language": "en",
    },
    {
        "name": "aventus_similarity_less_smoky",
        "query": "I want something similar to Aventus but less smoky. Give me exactly 3 options.",
        "expected_language": "en",
    },
    {
        "name": "canonical_comparison",
        "query": "Compare Club de Nuit by Armaf with Team Five by Adidas. Explain their vibe and best use cases.",
        "expected_language": "en",
    },
    {
        "name": "turkish_profile_language",
        "query": "Turkish Leather by Pryn Parfum nasıl bir parfüm? Karakterini, kalıcılığını, yayılımını ve kullanım alanını anlat.",
        "expected_language": "tr",
    },
]

contract_results = []
for case in contract_cases:
    print("\n" + "=" * 110)
    print(case["name"], "|", case["query"])
    result = pipeline.run(case["query"])
    contract_results.append({"name": case["name"], **result})
    print("Route:", result["route"])
    print("Plan:", json.dumps(result["plan"], ensure_ascii=False))
    print("Candidates:", [candidate["label"] for candidate in result["candidates"]])
    print("Validation:", result["validation"])
    if result.get("generation_failures"):
        print("Rejected generations:", result["generation_failures"])
    print("Total seconds:", result["timings"]["total_seconds"])
    print("\n" + result["answer"])
    assert result["validation"]["pass"], result
    assert result["response_language"] == case["expected_language"], result
    assert response_language_matches(result["answer"], case["expected_language"]), result

by_name = {row["name"]: row for row in contract_results}
clean_case = by_name["clean_office_not_clean_brand"]
assert "requested_brand" not in clean_case["plan"], clean_case["plan"]
assert clean_case["retrieval"]["semantic_query"], clean_case["retrieval"]
assert "3" not in clean_case["retrieval"]["semantic_query"], clean_case["retrieval"]
assert sum(candidate["brand"] == "Clean" for candidate in clean_case["candidates"]) <= 1
assert all(not candidate_has_term(candidate, "vanilla") for candidate in clean_case["candidates"])

versace_case = by_name["explicit_versace_brand"]
assert versace_case["plan"].get("requested_brand") == "Versace", versace_case["plan"]
assert all(candidate["brand"] == "Versace" for candidate in versace_case["candidates"])

similarity_case = by_name["aventus_similarity_less_smoky"]
assert similarity_case["plan"]["intent"] in {"similarity", "alternative"}
assert all(not candidate_has_term(candidate, "smoky") for candidate in similarity_case["candidates"])

comparison_case = by_name["canonical_comparison"]
assert comparison_case["plan"]["intent"] == "comparison", comparison_case["plan"]
assert len(comparison_case["candidates"]) == 2, comparison_case["candidates"]

turkish_profile_case = by_name["turkish_profile_language"]
turkish_leather = turkish_profile_case["candidates"][0]
assert "very strong" in calibrated_performance("longevity", turkish_leather["longevity"])
assert "noticeable/strong" in calibrated_performance("sillage", turkish_leather["sillage"])
assert not turkish_profile_case["validation"]["performance_calibration_violations"], turkish_profile_case["validation"]

report = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "stage": "stage3_full_grounded_pipeline",
    "inference_models": sorted(served_model_ids),
    "adapter_dir": str(ADAPTER_DIR),
    "retrieval_health": retrieval_client.health(),
    "all_contracts_passed": True,
    "results": contract_results,
}
REPORT_PATH = PROJECT_DIR / "runs" / "stage3_pipeline_report.json"
REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)
REPORT_PATH.write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding="utf-8")
print("\nSTAGE 3 END-TO-END CONTRACT TEST: PASSED")
print("Saved:", REPORT_PATH)

## Longevity and sillage calibration contracts

These cases force the advisor to interpret performance across very strong,
moderate, and light catalog bands in both Turkish and English. The report keeps
the parsed performance labels per candidate so a passing validator cannot hide
an omitted or vague performance discussion.

In [ ]:
from datetime import datetime, timezone

PERFORMANCE_CASES = [
    {
        "name": "tr_very_strong_profile",
        "query": "Turkish Leather by Pryn Parfum'un kalıcılığı ve yayılımı nasıl? İkisini de düşük, orta veya güçlü olarak yorumla.",
        "expected_language": "tr",
        "expected": {
            "Turkish Leather by Pryn Parfum": {"longevity": "high", "sillage": "high"},
        },
    },
    {
        "name": "en_light_profile",
        "query": "How are the longevity and projection of Team Five by Adidas? Classify both as light, moderate, or strong.",
        "expected_language": "en",
        "expected": {
            "Team Five by Adidas": {"longevity": "low", "sillage": "low"},
        },
    },
    {
        "name": "en_moderate_profile",
        "query": "How are the longevity and projection of Versace Pour Homme by Versace? Classify both as light, moderate, or strong.",
        "expected_language": "en",
        "expected": {
            "Versace Pour Homme by Versace": {"longevity": "moderate", "sillage": "moderate"},
        },
    },
    {
        "name": "en_high_vs_low_comparison",
        "query": "Compare the longevity and projection of Tobacco Vanille by Tom Ford with Team Five by Adidas. State whether each is light, moderate, or strong.",
        "expected_language": "en",
        "expected": {
            "Tobacco Vanille by Tom Ford": {"longevity": "high", "sillage": "high"},
            "Team Five by Adidas": {"longevity": "low", "sillage": "low"},
        },
    },
]

performance_results = []
for case in PERFORMANCE_CASES:
    print("\n" + "=" * 110)
    print(case["name"], "|", case["query"])
    result = pipeline.run(case["query"])
    sections = candidate_answer_sections(result["answer"], result["candidates"])
    observed = {
        candidate["label"]: {
            metric: sorted(explicit_performance_labels(sections.get(candidate["label"], ""), metric))
            for metric in ("longevity", "sillage")
        }
        for candidate in result["candidates"]
        if candidate["label"] in case["expected"]
    }
    calibration = {
        candidate["label"]: {
            metric: {
                "score": candidate.get(metric),
                "card_text": calibrated_performance(metric, candidate.get(metric)),
                "expected_group": expected_performance_group(metric, candidate.get(metric)),
            }
            for metric in ("longevity", "sillage")
        }
        for candidate in result["candidates"]
        if candidate["label"] in case["expected"]
    }
    row = {
        "name": case["name"],
        "expected": case["expected"],
        "observed_explicit_labels": observed,
        "calibration": calibration,
        **result,
    }
    performance_results.append(row)
    print("Route:", result["route"])
    print("Language:", result["response_language"])
    print("Calibration:", json.dumps(calibration, ensure_ascii=False, indent=2))
    print("Observed labels:", json.dumps(observed, ensure_ascii=False, indent=2))
    print("Validation:", result["validation"])
    print("Attempts:", result["generation_attempts"])
    print("Seconds:", result["timings"]["total_seconds"])
    print("\n" + result["answer"])

    assert result["response_language"] == case["expected_language"], result
    assert result["validation"]["pass"], result
    assert not result["validation"]["performance_calibration_violations"], result["validation"]
    assert set(calibration) == set(case["expected"]), calibration
    for label, expected_metrics in case["expected"].items():
        for metric, expected_group in expected_metrics.items():
            assert calibration[label][metric]["expected_group"] == expected_group, calibration[label]
            assert observed[label][metric] == [expected_group], {
                "message": "The answer did not state one clear calibrated performance class.",
                "case": case["name"],
                "candidate": label,
                "metric": metric,
                "expected": expected_group,
                "observed": observed[label][metric],
                "answer": result["answer"],
            }

performance_report = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "stage": "stage3_performance_calibration",
    "catalog_snapshot_count": 131930,
    "all_contracts_passed": True,
    "results": performance_results,
}
PERFORMANCE_REPORT_PATH = PROJECT_DIR / "runs" / "stage3_performance_calibration_report.json"
PERFORMANCE_REPORT_PATH.write_text(
    json.dumps(performance_report, indent=2, ensure_ascii=False),
    encoding="utf-8",
)
print("\nSTAGE 3 PERFORMANCE CALIBRATION CONTRACT: PASSED")
print("Saved:", PERFORMANCE_REPORT_PATH)

## Popularity and conversation-memory contracts

Balanced discovery combines semantic relevance with a catalog-wide popularity
pool; it does not hardcode perfume names. The session contract then verifies
that a natural follow-up keeps the previous constraints while excluding items
already recommended in the conversation.

In [ ]:
popular_vanilla = retrieval_client.search({
    "query": "versatile vanilla fragrance",
    "top_k": 10,
    "required_terms": ["vanilla"],
    "discovery_mode": "balanced",
})
popular_vanilla_candidates = [candidate_record(item) for item in popular_vanilla["results"]]
assert len(popular_vanilla_candidates) == 10, popular_vanilla_candidates
assert all(candidate_has_term(candidate, "vanilla") for candidate in popular_vanilla_candidates)
assert sum(int(candidate.get("popularity") or 0) >= 10_000 for candidate in popular_vanilla_candidates[:5]) >= 3, popular_vanilla_candidates[:5]
assert any(
    "catalog_popular" in item.get("reasons", {}).get("candidate_sources", [])
    for item in popular_vanilla["results"][:5]
), popular_vanilla["results"][:5]

contract_session = ScentAISession(pipeline)
first_turn = contract_session.run("Recommend exactly 3 popular fragrances that must have vanilla.")
second_turn = contract_session.run("I want three different options under the same requirements.")
assert first_turn["validation"]["pass"], first_turn
assert second_turn["validation"]["pass"], second_turn
assert second_turn["plan"]["conversation_action"] == "more_options", second_turn["plan"]
assert "vanilla" in second_turn["plan"].get("required_terms", []), second_turn["plan"]
assert second_turn["plan"].get("discovery_mode") == "mainstream", second_turn["plan"]
first_ids = set(first_turn["validation"].get("mentioned_candidate_ids", []))
second_ids = set(second_turn["validation"].get("mentioned_candidate_ids", []))
if not first_ids:
    first_labels = set(first_turn["validation"].get("mentioned_candidates", []))
    first_ids = {candidate["perfume_id"] for candidate in first_turn["candidates"] if candidate["label"] in first_labels}
if not second_ids:
    second_labels = set(second_turn["validation"].get("mentioned_candidates", []))
    second_ids = {candidate["perfume_id"] for candidate in second_turn["candidates"] if candidate["label"] in second_labels}
assert first_ids and second_ids and first_ids.isdisjoint(second_ids), {"first": first_ids, "second": second_ids}

conversation_report = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "popular_vanilla_labels": [candidate["label"] for candidate in popular_vanilla_candidates],
    "first_turn": first_turn,
    "second_turn": second_turn,
}
CONVERSATION_REPORT_PATH = PROJECT_DIR / "runs" / "stage3_popularity_conversation_contract.json"
CONVERSATION_REPORT_PATH.write_text(json.dumps(conversation_report, indent=2, ensure_ascii=False), encoding="utf-8")
print("STAGE 3 POPULARITY + CONVERSATION CONTRACT: PASSED")
print("Saved:", CONVERSATION_REPORT_PATH)

## Advisor A/B diagnosis

This one-time diagnostic uses the same planner and retrieval pipeline for three
answer variants: the previous LoRA prompt, the new advisor LoRA prompt, and the
new advisor prompt on base Gemma. Compare decision value, naturalness,
specificity, repetition, and grounding. Retrieval candidate fingerprints are
checked so prose is not compared across different shortlists.

In [ ]:
ADVISOR_AB_CASES = [
    "Bana vanilyalı, sıcak ama boğucu olmayan tam 3 parfüm öner. Seçeneklerin karakter farklarını anlat.",
    "Ofiste kullanabileceğim temiz ve profesyonel, vanilyasız tam 3 parfüm öner.",
    "Date akşamı için baharatlı ve vanilyalı tam 3 parfüm öner; hangisinin nasıl bir izlenim verdiğini anlat.",
]

from datetime import datetime, timezone

ADVISOR_AB_VARIANTS = [
    {"name": "legacy_lora", "model": LORA_NAME, "prompt": LEGACY_ANSWER_PROMPT},
    {"name": "advisor_lora", "model": LORA_NAME, "prompt": ADVISOR_ANSWER_PROMPT},
    {"name": "advisor_base", "model": MODEL_NAME, "prompt": ADVISOR_ANSWER_PROMPT},
]


def run_advisor_ab(cases=ADVISOR_AB_CASES):
    report_rows = []
    for case_index, query in enumerate(cases, 1):
        print("\n" + "=" * 118)
        print(f"A/B CASE {case_index}: {query}")
        variants = []
        candidate_fingerprint = None
        for variant in ADVISOR_AB_VARIANTS:
            result = pipeline.run(
                query,
                answer_prompt_override=variant["prompt"],
                answer_model_override=variant["model"],
            )
            fingerprint = [candidate["perfume_id"] for candidate in result.get("candidates", [])]
            if candidate_fingerprint is None:
                candidate_fingerprint = fingerprint
            assert fingerprint == candidate_fingerprint, {
                "message": "A/B variants received different retrieval candidates",
                "expected": candidate_fingerprint,
                "actual": fingerprint,
                "variant": variant["name"],
            }
            variants.append({"variant": variant["name"], **result})
            print("\n" + "-" * 118)
            print(
                variant["name"],
                "| route=", result["route"],
                "| valid=", result["validation"]["pass"],
                "| seconds=", result["timings"]["total_seconds"],
            )
            print(result["answer"])
        report_rows.append({
            "query": query,
            "candidate_fingerprint": candidate_fingerprint,
            "candidate_labels": [candidate["label"] for candidate in variants[0]["candidates"]],
            "variants": variants,
        })

    report = {
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "stage": "stage3_advisor_ab",
        "rubric": [
            "decision_value",
            "natural_warmth",
            "query_specificity",
            "meaningful_contrast",
            "low_template_repetition",
            "grounding_and_filter_compliance",
        ],
        "cases": report_rows,
    }
    path = PROJECT_DIR / "runs" / "stage3_advisor_ab_report.json"
    path.write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding="utf-8")
    print("\nADVISOR A/B DIAGNOSIS COMPLETE")
    print("Saved:", path)
    return report


advisor_ab_report = run_advisor_ab()

## Retrieval bias audit

This fast red-team suite checks whether explicit traits are satisfied by actual
accord/note fields rather than merely echoed in perfume names. It also checks
brand diversity and the known Clean/clean collision. No LLM generation is used.

In [ ]:
bias_cases = [
    {"trait": "vanilla", "semantic_query": "romantic date night fragrance"},
    {"trait": "spicy", "semantic_query": "energetic date night fragrance"},
    {"trait": "oud", "semantic_query": "dark resinous evening fragrance"},
    {"trait": "rose", "semantic_query": "elegant floral spring fragrance"},
    {"trait": "musk", "semantic_query": "soft intimate skin scent"},
    {"trait": "amber", "semantic_query": "warm enveloping evening fragrance"},
    {"trait": "leather", "semantic_query": "confident dressed-up evening fragrance"},
    {"trait": "coffee", "semantic_query": "cozy gourmand winter fragrance"},
    {"trait": "coconut", "semantic_query": "tropical summer holiday fragrance"},
    {"trait": "pineapple", "semantic_query": "bright fruity summer fragrance"},
]

bias_rows = []
for case in bias_cases:
    response = retrieval_client.search({
        "query": case["semantic_query"],
        "top_k": 10,
        "required_terms": [case["trait"]],
    })
    candidates = [candidate_record(item) for item in response["results"]]
    compliant = [candidate for candidate in candidates if candidate_has_term(candidate, case["trait"])]
    name_echoes = [
        candidate for candidate in candidates
        if f" {normalize_text(case['trait'])} " in f" {normalize_text(candidate['name'])} "
    ]
    row = {
        **case,
        "result_count": len(candidates),
        "trait_compliance_rate": round(len(compliant) / len(candidates), 4) if candidates else 0.0,
        "name_echo_rate": round(len(name_echoes) / len(candidates), 4) if candidates else 0.0,
        "unique_brand_count": len({candidate["brand"] for candidate in candidates}),
        "labels": [candidate["label"] for candidate in candidates],
    }
    row["pass"] = (
        row["result_count"] >= 5
        and row["trait_compliance_rate"] == 1.0
        and row["name_echo_rate"] <= 0.6
        and row["unique_brand_count"] >= 4
    )
    bias_rows.append(row)
    print(
        case["trait"],
        "count=", row["result_count"],
        "trait=", row["trait_compliance_rate"],
        "name_echo=", row["name_echo_rate"],
        "brands=", row["unique_brand_count"],
        "pass=", row["pass"],
    )

clean_response = retrieval_client.search({
    "query": "professional clean office fragrance",
    "top_k": 10,
    "exclude_terms": ["vanilla"],
})
clean_candidates = [candidate_record(item) for item in clean_response["results"]]
clean_guard = {
    "clean_brand_count": sum(candidate["brand"] == "Clean" for candidate in clean_candidates),
    "vanilla_violation_count": sum(candidate_has_term(candidate, "vanilla") for candidate in clean_candidates),
    "labels": [candidate["label"] for candidate in clean_candidates],
}
clean_guard["pass"] = clean_guard["clean_brand_count"] <= 1 and clean_guard["vanilla_violation_count"] == 0

bias_report = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "stage": "stage3_retrieval_bias_audit",
    "all_passed": all(row["pass"] for row in bias_rows) and clean_guard["pass"],
    "trait_cases": bias_rows,
    "clean_brand_collision": clean_guard,
}
BIAS_REPORT_PATH = PROJECT_DIR / "runs" / "stage3_retrieval_bias_audit.json"
BIAS_REPORT_PATH.write_text(json.dumps(bias_report, indent=2, ensure_ascii=False), encoding="utf-8")
assert bias_report["all_passed"], bias_report
print("\nSTAGE 3 RETRIEVAL BIAS AUDIT: PASSED")
print("Saved:", BIAS_REPORT_PATH)

## Final evaluation v1 - 120 frozen cases

This is the release-quality evaluation pass. It covers recommendation, perfume
profiles, comparisons, similarity, hard filters, entity resolution, stateful
conversation, unsupported requests, and noisy Turkish/English input. Every row
is flushed to Drive immediately and the runner resumes completed cases after a
disconnect. It writes an automatic summary and a stratified 40-row human-review
CSV.

In [ ]:
FINAL_EVAL_SET_SOURCE = '{"id": "fev1_rec_001", "version": "final_eval_v1", "category": "recommendation", "language": "en", "query": "I need exactly 3 clean professional office fragrances without vanilla.", "tags": [], "expected": {"response_language": "en", "intent_in": ["recommendation"], "requested_count": 3, "required_terms": [], "excluded_terms": ["vanilla"]}}\n{"id": "fev1_rec_002", "version": "final_eval_v1", "category": "recommendation", "language": "tr", "query": "Date akşamı için vanilya ve baharat mutlaka bulunan tam 3 parfüm öner.", "tags": [], "expected": {"response_language": "tr", "intent_in": ["recommendation"], "requested_count": 3, "required_terms": ["vanilla", "spicy"], "excluded_terms": [], "time_profile": "night"}}\n{"id": "fev1_rec_003", "version": "final_eval_v1", "category": "recommendation", "language": "en", "query": "Recommend exactly 3 fresh citrus summer fragrances for men.", "tags": [], "expected": {"response_language": "en", "intent_in": ["recommendation"], "requested_count": 3, "required_terms": [], "excluded_terms": [], "gender": "male", "season": "summer"}}\n{"id": "fev1_rec_004", "version": "final_eval_v1", "category": "recommendation", "language": "tr", "query": "Kadınlar için kış gecesine uygun, vanilyalı tam 3 parfüm öner.", "tags": [], "expected": {"response_language": "tr", "intent_in": ["recommendation"], "requested_count": 3, "required_terms": ["vanilla"], "excluded_terms": [], "gender": "female", "season": "winter", "time_profile": "night"}}\n{"id": "fev1_rec_005", "version": "final_eval_v1", "category": "recommendation", "language": "en", "query": "Recommend exactly 3 popular and recognizable woody fragrances for versatile wear.", "tags": [], "expected": {"response_language": "en", "intent_in": ["recommendation"], "requested_count": 3, "required_terms": [], "excluded_terms": [], "discovery_mode": "mainstream"}}\n{"id": "fev1_rec_006", "version": "final_eval_v1", "category": "recommendation", "language": "tr", "query": "İlkbahar için az bilinen, niş ve yeşil karakterli tam 3 parfüm öner.", "tags": [], "expected": {"response_language": "tr", "intent_in": ["recommendation"], "requested_count": 3, "required_terms": [], "excluded_terms": [], "discovery_mode": "niche", "season": "spring"}}\n{"id": "fev1_rec_007", "version": "final_eval_v1", "category": "recommendation", "language": "en", "query": "Recommend exactly 3 unisex smoky fragrances for autumn nights.", "tags": [], "expected": {"response_language": "en", "intent_in": ["recommendation"], "requested_count": 3, "required_terms": [], "excluded_terms": [], "gender": "unisex", "season": "autumn", "time_profile": "night"}}\n{"id": "fev1_rec_008", "version": "final_eval_v1", "category": "recommendation", "language": "tr", "query": "Hindistan cevizi mutlaka bulunan tropikal yazlık tam 3 parfüm öner.", "tags": [], "expected": {"response_language": "tr", "intent_in": ["recommendation"], "requested_count": 3, "required_terms": ["coconut"], "excluded_terms": [], "season": "summer"}}\n{"id": "fev1_rec_009", "version": "final_eval_v1", "category": "recommendation", "language": "en", "query": "Recommend exactly 3 spring fragrances for women that must feature rose.", "tags": [], "expected": {"response_language": "en", "intent_in": ["recommendation"], "requested_count": 3, "required_terms": ["rose"], "excluded_terms": [], "gender": "female", "season": "spring"}}\n{"id": "fev1_rec_010", "version": "final_eval_v1", "category": "recommendation", "language": "tr", "query": "Erkekler için oud ve deri mutlaka bulunan, akşama uygun tam 3 parfüm öner.", "tags": [], "expected": {"response_language": "tr", "intent_in": ["recommendation"], "requested_count": 3, "required_terms": ["oud", "leather"], "excluded_terms": [], "gender": "male", "time_profile": "night"}}\n{"id": "fev1_rec_011", "version": "final_eval_v1", "category": "recommendation", "language": "en", "query": "Recommend exactly 3 cozy winter fragrances that must have coffee.", "tags": [], "expected": {"response_language": "en", "intent_in": ["recommendation"], "requested_count": 3, "required_terms": ["coffee"], "excluded_terms": [], "season": "winter"}}\n{"id": "fev1_rec_012", "version": "final_eval_v1", "category": "recommendation", "language": "tr", "query": "Misk mutlaka bulunan, tene yakın ve temiz hissettiren tam 3 parfüm öner.", "tags": [], "expected": {"response_language": "tr", "intent_in": ["recommendation"], "requested_count": 3, "required_terms": ["musk"], "excluded_terms": []}}\n{"id": "fev1_rec_013", "version": "final_eval_v1", "category": "recommendation", "language": "en", "query": "Recommend exactly 3 aromatic men\'s fragrances for daytime gym wear.", "tags": [], "expected": {"response_language": "en", "intent_in": ["recommendation"], "requested_count": 3, "required_terms": [], "excluded_terms": [], "gender": "male", "time_profile": "day"}}\n{"id": "fev1_rec_014", "version": "final_eval_v1", "category": "recommendation", "language": "tr", "query": "Ofis için pudralı ve iris karakterli tam 3 parfüm öner.", "tags": [], "expected": {"response_language": "tr", "intent_in": ["recommendation"], "requested_count": 3, "required_terms": ["iris"], "excluded_terms": []}}\n{"id": "fev1_rec_015", "version": "final_eval_v1", "category": "recommendation", "language": "en", "query": "Recommend exactly 3 unisex evening fragrances that must have amber.", "tags": [], "expected": {"response_language": "en", "intent_in": ["recommendation"], "requested_count": 3, "required_terms": ["amber"], "excluded_terms": [], "gender": "unisex", "time_profile": "night"}}\n{"id": "fev1_rec_016", "version": "final_eval_v1", "category": "recommendation", "language": "tr", "query": "Günlük kullanım için ferah baharatlı tam 3 erkek parfümü öner.", "tags": [], "expected": {"response_language": "tr", "intent_in": ["recommendation"], "requested_count": 3, "required_terms": [], "excluded_terms": [], "gender": "male"}}\n{"id": "fev1_rec_017", "version": "final_eval_v1", "category": "recommendation", "language": "en", "query": "Recommend exactly 3 popular winter-night fragrances that must feature tobacco.", "tags": [], "expected": {"response_language": "en", "intent_in": ["recommendation"], "requested_count": 3, "required_terms": ["tobacco"], "excluded_terms": [], "discovery_mode": "mainstream", "season": "winter", "time_profile": "night"}}\n{"id": "fev1_rec_018", "version": "final_eval_v1", "category": "recommendation", "language": "tr", "query": "Tütsü mutlaka bulunan, sıra dışı ve niş tam 3 parfüm öner.", "tags": [], "expected": {"response_language": "tr", "intent_in": ["recommendation"], "requested_count": 3, "required_terms": ["incense"], "excluded_terms": [], "discovery_mode": "niche"}}\n{"id": "fev1_rec_019", "version": "final_eval_v1", "category": "recommendation", "language": "en", "query": "Recommend exactly 3 fruity sweet date fragrances with no vanilla.", "tags": [], "expected": {"response_language": "en", "intent_in": ["recommendation"], "requested_count": 3, "required_terms": [], "excluded_terms": ["vanilla"], "time_profile": "night"}}\n{"id": "fev1_rec_020", "version": "final_eval_v1", "category": "recommendation", "language": "tr", "query": "Yaz için narenciye mutlaka bulunan ama misk içermeyen tam 3 parfüm öner.", "tags": [], "expected": {"response_language": "tr", "intent_in": ["recommendation"], "requested_count": 3, "required_terms": ["citrus"], "excluded_terms": ["musk"], "season": "summer"}}\n{"id": "fev1_profile_001", "version": "final_eval_v1", "category": "perfume_profile", "language": "tr", "query": "Turkish Leather by Pryn Parfum nasıl bir parfüm? Karakterini ve performansını anlat.", "tags": [], "expected": {"response_language": "tr", "intent_in": ["perfume_profile"], "resolved_labels": ["Turkish Leather by Pryn Parfum"]}}\n{"id": "fev1_profile_002", "version": "final_eval_v1", "category": "perfume_profile", "language": "en", "query": "What is Tobacco Vanille by Tom Ford like? Explain its character and performance.", "tags": [], "expected": {"response_language": "en", "intent_in": ["perfume_profile"], "resolved_labels": ["Tobacco Vanille by Tom Ford"]}}\n{"id": "fev1_profile_003", "version": "final_eval_v1", "category": "perfume_profile", "language": "tr", "query": "Team Five by Adidas nasıl bir parfüm? En uygun kullanım alanını anlat.", "tags": [], "expected": {"response_language": "tr", "intent_in": ["perfume_profile"], "resolved_labels": ["Team Five by Adidas"]}}\n{"id": "fev1_profile_004", "version": "final_eval_v1", "category": "perfume_profile", "language": "en", "query": "Tell me what Versace Pour Homme by Versace is like and where it works best.", "tags": [], "expected": {"response_language": "en", "intent_in": ["perfume_profile"], "resolved_labels": ["Versace Pour Homme by Versace"]}}\n{"id": "fev1_profile_005", "version": "final_eval_v1", "category": "perfume_profile", "language": "tr", "query": "Imagination by Louis Vuitton hakkında danışman gibi bilgi ver.", "tags": [], "expected": {"response_language": "tr", "intent_in": ["perfume_profile"], "resolved_labels": ["Imagination by Louis Vuitton"]}}\n{"id": "fev1_profile_006", "version": "final_eval_v1", "category": "perfume_profile", "language": "en", "query": "Describe the vibe, performance, and best use cases of Santal 33 by Le Labo.", "tags": [], "expected": {"response_language": "en", "intent_in": ["perfume_profile"], "resolved_labels": ["Santal 33 by Le Labo"]}}\n{"id": "fev1_profile_007", "version": "final_eval_v1", "category": "perfume_profile", "language": "tr", "query": "Coco Mademoiselle by Chanel nasıl bir karaktere sahip?", "tags": [], "expected": {"response_language": "tr", "intent_in": ["perfume_profile"], "resolved_labels": ["Coco Mademoiselle by Chanel"]}}\n{"id": "fev1_profile_008", "version": "final_eval_v1", "category": "perfume_profile", "language": "en", "query": "What does Terre d\'Hermès by Hermès feel like, and when should it be worn?", "tags": [], "expected": {"response_language": "en", "intent_in": ["perfume_profile"], "resolved_labels": ["Terre d\'Hermès by Hermès"]}}\n{"id": "fev1_profile_009", "version": "final_eval_v1", "category": "perfume_profile", "language": "tr", "query": "Spicebomb Extreme by Viktor&Rolf\'un karakterini ve kullanımını anlat.", "tags": [], "expected": {"response_language": "tr", "intent_in": ["perfume_profile"], "resolved_labels": ["Spicebomb Extreme by Viktor&Rolf"]}}\n{"id": "fev1_profile_010", "version": "final_eval_v1", "category": "perfume_profile", "language": "en", "query": "Give me a grounded profile of Grand Soir by Maison Francis Kurkdjian.", "tags": [], "expected": {"response_language": "en", "intent_in": ["perfume_profile"], "resolved_labels": ["Grand Soir by Maison Francis Kurkdjian"]}}\n{"id": "fev1_profile_011", "version": "final_eval_v1", "category": "perfume_profile", "language": "tr", "query": "By the Fireplace by Maison Martin Margiela nasıl bir atmosfer yaratıyor?", "tags": [], "expected": {"response_language": "tr", "intent_in": ["perfume_profile"], "resolved_labels": ["By the Fireplace by Maison Martin Margiela"]}}\n{"id": "fev1_profile_012", "version": "final_eval_v1", "category": "perfume_profile", "language": "en", "query": "Explain the character and practical wear of Prada L\'Homme by Prada.", "tags": [], "expected": {"response_language": "en", "intent_in": ["perfume_profile"], "resolved_labels": ["Prada L\'Homme by Prada"]}}\n{"id": "fev1_profile_013", "version": "final_eval_v1", "category": "perfume_profile", "language": "tr", "query": "Y Eau de Parfum by Yves Saint Laurent hakkında detaylı ama kısa bilgi ver.", "tags": [], "expected": {"response_language": "tr", "intent_in": ["perfume_profile"], "resolved_labels": ["Y Eau de Parfum by Yves Saint Laurent"]}}\n{"id": "fev1_profile_014", "version": "final_eval_v1", "category": "perfume_profile", "language": "en", "query": "What is Reflection Man by Amouage like as a personal style choice?", "tags": [], "expected": {"response_language": "en", "intent_in": ["perfume_profile"], "resolved_labels": ["Reflection Man by Amouage"]}}\n{"id": "fev1_profile_015", "version": "final_eval_v1", "category": "perfume_profile", "language": "tr", "query": "Club de Nuit Intense Man by Armaf nasıl bir parfüm ve performansı nasıl?", "tags": [], "expected": {"response_language": "tr", "intent_in": ["perfume_profile"], "resolved_labels": ["Club de Nuit Intense Man by Armaf"]}}\n{"id": "fev1_cmp_001", "version": "final_eval_v1", "category": "comparison", "language": "en", "query": "Compare Club de Nuit by Armaf with Team Five by Adidas for vibe, performance, and use.", "tags": [], "expected": {"response_language": "en", "intent_in": ["comparison"], "resolved_labels": ["Club de Nuit Intense Man by Armaf", "Team Five by Adidas"]}}\n{"id": "fev1_cmp_002", "version": "final_eval_v1", "category": "comparison", "language": "tr", "query": "Burberry Hero ile Tobacco Vanille by Tom Ford\'u karakter ve kullanım açısından karşılaştır.", "tags": [], "expected": {"response_language": "tr", "intent_in": ["comparison"], "resolved_labels": ["Hero by Burberry", "Tobacco Vanille by Tom Ford"]}}\n{"id": "fev1_cmp_003", "version": "final_eval_v1", "category": "comparison", "language": "en", "query": "Compare Imagination by Louis Vuitton and Versace Pour Homme by Versace for office wear.", "tags": [], "expected": {"response_language": "en", "intent_in": ["comparison"], "resolved_labels": ["Imagination by Louis Vuitton", "Versace Pour Homme by Versace"]}}\n{"id": "fev1_cmp_004", "version": "final_eval_v1", "category": "comparison", "language": "tr", "query": "Santal 33 by Le Labo ile Prada L\'Homme by Prada\'yı tarz ve kullanım açısından karşılaştır.", "tags": [], "expected": {"response_language": "tr", "intent_in": ["comparison"], "resolved_labels": ["Santal 33 by Le Labo", "Prada L\'Homme by Prada"]}}\n{"id": "fev1_cmp_005", "version": "final_eval_v1", "category": "comparison", "language": "en", "query": "Compare Spicebomb Extreme by Viktor&Rolf with Le Male Le Parfum by Jean Paul Gaultier for a date night.", "tags": [], "expected": {"response_language": "en", "intent_in": ["comparison"], "resolved_labels": ["Spicebomb Extreme by Viktor&Rolf", "Le Male Le Parfum by Jean Paul Gaultier"]}}\n{"id": "fev1_cmp_006", "version": "final_eval_v1", "category": "comparison", "language": "tr", "query": "Grand Soir ile By the Fireplace\'ı sıcaklık, karakter ve ortam açısından karşılaştır.", "tags": [], "expected": {"response_language": "tr", "intent_in": ["comparison"], "resolved_labels": ["Grand Soir by Maison Francis Kurkdjian", "By the Fireplace by Maison Martin Margiela"]}}\n{"id": "fev1_cmp_007", "version": "final_eval_v1", "category": "comparison", "language": "en", "query": "Compare Aventus by Creed and Explorer by Montblanc without declaring a winner.", "tags": [], "expected": {"response_language": "en", "intent_in": ["comparison"], "resolved_labels": ["Aventus by Creed", "Explorer by Montblanc"]}}\n{"id": "fev1_cmp_008", "version": "final_eval_v1", "category": "comparison", "language": "tr", "query": "Coco Mademoiselle ile Hypnotic Poison\'ı verdikleri izlenim açısından karşılaştır.", "tags": [], "expected": {"response_language": "tr", "intent_in": ["comparison"], "resolved_labels": ["Coco Mademoiselle by Chanel", "Hypnotic Poison by Dior"]}}\n{"id": "fev1_cmp_009", "version": "final_eval_v1", "category": "comparison", "language": "en", "query": "Compare Terre d\'Hermès by Hermès with Bleu de Chanel Eau de Parfum by Chanel for professional wear.", "tags": [], "expected": {"response_language": "en", "intent_in": ["comparison"], "resolved_labels": ["Terre d\'Hermès by Hermès", "Bleu de Chanel Eau de Parfum by Chanel"]}}\n{"id": "fev1_cmp_010", "version": "final_eval_v1", "category": "comparison", "language": "tr", "query": "Versace Eros ile Eros Flame\'i karakter ve mevsim açısından karşılaştır.", "tags": [], "expected": {"response_language": "tr", "intent_in": ["comparison"], "resolved_labels": ["Eros by Versace", "Eros Flame by Versace"]}}\n{"id": "fev1_cmp_011", "version": "final_eval_v1", "category": "comparison", "language": "en", "query": "Compare Reflection Man by Amouage and Dior Homme Intense 2011 by Dior for formal occasions.", "tags": [], "expected": {"response_language": "en", "intent_in": ["comparison"], "resolved_labels": ["Reflection Man by Amouage", "Dior Homme Intense 2011 by Dior"]}}\n{"id": "fev1_cmp_012", "version": "final_eval_v1", "category": "comparison", "language": "tr", "query": "Light Blue pour Homme ile Acqua di Giò Profondo\'yu yaz kullanımı için karşılaştır.", "tags": [], "expected": {"response_language": "tr", "intent_in": ["comparison"], "resolved_labels": ["Light Blue pour Homme by Dolce&Gabbana", "Acqua di Giò Profondo by Giorgio Armani"]}}\n{"id": "fev1_cmp_013", "version": "final_eval_v1", "category": "comparison", "language": "en", "query": "Compare Angels\' Share by By Kilian and Khamrah Qahwa by Lattafa Perfumes as gourmand evening scents.", "tags": [], "expected": {"response_language": "en", "intent_in": ["comparison"], "resolved_labels": ["Angels\' Share by By Kilian", "Khamrah Qahwa by Lattafa Perfumes"]}}\n{"id": "fev1_cmp_014", "version": "final_eval_v1", "category": "comparison", "language": "tr", "query": "Black Orchid ile Libre Intense\'i gece kullanımı ve karakter açısından karşılaştır.", "tags": [], "expected": {"response_language": "tr", "intent_in": ["comparison"], "resolved_labels": ["Black Orchid by Tom Ford", "Libre Intense by Yves Saint Laurent"]}}\n{"id": "fev1_cmp_015", "version": "final_eval_v1", "category": "comparison", "language": "en", "query": "Compare Oud for Greatness and Ombre Nomade for oud character, performance, and occasion.", "tags": [], "expected": {"response_language": "en", "intent_in": ["comparison"], "resolved_labels": ["Oud for Greatness by Initio Parfums Prives", "Ombre Nomade by Louis Vuitton"]}}\n{"id": "fev1_sim_001", "version": "final_eval_v1", "category": "similarity_alternative", "language": "en", "query": "I want exactly 3 fragrances similar to Aventus by Creed but less smoky.", "tags": [], "expected": {"response_language": "en", "intent_in": ["similarity", "alternative"], "requested_count": 3, "reference_label": "Aventus by Creed", "excluded_terms": ["smoky"]}}\n{"id": "fev1_sim_002", "version": "final_eval_v1", "category": "similarity_alternative", "language": "tr", "query": "Tobacco Vanille\'e benzeyen ama tütünsüz tam 3 seçenek öner.", "tags": [], "expected": {"response_language": "tr", "intent_in": ["similarity", "alternative"], "requested_count": 3, "reference_label": "Tobacco Vanille by Tom Ford", "excluded_terms": ["tobacco"]}}\n{"id": "fev1_sim_003", "version": "final_eval_v1", "category": "similarity_alternative", "language": "en", "query": "Give me exactly 3 fragrances similar to Santal 33 by Le Labo but without leather.", "tags": [], "expected": {"response_language": "en", "intent_in": ["similarity", "alternative"], "requested_count": 3, "reference_label": "Santal 33 by Le Labo", "excluded_terms": ["leather"]}}\n{"id": "fev1_sim_004", "version": "final_eval_v1", "category": "similarity_alternative", "language": "tr", "query": "Bleu de Chanel EDP\'ye alternatif, daha narenciyeli tam 3 parfüm öner.", "tags": [], "expected": {"response_language": "tr", "intent_in": ["similarity", "alternative"], "requested_count": 3, "reference_label": "Bleu de Chanel Eau de Parfum by Chanel", "excluded_terms": []}}\n{"id": "fev1_sim_005", "version": "final_eval_v1", "category": "similarity_alternative", "language": "en", "query": "Find exactly 3 scents like By the Fireplace by Maison Martin Margiela but less smoky.", "tags": [], "expected": {"response_language": "en", "intent_in": ["similarity", "alternative"], "requested_count": 3, "reference_label": "By the Fireplace by Maison Martin Margiela", "excluded_terms": ["smoky"]}}\n{"id": "fev1_sim_006", "version": "final_eval_v1", "category": "similarity_alternative", "language": "tr", "query": "Montblanc Explorer\'a benzeyen ama daha meyvemsi tam 3 parfüm öner.", "tags": [], "expected": {"response_language": "tr", "intent_in": ["similarity", "alternative"], "requested_count": 3, "reference_label": "Explorer by Montblanc", "excluded_terms": []}}\n{"id": "fev1_sim_007", "version": "final_eval_v1", "category": "similarity_alternative", "language": "en", "query": "Give me exactly 3 alternatives to Grand Soir by Maison Francis Kurkdjian with a softer amber profile.", "tags": [], "expected": {"response_language": "en", "intent_in": ["similarity", "alternative"], "requested_count": 3, "reference_label": "Grand Soir by Maison Francis Kurkdjian", "excluded_terms": []}}\n{"id": "fev1_sim_008", "version": "final_eval_v1", "category": "similarity_alternative", "language": "tr", "query": "Prada L\'Homme\'a benzeyen ama pudralı olmayan tam 3 parfüm öner.", "tags": [], "expected": {"response_language": "tr", "intent_in": ["similarity", "alternative"], "requested_count": 3, "reference_label": "Prada L\'Homme by Prada", "excluded_terms": ["powdery"]}}\n{"id": "fev1_sim_009", "version": "final_eval_v1", "category": "similarity_alternative", "language": "en", "query": "Recommend exactly 3 fragrances similar to Imagination by Louis Vuitton but woodier.", "tags": [], "expected": {"response_language": "en", "intent_in": ["similarity", "alternative"], "requested_count": 3, "reference_label": "Imagination by Louis Vuitton", "excluded_terms": []}}\n{"id": "fev1_sim_010", "version": "final_eval_v1", "category": "similarity_alternative", "language": "tr", "query": "Light Blue pour Homme benzeri ama daha karakterli tam 3 seçenek ver.", "tags": [], "expected": {"response_language": "tr", "intent_in": ["similarity", "alternative"], "requested_count": 3, "reference_label": "Light Blue pour Homme by Dolce&Gabbana", "excluded_terms": []}}\n{"id": "fev1_sim_011", "version": "final_eval_v1", "category": "similarity_alternative", "language": "en", "query": "Find exactly 3 alternatives to Eros by Versace without vanilla.", "tags": [], "expected": {"response_language": "en", "intent_in": ["similarity", "alternative"], "requested_count": 3, "reference_label": "Eros by Versace", "excluded_terms": ["vanilla"]}}\n{"id": "fev1_sim_012", "version": "final_eval_v1", "category": "similarity_alternative", "language": "tr", "query": "Coco Mademoiselle\'e benzeyen ama gül içermeyen tam 3 parfüm öner.", "tags": [], "expected": {"response_language": "tr", "intent_in": ["similarity", "alternative"], "requested_count": 3, "reference_label": "Coco Mademoiselle by Chanel", "excluded_terms": ["rose"]}}\n{"id": "fev1_sim_013", "version": "final_eval_v1", "category": "similarity_alternative", "language": "en", "query": "Recommend exactly 3 scents like Spicebomb Extreme by Viktor&Rolf without tobacco.", "tags": [], "expected": {"response_language": "en", "intent_in": ["similarity", "alternative"], "requested_count": 3, "reference_label": "Spicebomb Extreme by Viktor&Rolf", "excluded_terms": ["tobacco"]}}\n{"id": "fev1_sim_014", "version": "final_eval_v1", "category": "similarity_alternative", "language": "tr", "query": "Reflection Man\'e benzeyen ama daha narenciyeli tam 3 seçenek öner.", "tags": [], "expected": {"response_language": "tr", "intent_in": ["similarity", "alternative"], "requested_count": 3, "reference_label": "Reflection Man by Amouage", "excluded_terms": []}}\n{"id": "fev1_sim_015", "version": "final_eval_v1", "category": "similarity_alternative", "language": "en", "query": "Give me exactly 3 alternatives to Angels\' Share by By Kilian without cinnamon.", "tags": [], "expected": {"response_language": "en", "intent_in": ["similarity", "alternative"], "requested_count": 3, "reference_label": "Angels\' Share by By Kilian", "excluded_terms": ["cinnamon"]}}\n{"id": "fev1_filter_001", "version": "final_eval_v1", "category": "hard_filters", "language": "en", "query": "Recommend exactly 3 office fragrances with no vanilla, oud, or smoky accords.", "tags": [], "expected": {"response_language": "en", "intent_in": ["recommendation"], "requested_count": 3, "required_terms": [], "excluded_terms": ["vanilla", "oud", "smoky"]}}\n{"id": "fev1_filter_002", "version": "final_eval_v1", "category": "hard_filters", "language": "tr", "query": "Date akşamı için vanilya mutlaka bulunan ama tütün içermeyen tam 3 parfüm öner.", "tags": [], "expected": {"response_language": "tr", "intent_in": ["recommendation"], "requested_count": 3, "required_terms": ["vanilla"], "excluded_terms": ["tobacco"], "time_profile": "night"}}\n{"id": "fev1_filter_003", "version": "final_eval_v1", "category": "hard_filters", "language": "en", "query": "Recommend exactly 3 rose fragrances that must not contain musk.", "tags": [], "expected": {"response_language": "en", "intent_in": ["recommendation"], "requested_count": 3, "required_terms": ["rose"], "excluded_terms": ["musk"]}}\n{"id": "fev1_filter_004", "version": "final_eval_v1", "category": "hard_filters", "language": "tr", "query": "Narenciye mutlaka bulunan ama çiçeksi olmayan tam 3 parfüm öner.", "tags": [], "expected": {"response_language": "tr", "intent_in": ["recommendation"], "requested_count": 3, "required_terms": ["citrus"], "excluded_terms": ["floral"]}}\n{"id": "fev1_filter_005", "version": "final_eval_v1", "category": "hard_filters", "language": "en", "query": "Recommend exactly 3 coconut fragrances without vanilla.", "tags": [], "expected": {"response_language": "en", "intent_in": ["recommendation"], "requested_count": 3, "required_terms": ["coconut"], "excluded_terms": ["vanilla"]}}\n{"id": "fev1_filter_006", "version": "final_eval_v1", "category": "hard_filters", "language": "tr", "query": "Kahve mutlaka bulunan ama karamel içermeyen tam 3 parfüm öner.", "tags": [], "expected": {"response_language": "tr", "intent_in": ["recommendation"], "requested_count": 3, "required_terms": ["coffee"], "excluded_terms": ["caramel"]}}\n{"id": "fev1_filter_007", "version": "final_eval_v1", "category": "hard_filters", "language": "en", "query": "Recommend exactly 4 men\'s fragrances from Versace without vanilla.", "tags": [], "expected": {"response_language": "en", "intent_in": ["recommendation"], "requested_count": 4, "required_terms": [], "excluded_terms": ["vanilla"], "requested_brand": "Versace", "gender": "male"}}\n{"id": "fev1_filter_008", "version": "final_eval_v1", "category": "hard_filters", "language": "tr", "query": "Kadınlar için ilkbahara uygun ama paçuli içermeyen tam 3 parfüm öner.", "tags": [], "expected": {"response_language": "tr", "intent_in": ["recommendation"], "requested_count": 3, "required_terms": [], "excluded_terms": ["patchouli"], "gender": "female", "season": "spring"}}\n{"id": "fev1_filter_009", "version": "final_eval_v1", "category": "hard_filters", "language": "en", "query": "Recommend exactly 3 unisex winter fragrances that must have oud but no rose.", "tags": [], "expected": {"response_language": "en", "intent_in": ["recommendation"], "requested_count": 3, "required_terms": ["oud"], "excluded_terms": ["rose"], "gender": "unisex", "season": "winter"}}\n{"id": "fev1_filter_010", "version": "final_eval_v1", "category": "hard_filters", "language": "tr", "query": "Erkekler için aromatik akor mutlaka bulunan ama deri içermeyen tam 3 parfüm öner.", "tags": [], "expected": {"response_language": "tr", "intent_in": ["recommendation"], "requested_count": 3, "required_terms": ["aromatic"], "excluded_terms": ["leather"], "gender": "male"}}\n{"id": "fev1_filter_011", "version": "final_eval_v1", "category": "hard_filters", "language": "en", "query": "Recommend exactly 3 iris fragrances without vanilla.", "tags": [], "expected": {"response_language": "en", "intent_in": ["recommendation"], "requested_count": 3, "required_terms": ["iris"], "excluded_terms": ["vanilla"]}}\n{"id": "fev1_filter_012", "version": "final_eval_v1", "category": "hard_filters", "language": "tr", "query": "Ananas mutlaka bulunan ama dumanlı olmayan tam 3 parfüm öner.", "tags": [], "expected": {"response_language": "tr", "intent_in": ["recommendation"], "requested_count": 3, "required_terms": ["pineapple"], "excluded_terms": ["smoky"]}}\n{"id": "fev1_filter_013", "version": "final_eval_v1", "category": "hard_filters", "language": "en", "query": "Recommend exactly 3 incense fragrances without oud.", "tags": [], "expected": {"response_language": "en", "intent_in": ["recommendation"], "requested_count": 3, "required_terms": ["incense"], "excluded_terms": ["oud"]}}\n{"id": "fev1_filter_014", "version": "final_eval_v1", "category": "hard_filters", "language": "tr", "query": "Yeşil karakterli ama tatlı olmayan tam 3 parfüm öner.", "tags": [], "expected": {"response_language": "tr", "intent_in": ["recommendation"], "requested_count": 3, "required_terms": ["green"], "excluded_terms": ["sweet"]}}\n{"id": "fev1_filter_015", "version": "final_eval_v1", "category": "hard_filters", "language": "en", "query": "Recommend exactly 3 leather fragrances without animalic accords.", "tags": [], "expected": {"response_language": "en", "intent_in": ["recommendation"], "requested_count": 3, "required_terms": ["leather"], "excluded_terms": ["animalic"]}}\n{"id": "fev1_entity_001", "version": "final_eval_v1", "category": "entity_resolution", "language": "tr", "query": "Club de Nuit by Armaf nasıl bir parfüm?", "tags": [], "expected": {"response_language": "tr", "intent_in": ["perfume_profile"], "resolved_labels": ["Club de Nuit Intense Man by Armaf"]}}\n{"id": "fev1_entity_002", "version": "final_eval_v1", "category": "entity_resolution", "language": "en", "query": "Tell me about Aventus by Creed.", "tags": [], "expected": {"response_language": "en", "intent_in": ["perfume_profile"], "resolved_labels": ["Aventus by Creed"]}}\n{"id": "fev1_entity_003", "version": "final_eval_v1", "category": "entity_resolution", "language": "tr", "query": "Burberry Hero hakkında bilgi ver.", "tags": [], "expected": {"response_language": "tr", "intent_in": ["perfume_profile"], "resolved_labels": ["Hero by Burberry"]}}\n{"id": "fev1_entity_004", "version": "final_eval_v1", "category": "entity_resolution", "language": "en", "query": "What is YSL Y EDP like?", "tags": [], "expected": {"response_language": "en", "intent_in": ["perfume_profile"], "resolved_labels": ["Y Eau de Parfum by Yves Saint Laurent"]}}\n{"id": "fev1_entity_005", "version": "final_eval_v1", "category": "entity_resolution", "language": "tr", "query": "Prada L\'Homme nasıl bir parfüm?", "tags": [], "expected": {"response_language": "tr", "intent_in": ["perfume_profile"], "resolved_labels": ["Prada L\'Homme by Prada"]}}\n{"id": "fev1_entity_006", "version": "final_eval_v1", "category": "entity_resolution", "language": "en", "query": "Tell me about LV Imagination.", "tags": [], "expected": {"response_language": "en", "intent_in": ["perfume_profile"], "resolved_labels": ["Imagination by Louis Vuitton"]}}\n{"id": "fev1_entity_007", "version": "final_eval_v1", "category": "entity_resolution", "language": "tr", "query": "MFK Grand Soir hakkında bilgi ver.", "tags": [], "expected": {"response_language": "tr", "intent_in": ["perfume_profile"], "resolved_labels": ["Grand Soir by Maison Francis Kurkdjian"]}}\n{"id": "fev1_entity_008", "version": "final_eval_v1", "category": "entity_resolution", "language": "en", "query": "What is Adidas Team Five like?", "tags": [], "expected": {"response_language": "en", "intent_in": ["perfume_profile"], "resolved_labels": ["Team Five by Adidas"]}}\n{"id": "fev1_entity_009", "version": "final_eval_v1", "category": "entity_resolution", "language": "tr", "query": "Tom Ford Tobacco Vanille nasıl bir parfüm?", "tags": [], "expected": {"response_language": "tr", "intent_in": ["perfume_profile"], "resolved_labels": ["Tobacco Vanille by Tom Ford"]}}\n{"id": "fev1_entity_010", "version": "final_eval_v1", "category": "entity_resolution", "language": "en", "query": "Tell me about Bleu de Chanel EDP.", "tags": [], "expected": {"response_language": "en", "intent_in": ["perfume_profile"], "resolved_labels": ["Bleu de Chanel Eau de Parfum by Chanel"]}}\n{"id": "fev1_conv_001", "version": "final_eval_v1", "category": "conversation", "language": "en", "query": "Recommend exactly 3 popular fragrances that must have vanilla.", "tags": [], "expected": {"response_language": "en", "intent_in": ["recommendation"], "requested_count": 3, "conversation_action": "new_request", "required_terms": ["vanilla"], "excluded_terms": [], "no_repeat_previous": false, "discovery_mode": "mainstream"}, "session_id": "conv_vanilla", "turn": 1}\n{"id": "fev1_conv_002", "version": "final_eval_v1", "category": "conversation", "language": "en", "query": "Give me three different options under the same requirements.", "tags": [], "expected": {"response_language": "en", "intent_in": ["recommendation"], "requested_count": 3, "conversation_action": "more_options", "required_terms": ["vanilla"], "excluded_terms": [], "no_repeat_previous": true, "discovery_mode": "mainstream"}, "session_id": "conv_vanilla", "turn": 2}\n{"id": "fev1_conv_003", "version": "final_eval_v1", "category": "conversation", "language": "tr", "query": "Ofis için vanilyasız tam 3 temiz parfüm öner.", "tags": [], "expected": {"response_language": "tr", "intent_in": ["recommendation"], "requested_count": 3, "conversation_action": "new_request", "required_terms": [], "excluded_terms": ["vanilla"], "no_repeat_previous": false}, "session_id": "conv_office", "turn": 1}\n{"id": "fev1_conv_004", "version": "final_eval_v1", "category": "conversation", "language": "tr", "query": "Aynı koşullarda daha sportif üç farklı seçenek ver.", "tags": [], "expected": {"response_language": "tr", "intent_in": ["recommendation"], "requested_count": 3, "conversation_action": "refine_previous", "required_terms": [], "excluded_terms": ["vanilla"], "no_repeat_previous": true}, "session_id": "conv_office", "turn": 2}\n{"id": "fev1_conv_005", "version": "final_eval_v1", "category": "conversation", "language": "en", "query": "Recommend exactly 3 citrus summer fragrances for men.", "tags": [], "expected": {"response_language": "en", "intent_in": ["recommendation"], "requested_count": 3, "conversation_action": "new_request", "required_terms": [], "excluded_terms": [], "no_repeat_previous": false}, "session_id": "conv_summer", "turn": 1}\n{"id": "fev1_conv_006", "version": "final_eval_v1", "category": "conversation", "language": "en", "query": "Make them unisex and give me three new options.", "tags": [], "expected": {"response_language": "en", "intent_in": ["recommendation"], "requested_count": 3, "conversation_action": "refine_previous", "required_terms": [], "excluded_terms": [], "no_repeat_previous": true}, "session_id": "conv_summer", "turn": 2}\n{"id": "fev1_conv_007", "version": "final_eval_v1", "category": "conversation", "language": "tr", "query": "Bana popüler, vanilyalı tam 3 parfüm öner.", "tags": [], "expected": {"response_language": "tr", "intent_in": ["recommendation"], "requested_count": 3, "conversation_action": "new_request", "required_terms": ["vanilla"], "excluded_terms": [], "no_repeat_previous": false, "discovery_mode": "mainstream"}, "session_id": "conv_discovery", "turn": 1}\n{"id": "fev1_conv_008", "version": "final_eval_v1", "category": "conversation", "language": "tr", "query": "Şimdi aynı isteğe uygun ama daha niş üç farklı seçenek ver.", "tags": [], "expected": {"response_language": "tr", "intent_in": ["recommendation"], "requested_count": 3, "conversation_action": "refine_previous", "required_terms": ["vanilla"], "excluded_terms": [], "no_repeat_previous": true, "discovery_mode": "niche"}, "session_id": "conv_discovery", "turn": 2}\n{"id": "fev1_conv_009", "version": "final_eval_v1", "category": "conversation", "language": "en", "query": "Recommend exactly 3 warm spicy fragrances for a date night.", "tags": [], "expected": {"response_language": "en", "intent_in": ["recommendation"], "requested_count": 3, "conversation_action": "new_request", "required_terms": [], "excluded_terms": [], "no_repeat_previous": false}, "session_id": "conv_date", "turn": 1}\n{"id": "fev1_conv_010", "version": "final_eval_v1", "category": "conversation", "language": "en", "query": "Keep the same request, but exclude tobacco and show three different choices.", "tags": [], "expected": {"response_language": "en", "intent_in": ["recommendation"], "requested_count": 3, "conversation_action": "refine_previous", "required_terms": [], "excluded_terms": ["tobacco"], "no_repeat_previous": true}, "session_id": "conv_date", "turn": 2}\n{"id": "fev1_unsupported_001", "version": "final_eval_v1", "category": "unsupported", "language": "en", "query": "What is the current price of Aventus by Creed?", "tags": [], "expected": {"response_language": "en", "intent_in": ["unsupported_price"], "route": "unsupported_price", "generation_attempts": 0}}\n{"id": "fev1_unsupported_002", "version": "final_eval_v1", "category": "unsupported", "language": "tr", "query": "Tobacco Vanille şu anda Türkiye\'de stokta mı?", "tags": [], "expected": {"response_language": "tr", "intent_in": ["unsupported_availability"], "route": "unsupported_availability", "generation_attempts": 0}}\n{"id": "fev1_unsupported_003", "version": "final_eval_v1", "category": "unsupported", "language": "en", "query": "Is Santal 33 safe for someone with a severe fragrance allergy?", "tags": [], "expected": {"response_language": "en", "intent_in": ["unsupported_medical"], "route": "unsupported_medical", "generation_attempts": 0}}\n{"id": "fev1_unsupported_004", "version": "final_eval_v1", "category": "unsupported", "language": "tr", "query": "En çok iltifat getirecek parfüm hangisi?", "tags": [], "expected": {"response_language": "tr", "intent_in": ["unsupported_social_claim"], "route": "unsupported_social_claim", "generation_attempts": 0}}\n{"id": "fev1_unsupported_005", "version": "final_eval_v1", "category": "unsupported", "language": "en", "query": "How should I layer Aventus with Tobacco Vanille?", "tags": [], "expected": {"response_language": "en", "intent_in": ["unsupported_layering"], "route": "unsupported_layering", "generation_attempts": 0}}\n{"id": "fev1_unsupported_006", "version": "final_eval_v1", "category": "unsupported", "language": "tr", "query": "Grand Soir\'ın güncel fiyatı ne kadar?", "tags": [], "expected": {"response_language": "tr", "intent_in": ["unsupported_price"], "route": "unsupported_price", "generation_attempts": 0}}\n{"id": "fev1_unsupported_007", "version": "final_eval_v1", "category": "unsupported", "language": "en", "query": "Can you confirm whether Prada L\'Homme is discontinued and in stock?", "tags": [], "expected": {"response_language": "en", "intent_in": ["unsupported_availability"], "route": "unsupported_availability", "generation_attempts": 0}}\n{"id": "fev1_unsupported_008", "version": "final_eval_v1", "category": "unsupported", "language": "tr", "query": "Bu parfüm astımımı tetikler mi, güvenli mi?", "tags": [], "expected": {"response_language": "tr", "intent_in": ["unsupported_medical"], "route": "unsupported_medical", "generation_attempts": 0}}\n{"id": "fev1_unsupported_009", "version": "final_eval_v1", "category": "unsupported", "language": "en", "query": "Which perfume is guaranteed to attract compliments on a date?", "tags": [], "expected": {"response_language": "en", "intent_in": ["unsupported_social_claim"], "route": "unsupported_social_claim", "generation_attempts": 0}}\n{"id": "fev1_unsupported_010", "version": "final_eval_v1", "category": "unsupported", "language": "tr", "query": "Santal 33 ile Another 13\'ü nasıl katmanlamalıyım?", "tags": [], "expected": {"response_language": "tr", "intent_in": ["unsupported_layering"], "route": "unsupported_layering", "generation_attempts": 0}}\n{"id": "fev1_noisy_001", "version": "final_eval_v1", "category": "multilingual_noisy", "language": "tr", "query": "bna yaz icin ferah tam 3 erkek parfumu oner", "tags": ["noisy_or_colloquial"], "expected": {"response_language": "tr", "intent_in": ["recommendation"], "requested_count": 3, "gender": "male"}}\n{"id": "fev1_noisy_002", "version": "final_eval_v1", "category": "multilingual_noisy", "language": "en", "query": "need exactly 3 ofice scents no vanila pls", "tags": ["noisy_or_colloquial"], "expected": {"response_language": "en", "intent_in": ["recommendation"], "requested_count": 3, "excluded_terms": ["vanilla"]}}\n{"id": "fev1_noisy_003", "version": "final_eval_v1", "category": "multilingual_noisy", "language": "tr", "query": "aventus nasil bi parfum", "tags": ["noisy_or_colloquial"], "expected": {"response_language": "tr", "intent_in": ["perfume_profile"], "resolved_labels": ["Aventus by Creed"]}}\n{"id": "fev1_noisy_004", "version": "final_eval_v1", "category": "multilingual_noisy", "language": "en", "query": "cmpare eros by versace n eros flame by versace", "tags": ["noisy_or_colloquial"], "expected": {"response_language": "en", "intent_in": ["comparison"], "resolved_labels": ["Eros by Versace", "Eros Flame by Versace"]}}\n{"id": "fev1_noisy_005", "version": "final_eval_v1", "category": "multilingual_noisy", "language": "tr", "query": "bana date night icin baharatli tam 3 parfum", "tags": ["noisy_or_colloquial"], "expected": {"response_language": "tr", "intent_in": ["recommendation"], "requested_count": 3}}\n{"id": "fev1_noisy_006", "version": "final_eval_v1", "category": "multilingual_noisy", "language": "en", "query": "wht is turkish leather by pryn parfum like", "tags": ["noisy_or_colloquial"], "expected": {"response_language": "en", "intent_in": ["perfume_profile"], "resolved_labels": ["Turkish Leather by Pryn Parfum"]}}\n{"id": "fev1_noisy_007", "version": "final_eval_v1", "category": "multilingual_noisy", "language": "tr", "query": "Versace\'den tam 4 erkek parfumu oner", "tags": ["noisy_or_colloquial"], "expected": {"response_language": "tr", "intent_in": ["recommendation"], "requested_count": 4, "requested_brand": "Versace", "gender": "male"}}\n{"id": "fev1_noisy_008", "version": "final_eval_v1", "category": "multilingual_noisy", "language": "tr", "query": "clean ama sweet olmayan tam 3 bisey oner", "tags": ["noisy_or_colloquial"], "expected": {"response_language": "tr", "intent_in": ["recommendation"], "requested_count": 3, "excluded_terms": ["sweet"]}}\n{"id": "fev1_noisy_009", "version": "final_eval_v1", "category": "multilingual_noisy", "language": "en", "query": "smth like santal 33 by le labo but less leathery exactly 3", "tags": ["noisy_or_colloquial"], "expected": {"response_language": "en", "intent_in": ["similarity", "alternative"], "requested_count": 3, "reference_label": "Santal 33 by Le Labo", "excluded_terms": ["leather"]}}\n{"id": "fev1_noisy_010", "version": "final_eval_v1", "category": "multilingual_noisy", "language": "tr", "query": "LV imagination hakkinda ne dusunuyosun", "tags": ["noisy_or_colloquial"], "expected": {"response_language": "tr", "intent_in": ["perfume_profile"], "resolved_labels": ["Imagination by Louis Vuitton"]}}\n'
FINAL_EVAL_RUNTIME_SOURCE = 'from __future__ import annotations\n\nimport csv\nimport json\nimport math\nimport os\nimport statistics\nimport time\nfrom collections import Counter, defaultdict\nfrom datetime import datetime, timezone\nfrom pathlib import Path\nfrom typing import Any\n\n\nFINAL_EVAL_GATES = {\n    "overall_pass_rate": (">=", 0.95),\n    "language_pass_rate": (">=", 1.0),\n    "requested_count_pass_rate": (">=", 1.0),\n    "hard_filter_pass_rate": (">=", 1.0),\n    "entity_resolution_pass_rate": (">=", 1.0),\n    "unsupported_route_pass_rate": (">=", 1.0),\n    "conversation_no_repeat_pass_rate": (">=", 1.0),\n    "performance_calibration_pass_rate": (">=", 1.0),\n    "first_attempt_rate": (">=", 0.90),\n    "fallback_rate": ("<=", 0.05),\n    "p50_latency_seconds": ("<=", 12.0),\n    "p95_latency_seconds": ("<=", 20.0),\n}\n\n\ndef read_jsonl(path: Path) -> list[dict[str, Any]]:\n    rows: list[dict[str, Any]] = []\n    with path.open(encoding="utf-8") as handle:\n        for line_number, line in enumerate(handle, 1):\n            line = line.strip()\n            if not line:\n                continue\n            try:\n                row = json.loads(line)\n            except json.JSONDecodeError as exc:\n                raise ValueError(f"Malformed JSONL at {path}:{line_number}") from exc\n            if not isinstance(row, dict):\n                raise ValueError(f"JSONL row must be an object at {path}:{line_number}")\n            rows.append(row)\n    return rows\n\n\ndef read_completed_outputs(path: Path) -> tuple[dict[str, dict[str, Any]], int]:\n    if not path.exists():\n        return {}, 0\n    completed: dict[str, dict[str, Any]] = {}\n    malformed = 0\n    with path.open(encoding="utf-8") as handle:\n        for line in handle:\n            line = line.strip()\n            if not line:\n                continue\n            try:\n                row = json.loads(line)\n            except json.JSONDecodeError:\n                malformed += 1\n                continue\n            case_id = str(row.get("id") or "")\n            if case_id:\n                completed[case_id] = row\n    return completed, malformed\n\n\ndef write_completed_snapshot(\n    path: Path,\n    cases: list[dict[str, Any]],\n    completed: dict[str, dict[str, Any]],\n) -> None:\n    path.parent.mkdir(parents=True, exist_ok=True)\n    with path.open("w", encoding="utf-8") as handle:\n        for case in cases:\n            row = completed.get(case["id"])\n            if row:\n                handle.write(json.dumps(row, ensure_ascii=False) + "\\n")\n        handle.flush()\n        os.fsync(handle.fileno())\n\n\ndef selected_candidates(result: dict[str, Any]) -> list[dict[str, Any]]:\n    validation = result.get("validation") or {}\n    labels = validation.get("numbered_recommendations") or validation.get("mentioned_candidates") or []\n    selected = [candidate for candidate in result.get("candidates", []) if candidate.get("label") in labels]\n    return selected\n\n\ndef selected_candidate_ids(result: dict[str, Any]) -> list[int]:\n    return [int(candidate["perfume_id"]) for candidate in selected_candidates(result)]\n\n\ndef restore_session_state(session: Any, row: dict[str, Any]) -> None:\n    result = row.get("result") or {}\n    session.last_result = result\n    session.recommendation_ids = [int(value) for value in row.get("session_recommendation_ids", [])]\n\n\ndef normalized_label_set(candidates: list[dict[str, Any]]) -> set[str]:\n    return {str(candidate.get("label") or "") for candidate in candidates if candidate.get("label")}\n\n\ndef evaluation_trait_matches(expected: str, actual: str) -> bool:\n    expected_norm = " ".join(str(expected or "").lower().replace("-", " ").split())\n    actual_norm = " ".join(str(actual or "").lower().replace("-", " ").split())\n    if expected_norm == actual_norm:\n        return True\n    if expected_norm == "spicy" and actual_norm in {"fresh spicy", "soft spicy", "warm spicy"}:\n        return True\n    families = (\n        {"smoke", "smoky"},\n        {"musk", "musky", "white musk"},\n        {"leather", "leathery", "suede"},\n        {"oud", "aoud", "agarwood"},\n    )\n    return any(expected_norm in family and actual_norm in family for family in families)\n\n\ndef plan_contains_traits(expected_terms: list[str], actual_terms: list[str]) -> bool:\n    return all(\n        any(evaluation_trait_matches(expected, actual) for actual in actual_terms)\n        for expected in expected_terms\n    )\n\n\ndef score_final_case(\n    case: dict[str, Any],\n    result: dict[str, Any],\n    *,\n    previous_selected_ids: set[int] | None = None,\n) -> dict[str, Any]:\n    expected = case["expected"]\n    plan = result.get("plan") or {}\n    validation = result.get("validation") or {}\n    candidates = result.get("candidates") or []\n    selected = selected_candidates(result)\n    selected_ids = set(selected_candidate_ids(result))\n    checks: dict[str, bool] = {}\n    details: dict[str, Any] = {}\n\n    checks["pipeline_validation"] = bool(validation.get("pass"))\n    checks["response_language"] = (\n        result.get("response_language") == expected["response_language"]\n        and response_language_matches(result.get("answer", ""), expected["response_language"])\n    )\n    checks["intent"] = plan.get("intent") in expected.get("intent_in", [])\n    checks["safe_route"] = result.get("route") not in {"retrieval_error", "comparison_unresolved", "no_safe_match"}\n\n    if "route" in expected:\n        checks["expected_route"] = result.get("route") == expected["route"]\n    if "generation_attempts" in expected:\n        checks["generation_attempts"] = int(result.get("generation_attempts") or 0) == expected["generation_attempts"]\n\n    if "requested_count" in expected:\n        count = int(expected["requested_count"])\n        checks["requested_count"] = (\n            int(plan.get("requested_count") or 0) == count\n            and len(validation.get("numbered_recommendations") or []) == count\n        )\n\n    expected_labels = expected.get("resolved_labels") or []\n    if expected_labels:\n        actual_labels = [candidate.get("label") for candidate in candidates]\n        checks["entity_resolution"] = len(actual_labels) == len(expected_labels) and set(actual_labels) == set(expected_labels)\n        details["expected_labels"] = expected_labels\n        details["actual_labels"] = actual_labels\n\n    reference_label = expected.get("reference_label")\n    if reference_label:\n        reference = result.get("reference") or {}\n        checks["reference_resolution"] = reference.get("label") == reference_label\n        details["expected_reference"] = reference_label\n        details["actual_reference"] = reference.get("label")\n\n    required_terms = expected.get("required_terms") or []\n    if required_terms:\n        checks["required_terms_in_plan"] = plan_contains_traits(\n            required_terms,\n            plan.get("required_terms") or [],\n        )\n        checks["required_terms_in_results"] = bool(selected) and all(\n            all(candidate_has_term(candidate, term) for term in required_terms)\n            for candidate in selected\n        )\n\n    excluded_terms = expected.get("excluded_terms") or []\n    actual_excluded_terms = plan.get("excluded_terms") or []\n    filter_relevant_intents = {"recommendation", "similarity", "alternative"}\n    if excluded_terms or (actual_excluded_terms and plan.get("intent") in filter_relevant_intents):\n        checks["no_unexpected_excluded_terms"] = all(\n            any(evaluation_trait_matches(expected_term, actual_term) for expected_term in excluded_terms)\n            for actual_term in actual_excluded_terms\n        )\n        if not checks["no_unexpected_excluded_terms"]:\n            details["unexpected_excluded_terms"] = [\n                actual_term\n                for actual_term in actual_excluded_terms\n                if not any(\n                    evaluation_trait_matches(expected_term, actual_term)\n                    for expected_term in excluded_terms\n                )\n            ]\n    if excluded_terms:\n        checks["excluded_terms_in_plan"] = plan_contains_traits(\n            excluded_terms,\n            actual_excluded_terms,\n        )\n        checks["excluded_terms_absent"] = all(\n            all(not candidate_has_term(candidate, term) for term in excluded_terms)\n            for candidate in selected\n        )\n\n    requested_brand = expected.get("requested_brand")\n    if requested_brand:\n        checks["requested_brand"] = plan.get("requested_brand") == requested_brand\n        checks["brand_results"] = bool(selected) and all(candidate.get("brand") == requested_brand for candidate in selected)\n\n    for field in ("gender", "season", "time_profile", "discovery_mode"):\n        if field in expected:\n            checks[field] = plan.get(field) == expected[field]\n    if "conversation_action" in expected:\n        expected_action = expected["conversation_action"]\n        actual_action = plan.get("conversation_action")\n        follow_up_actions = {"more_options", "refine_previous"}\n        checks["conversation_action"] = (\n            actual_action == expected_action\n            or {actual_action, expected_action}.issubset(follow_up_actions)\n        )\n\n    if expected.get("no_repeat_previous"):\n        previous = previous_selected_ids or set()\n        checks["conversation_no_repeat"] = bool(previous) and bool(selected_ids) and previous.isdisjoint(selected_ids)\n        details["previous_selected_ids"] = sorted(previous)\n        details["current_selected_ids"] = sorted(selected_ids)\n\n    performance_violations = validation.get("performance_calibration_violations") or []\n    checks["performance_calibration"] = not performance_violations\n    details["performance_calibration_violations"] = performance_violations\n\n    failures = [name for name, passed in checks.items() if not passed]\n    return {\n        "pass": not failures,\n        "failures": failures,\n        "checks": checks,\n        "details": details,\n        "selected_candidate_ids": sorted(selected_ids),\n        "selected_candidate_labels": [candidate["label"] for candidate in selected],\n    }\n\n\ndef percentile(values: list[float], probability: float) -> float:\n    if not values:\n        return 0.0\n    ordered = sorted(values)\n    position = (len(ordered) - 1) * probability\n    lower = math.floor(position)\n    upper = math.ceil(position)\n    if lower == upper:\n        return ordered[lower]\n    return ordered[lower] + (ordered[upper] - ordered[lower]) * (position - lower)\n\n\ndef rate(numerator: int, denominator: int) -> float:\n    return round(numerator / denominator, 4) if denominator else 1.0\n\n\ndef metric_rate(rows: list[dict[str, Any]], check_names: set[str]) -> float:\n    applicable = [row for row in rows if any(name in row["score"]["checks"] for name in check_names)]\n    passed = [\n        row for row in applicable\n        if all(row["score"]["checks"].get(name, True) for name in check_names)\n    ]\n    return rate(len(passed), len(applicable))\n\n\ndef grouped_summary(rows: list[dict[str, Any]], key: str) -> dict[str, Any]:\n    buckets: dict[str, list[dict[str, Any]]] = defaultdict(list)\n    for row in rows:\n        buckets[str(row.get(key) or "unknown")].append(row)\n    return {\n        name: {\n            "count": len(bucket),\n            "pass_count": sum(row["score"]["pass"] for row in bucket),\n            "pass_rate": rate(sum(row["score"]["pass"] for row in bucket), len(bucket)),\n        }\n        for name, bucket in sorted(buckets.items())\n    }\n\n\ndef evaluate_gate(value: float, operator: str, threshold: float) -> bool:\n    return value >= threshold if operator == ">=" else value <= threshold\n\n\ndef summarize_final_eval(rows: list[dict[str, Any]]) -> dict[str, Any]:\n    latencies = [float(row["result"].get("timings", {}).get("total_seconds") or 0.0) for row in rows]\n    total = len(rows)\n    pass_count = sum(row["score"]["pass"] for row in rows)\n    attempts = [int(row["result"].get("generation_attempts") or 0) for row in rows]\n    generated_rows = [row for row in rows if int(row["result"].get("generation_attempts") or 0) > 0]\n    fallback_count = sum(row["result"].get("route") == "validated_template_fallback" for row in rows)\n    failure_counter = Counter(\n        failure for row in rows for failure in row["score"].get("failures", [])\n    )\n    validator_reason_counter = Counter(\n        reason\n        for row in rows\n        for reason in row["result"].get("validation", {}).get("reasons", [])\n    )\n\n    metrics = {\n        "overall_pass_rate": rate(pass_count, total),\n        "language_pass_rate": metric_rate(rows, {"response_language"}),\n        "requested_count_pass_rate": metric_rate(rows, {"requested_count"}),\n        "hard_filter_pass_rate": metric_rate(rows, {\n            "required_terms_in_plan", "required_terms_in_results",\n            "excluded_terms_in_plan", "excluded_terms_absent", "no_unexpected_excluded_terms",\n        }),\n        "hard_filter_plan_pass_rate": metric_rate(rows, {\n            "required_terms_in_plan", "excluded_terms_in_plan", "no_unexpected_excluded_terms",\n        }),\n        "hard_filter_output_pass_rate": metric_rate(rows, {\n            "required_terms_in_results", "excluded_terms_absent",\n        }),\n        "entity_resolution_pass_rate": metric_rate(rows, {"entity_resolution", "reference_resolution"}),\n        "unsupported_route_pass_rate": metric_rate(rows, {"expected_route", "generation_attempts"}),\n        "conversation_no_repeat_pass_rate": metric_rate(rows, {"conversation_no_repeat"}),\n        "performance_calibration_pass_rate": metric_rate(rows, {"performance_calibration"}),\n        "first_attempt_rate": rate(\n            sum(int(row["result"].get("generation_attempts") or 0) == 1 for row in generated_rows),\n            len(generated_rows),\n        ),\n        "fallback_rate": rate(fallback_count, total),\n        "average_latency_seconds": round(statistics.mean(latencies), 4) if latencies else 0.0,\n        "p50_latency_seconds": round(percentile(latencies, 0.50), 4),\n        "p95_latency_seconds": round(percentile(latencies, 0.95), 4),\n        "max_latency_seconds": round(max(latencies), 4) if latencies else 0.0,\n    }\n    gates = {\n        name: {\n            "value": metrics[name],\n            "operator": operator,\n            "threshold": threshold,\n            "pass": evaluate_gate(float(metrics[name]), operator, threshold),\n        }\n        for name, (operator, threshold) in FINAL_EVAL_GATES.items()\n    }\n    return {\n        "version": "final_eval_v1",\n        "created_at_utc": datetime.now(timezone.utc).isoformat(),\n        "case_count": total,\n        "pass_count": pass_count,\n        "failure_count": total - pass_count,\n        "metrics": metrics,\n        "quality_gates": gates,\n        "all_quality_gates_passed": all(item["pass"] for item in gates.values()),\n        "category_summary": grouped_summary(rows, "category"),\n        "language_summary": grouped_summary(rows, "language"),\n        "route_counts": dict(Counter(row["result"].get("route") for row in rows)),\n        "generation_attempt_counts": dict(Counter(attempts)),\n        "failure_counts": dict(failure_counter),\n        "validator_reason_counts": dict(validator_reason_counter),\n        "failed_case_ids": [row["id"] for row in rows if not row["score"]["pass"]],\n    }\n\n\ndef select_human_review_rows(rows: list[dict[str, Any]], limit: int = 40) -> list[dict[str, Any]]:\n    selected: list[dict[str, Any]] = []\n    seen: set[str] = set()\n    for row in rows:\n        if not row["score"]["pass"]:\n            selected.append(row)\n            seen.add(row["id"])\n            if len(selected) >= limit:\n                return selected\n    buckets: dict[str, list[dict[str, Any]]] = defaultdict(list)\n    for row in rows:\n        if row["id"] not in seen:\n            buckets[row["category"]].append(row)\n    categories = sorted(buckets)\n    while len(selected) < limit and any(buckets.values()):\n        for category in categories:\n            if buckets[category] and len(selected) < limit:\n                row = buckets[category].pop(0)\n                selected.append(row)\n                seen.add(row["id"])\n    return selected\n\n\ndef write_human_review_csv(path: Path, rows: list[dict[str, Any]], limit: int = 40) -> None:\n    selected = select_human_review_rows(rows, limit=limit)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    fieldnames = [\n        "id", "category", "language", "query", "answer", "candidate_labels",\n        "route", "latency_seconds", "auto_pass", "auto_failures",\n        "grounding_1_5", "technical_accuracy_1_5", "advisor_value_1_5",\n        "naturalness_1_5", "review_notes",\n    ]\n    with path.open("w", encoding="utf-8", newline="") as handle:\n        writer = csv.DictWriter(handle, fieldnames=fieldnames)\n        writer.writeheader()\n        for row in selected:\n            result = row["result"]\n            writer.writerow({\n                "id": row["id"],\n                "category": row["category"],\n                "language": row["language"],\n                "query": row["query"],\n                "answer": result.get("answer", ""),\n                "candidate_labels": " | ".join(candidate.get("label", "") for candidate in result.get("candidates", [])),\n                "route": result.get("route"),\n                "latency_seconds": result.get("timings", {}).get("total_seconds"),\n                "auto_pass": row["score"]["pass"],\n                "auto_failures": " | ".join(row["score"]["failures"]),\n                "grounding_1_5": "",\n                "technical_accuracy_1_5": "",\n                "advisor_value_1_5": "",\n                "naturalness_1_5": "",\n                "review_notes": "",\n            })\n\n\ndef run_final_evaluation(\n    pipeline: Any,\n    cases: list[dict[str, Any]],\n    *,\n    outputs_path: Path,\n    summary_path: Path,\n    human_review_path: Path,\n    metadata_path: Path,\n    resume: bool = True,\n    limit: int | None = None,\n    run_metadata: dict[str, Any] | None = None,\n) -> dict[str, Any]:\n    active_cases = cases[:limit] if limit is not None else list(cases)\n    valid_ids = {case["id"] for case in active_cases}\n    completed, malformed_lines = read_completed_outputs(outputs_path) if resume else ({}, 0)\n    completed = {case_id: row for case_id, row in completed.items() if case_id in valid_ids}\n    write_completed_snapshot(outputs_path, active_cases, completed)\n    sessions: dict[str, Any] = {}\n    session_previous_ids: dict[str, set[int]] = defaultdict(set)\n    metadata = {\n        "version": "final_eval_v1",\n        "started_at_utc": datetime.now(timezone.utc).isoformat(),\n        "case_count": len(active_cases),\n        "resume": resume,\n        "already_completed": len(completed),\n        "malformed_output_lines_skipped": malformed_lines,\n        "outputs_path": str(outputs_path),\n        "runtime": run_metadata or {},\n    }\n    metadata_path.parent.mkdir(parents=True, exist_ok=True)\n    metadata_path.write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding="utf-8")\n\n    mode = "a" if completed else "w"\n    with outputs_path.open(mode, encoding="utf-8") as handle:\n        for index, case in enumerate(active_cases, 1):\n            session_id = case.get("session_id")\n            session = None\n            if session_id:\n                session = sessions.setdefault(session_id, ScentAISession(pipeline))\n            existing = completed.get(case["id"])\n            if existing:\n                if session is not None:\n                    restore_session_state(session, existing)\n                    session_previous_ids[session_id].update(existing.get("score", {}).get("selected_candidate_ids", []))\n                print(f"[{index}/{len(active_cases)}] resume {case[\'id\']}")\n                continue\n\n            started = time.perf_counter()\n            result = session.run(case["query"]) if session is not None else pipeline.run(case["query"])\n            runner_seconds = round(time.perf_counter() - started, 4)\n            previous_ids = set(session_previous_ids.get(session_id, set())) if session_id else set()\n            score = score_final_case(case, result, previous_selected_ids=previous_ids)\n            if session_id:\n                session_previous_ids[session_id].update(score["selected_candidate_ids"])\n            row = {\n                "id": case["id"],\n                "version": case["version"],\n                "category": case["category"],\n                "language": case["language"],\n                "query": case["query"],\n                "tags": case.get("tags", []),\n                "session_id": session_id,\n                "turn": case.get("turn"),\n                "expected": case["expected"],\n                "result": result,\n                "score": score,\n                "session_recommendation_ids": list(session.recommendation_ids) if session is not None else [],\n                "runner_seconds": runner_seconds,\n                "generated_at_utc": datetime.now(timezone.utc).isoformat(),\n            }\n            completed[case["id"]] = row\n            handle.write(json.dumps(row, ensure_ascii=False) + "\\n")\n            handle.flush()\n            os.fsync(handle.fileno())\n            print(\n                f"[{index}/{len(active_cases)}] {case[\'id\']} {case[\'category\']} "\n                f"pass={score[\'pass\']} route={result.get(\'route\')} "\n                f"attempts={result.get(\'generation_attempts\')} seconds={runner_seconds}"\n            )\n            if score["failures"]:\n                print("  failures:", score["failures"])\n\n    ordered_rows = [completed[case["id"]] for case in active_cases if case["id"] in completed]\n    write_completed_snapshot(outputs_path, active_cases, completed)\n    summary = summarize_final_eval(ordered_rows)\n    summary_path.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")\n    write_human_review_csv(human_review_path, ordered_rows, limit=min(40, len(ordered_rows)))\n    metadata.update({\n        "completed_at_utc": datetime.now(timezone.utc).isoformat(),\n        "completed_count": len(ordered_rows),\n        "summary_path": str(summary_path),\n        "human_review_path": str(human_review_path),\n    })\n    metadata_path.write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding="utf-8")\n    return summary\n'

compile(FINAL_EVAL_RUNTIME_SOURCE, "<scentai_final_eval_runtime>", "exec")
exec(compile(FINAL_EVAL_RUNTIME_SOURCE, "<scentai_final_eval_runtime>", "exec"), globals())

FINAL_EVAL_DIR = PROJECT_DIR / "runs" / "final_eval_v1_pipeline_v4"
FINAL_EVAL_CASES_PATH = FINAL_EVAL_DIR / "final_eval_v1.jsonl"
FINAL_EVAL_OUTPUTS_PATH = FINAL_EVAL_DIR / "final_eval_outputs.jsonl"
FINAL_EVAL_SUMMARY_PATH = FINAL_EVAL_DIR / "final_eval_summary.json"
FINAL_EVAL_HUMAN_REVIEW_PATH = FINAL_EVAL_DIR / "final_eval_human_review.csv"
FINAL_EVAL_METADATA_PATH = FINAL_EVAL_DIR / "final_eval_metadata.json"
FINAL_EVAL_RESUME = True
FINAL_EVAL_LIMIT = None  # Set to 10 only for a quick runner smoke test.

FINAL_EVAL_DIR.mkdir(parents=True, exist_ok=True)
FINAL_EVAL_CASES_PATH.write_text(FINAL_EVAL_SET_SOURCE, encoding="utf-8")
final_eval_cases = [json.loads(line) for line in FINAL_EVAL_SET_SOURCE.splitlines() if line.strip()]
assert len(final_eval_cases) == 120, len(final_eval_cases)

final_eval_summary = run_final_evaluation(
    pipeline,
    final_eval_cases,
    outputs_path=FINAL_EVAL_OUTPUTS_PATH,
    summary_path=FINAL_EVAL_SUMMARY_PATH,
    human_review_path=FINAL_EVAL_HUMAN_REVIEW_PATH,
    metadata_path=FINAL_EVAL_METADATA_PATH,
    resume=FINAL_EVAL_RESUME,
    limit=FINAL_EVAL_LIMIT,
    run_metadata={
        "model_name": MODEL_NAME,
        "answer_model": MODEL_NAME,
        "repair_model": LORA_NAME,
        "adapter_dir": str(ADAPTER_DIR),
        "retrieval_health": retrieval_client.health(),
    },
)

print("\n" + "=" * 110)
print("FINAL EVALUATION COMPLETE")
print(json.dumps(final_eval_summary, indent=2, ensure_ascii=False))
print("Outputs:", FINAL_EVAL_OUTPUTS_PATH)
print("Summary:", FINAL_EVAL_SUMMARY_PATH)
print("Human review:", FINAL_EVAL_HUMAN_REVIEW_PATH)
print("Metadata:", FINAL_EVAL_METADATA_PATH)

## Interactive full-pipeline use

In [ ]:
RESULTS_PATH = PROJECT_DIR / "runs" / "stage3_interactive_results.jsonl"
scentai_session = ScentAISession(pipeline)


def ask_scentai(query, *, save=True):
    result = scentai_session.run(query)
    print("Route:", result["route"])
    print("Planner:", result["plan"]["intent"], "| confidence=", result["plan"]["confidence"])
    print("Candidates:")
    for index, candidate in enumerate(result["candidates"], 1):
        print(f"  {index:02d}. {candidate['label']}")
    print("Validation:", result["validation"])
    print("Generation attempts:", result["generation_attempts"])
    print("Total time:", result["timings"]["total_seconds"], "seconds")
    print("\nAnswer:\n" + result["answer"])
    if save:
        RESULTS_PATH.parent.mkdir(parents=True, exist_ok=True)
        with RESULTS_PATH.open("a", encoding="utf-8") as handle:
            handle.write(json.dumps(result, ensure_ascii=False) + "\n")
        print("\nSaved:", RESULTS_PATH)
    return result


def reset_scentai_session():
    scentai_session.reset()
    print("ScentAI conversation memory cleared.")


my_query = "Bana popüler ve vanilyalı 3 parfüm öner."
my_result = ask_scentai(my_query)

# Natural follow-up example; this keeps the vanilla/mainstream constraints and
# excludes the perfumes that were actually recommended above:
# my_result = ask_scentai("Başka üç seçenek istiyorum.")
# Start an unrelated topic with: reset_scentai_session()